# <center>Kubernetes集群编排入门</center>

&emsp;&emsp;我们在前一门课《容器化部署 Docker + Compose》里,已经把文搜图五服务系统（etcd / minio / milvus / backend / frontend）用 docker-compose 在单机跑起来了。那是单机部署的终点。本课承接上一门,把同一个项目搬上 Kubernetes。K8s 不是<b>yaml 写得更复杂的 Compose</b>,而是把单机部署升级成集群编排:它解决的是 Compose 给不了的能力,包括<b>声明式调度、自愈、多副本、GPU 资源管理、滚动升级零停机</b>。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/课程封面-4c5985fd.png" width=80%></div>

&emsp;&emsp;本课同样使用一台 4× RTX 3090 24GB 的 GPU 服务器,重点理解 K8s 编排 GPU 推理服务的方式。整门课用同一个文搜图项目作为叙事锚点,每讲一个 K8s 概念都尽量落回项目对应位置,避免陷入"yaml 字段背诵";第四章会把 GPU 推理服务单独拎出来做专章,这是很多 K8s 入门课较少系统展开、但 AI 工程师很容易在真实项目里遇到的部分。

&emsp;&emsp;<b>关于实操环境</b>:本课不在 Mac / Windows 本机装 K8s。Mac/Win 上能用 Docker Desktop 内置 K8s / minikube / kind / k3d 这类<b>本地开发调试工具</b>装一套迷你集群,适合在本机验证 yaml 语法;但<b>生产环境的 K8s 几乎全跑在 Linux 上</b>——要么裸金属服务器,要么云厂商托管（AWS EKS / Azure AKS / Google GKE / 阿里云 ACK）。AI 推理服务尤其如此:NVIDIA GPU driver / CUDA / Device Plugin 这套都依赖 Linux——Mac Apple Silicon 没有 CUDA,Windows GPU 走 WSL2 能跑但生产几乎不这样部署。

&emsp;&emsp;本课直接对齐生产形态,所有 K8s 实操都在一台 Linux GPU 服务器上做,Mac/Win 那侧只起一个 ssh 终端工具角色——课件里产出的 yaml 和命令可以原样落到任何 Linux 集群上跑（裸金属 / 自建 / 云厂商 K8s 服务都行）。

&emsp;&emsp;本课承接《容器化部署 Docker + Compose》,大家只需要能读懂 Dockerfile、docker-compose,知道容器和镜像的区别。

---

## <center>第一章 K8s 概念和组成</center>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_第一章学习路径-9786a9e8.png" width=80%></div>

&emsp;&emsp;K8s 跟 Docker Compose 的差别不是"换个编排工具",而是换一种用集群的方式:我们不再说"去启动一个 nginx",而是说"我期望集群里始终有 3 个 nginx",剩下的让 K8s 持续对齐。本章先用一张回环图讲清<b>声明期望 + 持续对齐</b>这条核心机制,再拆开工作节点和控制面两层结构,最后落到 Service / Ingress / ConfigMap / Secret、无状态与有状态工作负载这些日常 yaml 里的高频对象,完成从单机部署思路到集群编排思路的切换。

### 1.1 K8s 全景

&emsp;&emsp;<b>Kubernetes（简称 K8s）是 Google 2014 年开源的容器编排系统</b>,设计原型源自 Google 内部跑了十年的 Borg 集群管理平台,目前由 CNCF（云原生计算基金会）托管,是云原生时代生产级容器编排的事实标准。"K8s"这个简写是把 Kubernetes 首尾的 K 和 s 留下、中间 8 个字母 ubernete 缩成数字 8 的写法,读音不变。

&emsp;&emsp;打一个我们熟悉的对比就清楚了:<b>Docker Desktop 是让我们在本机轻松跑容器的工具,更像"容器开发小助手";K8s 是"容器调度指挥官"</b>——它能管理成百上千台机器上的容器应用,把部署、扩缩、自愈等复杂任务自动化。前一门课我们用的 `docker compose` 已经是"指挥官"的雏形——能在单机上编排多个容器协作,带<b>进程级自愈</b>（`restart: always` 容器退出 docker daemon 自动重启）。K8s 在 compose 基础上补齐三件 compose 给不了的能力:<b>跨节点调度</b>（N 台机器统一编排）、<b>自动扩缩</b>（副本数动态调节、HPA）、<b>更强的自愈</b>——`livenessProbe` 失败自动重启容器、`readinessProbe` 失败自动从 Service 摘流量、副本数偏离自动补齐、节点宕机自动跨机迁移,而 compose 的 `restart` 只管容器进程退出。补齐这三件,正式跨入生产级。

&emsp;&emsp;<b>K8s 调度的资源范围也在演进</b>。早期 K8s 只调度 CPU/内存——pod yaml 里写 `resources.requests.cpu: 2 / memory: 4Gi`,scheduler 按节点剩余资源做 bin-packing,天然不认识 GPU。2017 年 NVIDIA 推出 <b>Device Plugin</b> 扩展机制,把宿主机 GPU 注册成 `nvidia.com/gpu` 扩展资源,scheduler 第一次能"看到"GPU 卡数,但调度粒度只到"整张卡 = 一个整数"。

&emsp;&emsp;<b>2025 年 8 月 K8s 1.34 把 DRA（Dynamic Resource Allocation）核心升级到 GA</b>,pod <b>可以不再</b>用传统 `nvidia.com/gpu: 1` 整卡申请,改成通过 DeviceClass（如 `gpu.nvidia.com`）上的 ResourceClaim 申请满足条件的设备,可按显存大小 / 算力等级 / 拓扑位置精细分配。2026 年 3 月 KubeCon Europe（阿姆斯特丹）上 NVIDIA 把 GPU DRA Driver 捐给 CNCF、KAI Scheduler 进 CNCF Sandbox——K8s 正式迈向 <b>GPU-aware 调度</b>时代,Google Cloud 和 Amazon EKS 也已开始支持 DRA。<b>但传统 Device Plugin 并没有被替代</b>:DRA 需要集群版本、DRA driver、云厂商都配齐才能用（例如 AWS EKS 的 Karpenter / Auto Mode 当前仍只支持 Device Plugin）,本课这套 3090 + k3s + Device Plugin 实验仍按"整卡调度"理解最稳。

&emsp;&emsp;<b>市场数据</b>:CNCF 调查显示 <b>66% 的组织已在 K8s 上跑生成式 AI 推理</b>,Kubernetes for AI workloads 市场规模预计 2034 年达 226 亿美元（CAGR 18.8%）。本课第四章会专门拆 K8s 编排 GPU 推理的关键能力——这就是 K8s 当下最热的延伸方向,也是 AI 工程师在生产环境绕不开的话题。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_K8s集群中心放射图-201b5f6d.png" width=80%></div>

&emsp;&emsp;先用<b>一句话抓住核心</b>,K8s 跟 Compose 最大的不同藏在这一句里:<b>我们写"我期望集群长什么样",K8s 不停循环对齐"实际"到"期望"</b>。我们不说"去启动一个 Pod",我们说"我要 3 个 frontend Pod 一直在线"。然后整个集群就开始一件事:不停检查"现在到底有几个、跟期望差多少",差了就补,多了就杀,Pod 挂了立刻重建——这种<b>"声明期望 + 持续对齐"的循环</b>叫 <b>reconcile loop</b>,是 K8s 一切机制的底层节奏。

&emsp;&emsp;再讲<b>谁在循环里干活</b>。集群里只分两类角色,分工像一家工厂:

- <b>控制面（Control Plane</b>）:只决策不干活——决定"应该有几个 Pod""哪个 Pod 落哪台机器""谁来对齐",但自己不跑业务容器。

- <b>工作节点（Worker Nodes</b>）:只干活不决策——管理层指哪打哪,容器真正跑在这里,我们的代码最终在工厂的某一台机器的某一个 Pod 里执行。

&emsp;&emsp;一句话记住:<b>K8s 集群只分这两层</b>。控制面负责决策,工作节点负责执行。我们写的所有 yaml 最终都会变成"控制面的一条期望记录 + 工作节点上的一个运行实例"。

&emsp;&emsp;<b>下两节的展开顺序</b>:1.2 节先展开工作节点——因为前一章 Docker 我们已经熟悉了"容器跑在机器上",工作节点是离我们最近的一层;1.3 节再回到控制面,看 reconcile loop 在工程上具体是谁让它转起来的。1.4 节起进入日常 yaml 里高频出场的资源对象。

### 1.2 工作节点 Worker Nodes

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_工作面全景-96fbe6df.png" width=80%></div>

&emsp;&emsp;工作节点是 K8s 集群里真正干活的地方,承接前一章 Docker——容器从"跑在我们笔记本上"变成"跑在集群里某台 Node 的某个 Pod 里"。这一节把工作节点上的<b>四样东西一次讲到位</b>:<b>Node</b>（机器载体）、<b>Pod</b>（K8s 最小调度单位）、<b>kubelet</b>（节点小管家）、<b>kube-proxy</b>（节点 iptables 规则录入器）,重点放在职责、作用和常见坑上,后面章节再提到这四个名词时不再重复展开。

> <font size=2><b>【名词解释】</b><font color=red><b>Pod</b></font>:K8s 的调度最小单位,包含 1 个或多个共享网络和存储的容器。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>kubelet</b></font>（K8s 节点代理）:每个节点上的本地代理进程,负责跟 api-server 通信、拉容器、做健康检查。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>kube-proxy</b></font>（K8s 节点 iptables 规则录入器）:每个节点上的 systemd 进程或 DaemonSet pod,<b>不参与数据包转发本身</b>——它只 watch api-server 上的 Service / EndpointSlice 变化,把变化翻译成 iptables（或 ipvs）规则写到节点内核里。真正负责转发数据包的是 <b>Linux 内核 netfilter</b> 子系统,数据流完全在内核态完成。kube-proxy 进程在节点没有规则变更时基本"睡觉",只在 Service / EndpointSlice 有增删改时被唤醒去刷规则。</font>

#### 1.2.1 Node 机器载体

&emsp;&emsp;<b>Node 是 K8s 集群里的一台机器</b>——物理机 / 虚拟机均可。它是 Pod 的真实运行载体。`kubectl get nodes` 列出集群所有节点的状态（Ready / NotReady / SchedulingDisabled）。

&emsp;&emsp;<b>Node 自己也是 K8s 资源对象</b>。节点要参与调度——scheduler 调度 Pod 时,会读所有 Node 的资源容量、label、taint;运维想下线某台机器做硬件维护,要 `kubectl drain` 把它上面的 Pod 迁走再 cordon——这些操作都把 Node 当成资源对象来读写。

#### 1.2.2 Pod K8s 最小调度单位

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_Pod最小调度单位-096ad4d9.png" width=80%></div>

&emsp;&emsp;<b>Pod 是 K8s 的调度最小单位</b>——一个 Pod 包含 1 个或多个紧耦合容器,<b>共享同一个网络命名空间和存储卷</b>。Pod 内的容器之间用 `localhost` 互访（共享 loopback）,但跨 Pod 必须用 Service DNS 名——这是 1.4 节 Service 会反复强调的易错点。

&emsp;&emsp;<b>Pod 作为最小调度单位,解决的是紧耦合容器同进同退的问题</b>。有些场景需要两个容器共享网络并一起启动、一起销毁——比如主应用 + 日志收集 sidecar、主应用 + 服务网格 proxy。把它们放在同一个 Pod 里调度,K8s 保证它们落在同一节点、共享 localhost 和 emptyDir 卷。<b>本课文搜图项目 5 个服务都是单容器 Pod</b>,我们先建立"1 Pod = 1 Container"的简单模型,知道 Pod 能装多个容器即可。

&emsp;&emsp;<b>容易踩的坑</b>:<b>Pod IP 不稳定</b>。Pod 不是长寿命对象——`kubectl delete pod xxx` 立刻销毁,下次 controller 重建的是一个全新 Pod,IP 重新分配。所以集群内访问<b>从来不用 Pod IP</b>,永远用 Service DNS 名——这正是 1.4 节 Service 这一对象要解决的核心问题,我们到了 1.4 节会回头收掉这个坑。

#### 1.2.3 kubelet 节点小管家

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_kubelet节点小管家-cfbe0603.png" width=80%></div>

&emsp;&emsp;<b>kubelet 是每个节点上的本地代理进程</b>,周期性从 api-server 获取分配给本节点的新 Pod 和清理任务,拿到任务就调 containerd 拉镜像、起容器、跑健康检查、把 Pod 实际状态汇报回 api-server。

> <font size=2><b>【名词解释】</b><font color=red><b>containerd</b></font>（高级容器运行时,读 "container-d",d 是 daemon 守护进程的意思）:Docker 公司 2017 年捐给 CNCF,2019 年毕业的容器运行时（Container Runtime,K8s 官方中文文档 / 知乎 / 阿里云 / 腾讯云 主流统一译"容器运行时",不是"容器进行时"）。containerd 实现了 <b>CRI</b>（Container Runtime Interface,容器运行时接口）——kubelet 通过 CRI 调它起容器。<b>分层</b>:containerd 是"高级运行时"（管镜像传输 / 容器生命周期 / CRI 通信）,底层调用更低级的 OCI runtime（`runc` 标准 Linux 容器 / `crun` C 语言实现 / `nvidia-container-runtime` GPU 容器）真正创建容器进程。本课 k3s 把 containerd 直接嵌进 super-binary,socket 在 `/run/k3s/containerd/containerd.sock`,跟宿主机 docker daemon 是两套独立运行时（2.3 节配 mirror 时讲过）。</font>

&emsp;&emsp;<b>kubelet 承担节点本地执行</b>。控制面只决策不干活,Pod 要真的被启动起来,需要一个节点本地的执行者——kubelet 就是这个角色。<b>kubelet 不直接调用其它组件</b>,它只跟 api-server 通信:从 api-server 读"绑定到我节点的 Pod 列表",把容器状态写回 api-server。这种"中心化 + 事件驱动"的设计让 kubelet 即使重启也不丢状态,所有信息都在 etcd 里。

&emsp;&emsp;kubelet 本身<b>不是 Pod,是节点上的系统进程</b>。标准 K8s 把 kubelet 作为 systemd 服务跑在每个节点上;k3s 把 kubelet 合并到了 `k3s server / agent` 进程里——server 节点也跑 kubelet/kube-proxy 功能,agent 节点负责跑业务容器,第二章会展开 k3s 跟生产 K8s 在组件部署形态上的差异。

#### 1.2.4 kube-proxy 节点 iptables 规则录入器

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_kube-proxy规则录入器-7b378a03.png" width=80%></div>

&emsp;&emsp;<b>kube-proxy 名字带 "proxy"（代理）,但它实际<u>不转发数据包</u></b>——它做的事是把 Service 和 EndpointSlice 的变化翻译成 Linux 内核里的 iptables（或 ipvs）规则,然后就"睡"了。<b>真正转发数据包的是 Linux 内核 netfilter 子系统</b>:数据包到达节点网卡,内核按 kube-proxy 写好的规则做 DNAT（把 Service ClusterIP 改写成具体 Pod IP）,整个过程完全在内核态完成,跟 kube-proxy 用户态进程无关。所以 kube-proxy 更准确的角色描述是<b>"iptables 规则录入器"或"netfilter 规则编程器"</b>。

&emsp;&emsp;<b>它的真实工作循环流程</b>:① watch api-server 上的 Service / EndpointSlice 资源;② 看到变化（新 Service 加进来 / Pod 数变化 / Pod 重建 IP 变了）时唤醒;③ 把变化转成 iptables 命令,在节点本地编程内核 netfilter;④ 编程完睡回去等下一次变化。<b>它不知道任何具体数据包</b>,也不参与数据转发——把内核当成它的代码运行环境就好。

&emsp;&emsp;<b>具体一个请求的数据路径</b>:集群里任何 Pod 访问 `http://frontend:80` 时,CoreDNS 把 `frontend` 解析到 Service 的 ClusterIP（虚拟 IP,不存在于任何机器网卡上）;数据包到达本节点网卡 → Linux 内核 netfilter 按 kube-proxy 之前写好的 iptables 规则匹配 ClusterIP → DNAT 改写目标地址到某个后端 Pod 的真实 IP → 包直达目标 Pod。<b>整个数据路径上没有 kube-proxy 进程参与</b>,负载均衡完全发生在内核态。Service 也只是个虚拟 IP + 一组 iptables 规则的匹配条件,本身不参与数据包流转。kube-proxy 工作在 <b>L4 四层</b>层面（规则按 IP+端口写）,不解 HTTP——七层路由是 Ingress Controller 的活,1.4 节展开。

&emsp;&emsp;<b>k3s 的 kube-proxy 特殊行为</b>:标准 K8s 把 kube-proxy 作为 DaemonSet Pod 跑在每个节点;<b>k3s 把 kube-proxy 的功能合并到了 k3s-agent 进程里</b>,不再有独立 kube-proxy Pod。意味着 `kubectl get pods -n kube-system` 看不到 kube-proxy Pod——这是<b>正常现象不是 bug</b>,功能上 Service ClusterIP 仍然正常工作。如果照着标准 K8s 示例去找 kube-proxy Pod,这里很容易误判。

#### 1.2.5 一句话总结工作节点

&emsp;&emsp;<b>应用不是跑在 api-server 上、也不是跑在 scheduler 上,应用最终是跑在 Node 里的 Pod 上</b>。每个 Node 上还有两个本地代理:kubelet 启动并管理容器,kube-proxy 编程 iptables 规则让 Service 流量能落到 Pod——真正转发数据的是内核 netfilter,kube-proxy 只负责写规则。下一节我们看控制面——是谁决定 Pod 长啥样、跑哪台机器、副本数变化谁来对齐。

### 1.3 控制面 Control Plane

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_控制面三件套-4d881266.png" width=80%></div>

&emsp;&emsp;看完了业务运行的位置,这一节看控制面如何决定业务的目标状态和调度结果:Pod 落到哪个 Node、Deployment 副本数变化时由谁创建新 Pod、Pod 挂掉后由谁补回来,这些"决策和管理"全部是控制面的活。控制面 4 件套:<b>api-server</b>（总枢纽）、<b>etcd</b>（集群记账本）、<b>scheduler</b>（调度决策者）、<b>controller-manager</b>（巡检对齐器）。我们<b>把 api-server 单独拎出来优先讲</b>——因为所有其它组件都通过它协作,先讲它后面才说得清楚。

> <font size=2><b>【名词解释】</b><font color=red><b>api-server</b></font>（API 服务器）:K8s 集群唯一对外入口和写入 + 广播枢纽。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>scheduler</b></font>（K8s 调度器）:决定每个 Pod 落到哪个 Node 上的控制面组件。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>controller-manager</b></font>（控制器管理器）:跑着几十个内置 controller 的进程,每个 controller 都是一个 reconcile loop。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>reconcile loop</b></font>（调谐循环）:"对齐期望与实际状态"的后台循环,是一切 controller 的工作模式。</font>

#### 1.3.1 api-server 所有组件之间的唯一通道

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_api-server请求流水线-49fa6854.png" width=80%></div>

&emsp;&emsp;<b>api-server 是控制面里地位最特殊的一个</b>——<b>所有组件之间都不直接通信,全部通过 api-server 协作</b>。kubectl 找它、scheduler 找它、controller-manager 找它、kubelet 也找它,大家不是互相乱连,而是都通过它读写集群状态。一个很关键的工程属性:<b>api-server 本身是完全无状态的</b>——它把所有记忆交给了 etcd。因此生产环境 api-server 可以横向扩展多副本且不会脑裂:状态全在 etcd,api-server 只是 etcd 前面的"流水线 + 广播器"。

&emsp;&emsp;上图把 api-server 内部处理流水线画清楚了,我们按"<b>写请求</b>"和"<b>读请求</b>"分开看——两者前 3 步一样,后 2 步不同:

1. <b>客户端请求</b>:kubectl / kubelet / controller 通过 HTTPS REST 把请求发过来——可能是 create/update/delete（写）,也可能是 get/list/watch（读）。

2. <b>认证（Authentication）+ 授权（Authorization）</b>:先验身份（谁来的）,再查 RBAC 权限（允不允许干这件事）。

3. <b>Admission 准入控制</b>:内置准入器 + 自定义 Webhook 在请求落库前做最后一道关——注入默认值、查配额（ResourceQuota）、查安全策略（PodSecurity）。

4. <b>写请求落 etcd</b>:校验通过后才把对象写到 etcd 作为新的"期望状态";<b>读请求</b>则走 api-server 的读取/watch 通道（直接从 etcd 取,或从 api-server 的 watch cache 取）。

5. <b>watch 广播订阅者</b>（写请求触发）:任何对象变化都会通过长连接通知正在 watch 的组件——scheduler 看到 Pending Pod 就开始调度,controller-manager 看到副本数变化就建 Pod,kubelet 看到"绑到我的 Pod"就拉镜像。

&emsp;&emsp;一句话记住:<b>api-server 是 K8s 集群里大多数跨组件通信的中枢——绝大部分"想知道集群状态、想宣告自己干了啥"的事件都走它</b>——少量例外比如 kubelet 直接调 container runtime / CNI 插件、节点本地的健康探针、组件间偶尔走 etcd 锁做 leader 选举。这种"中心化 + 事件驱动"设计是 K8s 在工程上最值得学的一件事——它让组件间的协作变成了"事件驱动 + 状态对齐",而不是"组件 A 直接 RPC 调用组件 B"。

#### 1.3.2 三件管事的 etcd / scheduler / controller-manager

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_三件管事_etcd_scheduler_controller_manager-194e18c7.png" width=80%></div>

&emsp;&emsp;api-server 是"通道",真正决策和持久化的是另外三件:

- <b>etcd（集群记账本）</b>:基于 Raft 共识的分布式 KV 数据库,K8s <b>唯一持久化层</b>。所有 yaml 提交后变成的"期望状态"（应该有几个 Pod、Service 长啥样、ConfigMap 里写了啥）全部存这里——其它组件挂了重启都不丢状态。生产环境 etcd 一般 3 或 5 副本做高可用,本课 k3s 默认用嵌入式 SQLite 替代,职责完全等价。

- <b>scheduler（调度决策者）</b>:对每个新创建的 Pending Pod 跑<b>两阶段决策</b>——先 filter 过滤掉资源不够、节点选择器不匹配、被 Taint 排斥的节点,再 score 给候选节点打分挑最优,最后把 Pod 的 `nodeName` 字段绑定到选中的 Node。绑定完 scheduler 的活就结束了,剩下的交给目标节点的 kubelet。

- <b>controller-manager（巡检对齐器）</b>:跑着<b>几十个内置 controller</b>（Deployment Controller / ReplicaSet Controller / Endpoints Controller / Node Controller ...）,每个都是一个 reconcile loop——周期性比较"期望状态 vs 实际状态",发现不一致就推动系统修正。<b>1.1 节的 reconcile loop 机制,实际就靠 controller-manager 里这几十个 loop 在转</b>。

&emsp;&emsp;这三件管事的全部跑在控制面节点上,工作节点上不需要装它们;生产环境一般把控制面做 3 副本以上的高可用。

#### 1.3.3 reconcile loop 落地实例 Deployment Controller + ReplicaSet Controller

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_Deployment三层结构-da16cd69.png" width=80%></div>

&emsp;&emsp;1.1 节我们立住了 reconcile loop 这条主线（"声明期望 + 持续对齐"）,controller-manager 这一段就是这条主线的工程落地。<b>挑两个最典型的 controller 展开</b>:Deployment Controller 和 ReplicaSet Controller。

&emsp;&emsp;<b>整个机制的转动过程</b>:用户写 yaml 声明"我要 3 个 frontend 副本" → Deployment Controller watch 到新 Deployment 对象,创建一个 ReplicaSet → ReplicaSet Controller watch 到新 ReplicaSet,发现"期望 3 个 Pod,实际 0 个",创建 3 个 Pending Pod。<b>更新镜像</b>时 Deployment 创建一份新 ReplicaSet（新副本从 0 递增）+ 缩老 ReplicaSet（从 3 递减到 0）,滚动过程中始终保证有可服务的 Pod。整个过程没有任何"主动调用",每个 controller 都只是 watch 自己关心的对象、对齐期望和实际、把结果写回 etcd——这就是 reconcile loop 在工程上的样子。

#### 1.3.4 kubectl apply 7 步链路 把控制面 + 工作节点串起来

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_kubectl_apply完整链路-b5693fd5.png" width=80%></div>

&emsp;&emsp;到这里控制面 4 件套加上 1.2 节工作节点 2 件套就齐了,我们用一行 `kubectl apply -f deployment.yaml` 把它们全串起来——这是 K8s 集群里发生频率最高的事,把整张全景变成一条可追踪的因果链:

1. 我们写好 `deployment.yaml`,声明"想要 3 个 frontend Pod"。

2. `kubectl apply -f` 把 yaml 发给 <b>api-server</b>。

3. api-server 走完认证、授权、admission 准入,把 Deployment 对象<b>写入 etcd</b>——这时 etcd 里就多了一份"期望状态"。

4. <b>controller-manager</b> 里的 Deployment Controller watch 到新对象,创建一个 ReplicaSet 对象;ReplicaSet Controller 再 watch 到,创建 3 个 Pending Pod。

5. <b>scheduler</b> watch 到 Pending Pod,跑 filter + score 两阶段决策,把 Pod 的 `nodeName` 字段绑定到选中的工作节点。

6. 选中节点上的 <b>kubelet</b> watch 到"这个 Pod 是我的",调 containerd 拉镜像、起容器,把 Pod 状态汇报回 api-server——<b>Pod Running</b>。

7. 如果这组 Pod 已有匹配的 Service,<b>EndpointSlice Controller</b> 根据 Pod Ready 状态更新 EndpointSlice 对象;节点上的 <b>kube-proxy</b> watch 到 EndpointSlice 变化,把新规则写进内核 netfilter,让"用 Service 名访问"的请求能落到这个新 Pod 上。要注意纯 Deployment apply 不会自动创建 Service,Service 得单独 yaml 声明。

&emsp;&emsp;这条链路里有<b>三件值得反复强调的事</b>:① 我们提交的是<b>期望状态</b>（我想要 3 个 Pod）,不是命令（去创建 Pod）;② 所有组件都通过 <b>api-server 读写 etcd</b> 协作,组件间从不直接通信;③ 整个过程是<b>事件驱动 + 持续 reconcile</b>——Pod 挂了 ReplicaSet Controller 会立刻看到"实际 ≠ 期望"再建一个,不需要任何人手工介入。

> 📌 <b>读到这里如果对 EndpointSlice / iptables / admission / reconcile 这些陌生名词感到困惑,完全不用担心</b>。本节的目标是让我们看清"从一行 `kubectl apply` 到容器跑起来"的整体流转方向——不靠组件直连,全靠对齐 etcd 期望状态。具体每个资源对象（Service / Ingress / ConfigMap / Secret / StatefulSet）,1.4-1.5 节会一个个拆开讲。

&emsp;&emsp;<b>下两节的路径</b>:1.4 节集中讲日常 yaml 高频出场的 4 个资源对象——Service / Ingress / ConfigMap / Secret;1.5 节讲无状态服务（Deployment yaml 写法） vs 有状态服务（StatefulSet + PVC + headless Service）。1.6 节用一张图收束,Compose 跟 K8s 在编排方式上有哪 5 点根本差异。

### 1.4 其他常用资源对象

&emsp;&emsp;<b>资源对象是 K8s 用 yaml 声明的所有非工作负载组件</b>——除了 Pod / Deployment / StatefulSet 这些直接跑业务容器的工作负载,K8s 还有一堆 yaml 对象负责"把流量送进 Pod"、"把配置注入 Pod"这类配套职责。日常 yaml 里高频出场的 4 个是:<b>Service / Ingress / ConfigMap / Secret</b>。其中 <b>Service / Ingress</b> 负责把流量送进 Pod——正好对接 1.2 节埋下的"Pod IP 不稳定"那个坑,<b>ConfigMap / Secret</b> 负责把配置送进 Pod。本节把每个对象的职责、作用和常见坑讲清楚。

> <font size=2><b>【名词解释】</b><font color=red><b>ClusterIP</b></font>（默认 Service 类型）:分配虚拟 IP 只在集群内可达。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>Ingress Controller</b></font>（Ingress 控制器）:真正执行路由的网关进程（Traefik / nginx-ingress）。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>Traefik</b></font>:Traefik Labs 维护的开源 L7 反向代理 / Ingress Controller, Go 语言写, 原生支持 K8s Ingress / Gateway API / CRD Middleware, 配置走 yaml + 自动 watch 集群变更——k3s 默认内置, 集群起来 `ingressClassName: traefik` 直接能用; 在 K8s 社区被广泛使用的 Ingress Controller 之一, 与 nginx-ingress 同为常见选择;k3s 默认内置 Traefik, 本课所有 Ingress yaml 均以此为准。</font>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_4_四件资源对象全景-08db7721.png" width=80%></div>

#### 1.4.1 Service 解决 Pod IP 不稳定

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_Service与Ingress流量路径-44d61f1e.png" width=80%></div>

&emsp;&emsp;<b>Service 是 K8s 里集群内服务发现的核心抽象</b>。它给一组 Pod 提供:固定的 ClusterIP、固定的 DNS 名、自动负载均衡——解决 1.2 节末尾埋下的那个坑:Pod IP 不稳定,集群内访问绝不能用 Pod IP,要有一个稳定入口。

&emsp;&emsp;<b>核心机制</b>:Service 通过 `selector` 选中所有符合 label 的 Pod → K8s 自动维护一个 <b>EndpointSlice 对象</b>记录这些 Pod 的 IP 列表 → 每个节点上的 <b>kube-proxy</b> watch EndpointSlice 变化,把规则编程进内核 netfilter（iptables/ipvs）→ 集群内任何 Pod 访问 `http://service-name:port` 时,DNS 解析到 ClusterIP（虚 IP）,数据包到节点后由内核 netfilter 按规则做 DNAT 改写到某个具体 Pod IP,包就直达后端。<b>label + selector 是 Service 跟 Pod 之间的胶水</b>——不靠"硬编码 Pod IP",靠 label 匹配自动维护后端列表,Pod 增减都自动反映在 EndpointSlice 里,kube-proxy 跟着刷规则。

&emsp;&emsp;<b>4 种 Service type 在本课的选型</b>:5 个服务之间的内部调用全用 <b>ClusterIP</b>（默认,集群内可达就够）;frontend 对外暴露走 <b>Ingress + ClusterIP</b>（下面 Ingress 段展开）;<b>NodePort</b> 只在调试时偶尔用（从宿主机直接打节点 IP+大端口验证 Service 通不通）;<b>LoadBalancer</b> 是云厂商场景（AWS/GCP 自动分配公网 IP）。

#### 1.4.2 Ingress 七层 HTTP 入口

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_Ingress资源拓扑-5eb07499.png" width=80%></div>

&emsp;&emsp;<b>Ingress 是集群外 HTTP 入口路由对象</b>——按域名/路径把外部请求路由到不同 Service。Ingress 跟 Service 是两个独立的资源对象,<b>不要混淆</b>:Service 解决 L4 四层"用 ClusterIP 找 Pod",Ingress 解决 L7 七层"用 `wensoutu.local/` 路由到 frontend Service"。

&emsp;&emsp;<b>Ingress 跟 Ingress Controller 的协作关系</b>:Ingress yaml 只写规则——`wensoutu.local` 的 `/` 路径请求转发到 `frontend` Service。规则真正能跑起来,要靠集群里的 <b>Ingress Controller</b>（Traefik / nginx-ingress / Contour）——它是一个真实的网关进程,watch 所有 Ingress 对象,把规则配置到自己的转发表上,外部流量打到 Ingress Controller 再落到目标 Service。

&emsp;&emsp;<b>本课最容易踩的坑</b>:k3s 默认 Ingress Controller 是 <b>Traefik</b>,不是 nginx-ingress。所以 Ingress yaml 必须写 `ingressClassName: traefik`——很多通用示例里会写 nginx,直接照搬会导致 Ingress 不工作。第三章 3.8 节我们会动手写 frontend 的 Ingress yaml,会反复强调这一点。

&emsp;&emsp;<b>Ingress 跟继任者 Gateway API 的现状</b>:K8s 社区从 2023 年起推出 <b>Gateway API</b>（`gateway.networking.k8s.io/v1`）作为 Ingress 的继任者,2023-10 v1.0 GA,2026-02 已经到 v1.5——Istio / NGINX / Traefik / AWS Load Balancer / Cilium 等 20+ controller 全部支持。这件事 2025-2026 年迎来两个关键节点:

- ① <b>Ingress API 本身没死,但 feature-frozen</b>——你写的 Ingress yaml 仍能跑,K8s 也仍会维护,但<b>不再加新功能</b>,所有路由能力创新都在 Gateway API。

- ② <b>ingress-nginx Controller 已被官方退役</b>:K8s SIG-Network 2025-11-11 公告,2026-03-31 EOL,之后<b>无安全补丁 / 无 bugfix / 无新版本兼容</b>。这是行业最广泛使用的 Ingress 实现,影响面巨大。

&emsp;&emsp;<b>本课为什么仍用 Ingress + Traefik</b>:① Traefik <b>同时支持 Ingress 和 Gateway API</b>,且 k3s 默认就装它,不受 ingress-nginx 退役影响;② Ingress 的概念模型更简单（yaml 一个对象就是规则集）,作为 K8s 路由入门第一站学习成本低;③ Ingress 在存量集群里仍是事实主流,你出门工作大概率先遇到 Ingress yaml;④ 学完 Ingress 再学 Gateway API 是<b>平滑升级</b>——核心概念（规则 + Controller 真正执行）一致,主要差异在 yaml 字段拆分——Gateway API 用 GatewayClass / Gateway / HTTPRoute 三对象代替 Ingress 一对象,更清晰地区分平台运维和应用开发的职责边界。

&emsp;&emsp;<b>生产新项目选型建议</b>:① 已有 Ingress 集群——保持现状,Ingress API 没死,但<b>如果用 ingress-nginx 必须计划迁移</b>（改用 Traefik / Istio / NGINX Gateway Fabric / Envoy Gateway 等）;② 新建集群——K8s 官方推荐 <b>Gateway API 作为默认选择</b>;③ 同集群可以共存——Ingress 和 Gateway API 不冲突,允许渐进迁移。

#### 1.4.3 ConfigMap 非敏感配置注入

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_ConfigMap注入方式-78db8f36.png" width=80%></div>

&emsp;&emsp;<b>ConfigMap 把端口、URL、特征开关、日志级别等明文配置从镜像里抽出来</b>,集中存放在 K8s 集群里,Pod 启动时从 ConfigMap 拉这些配置作为环境变量或者挂载成文件。文搜图项目的 backend 配置（milvus 地址、模型名、温度参数、日志级别等十来个 key）集中放一个 ConfigMap 里。

&emsp;&emsp;<b>ConfigMap 负责把配置和工作负载解耦</b>。配置直接写在 Pod yaml 里也能跑,但每改一次配置就要改 Deployment yaml 重新 apply;抽成 ConfigMap 后,我们可以只改 ConfigMap,然后只重启依赖它的 Pod。两种注入方式:`envFrom: configMapRef` 把所有 key 自动变环境变量;`volumeMounts` 把每个 key 变独立文件挂到容器某个目录。

&emsp;&emsp;<b>容易踩的坑</b>:ConfigMap 修改后<b>不会自动重载</b>到运行中的 Pod——Pod 启动时把环境变量值快照下来,后续 ConfigMap 改了也不刷新。要让新值生效必须重启 Pod（`kubectl rollout restart deployment xxx`）。这是 AI 工程师从 Compose 切过来常错的地方,第三章 3.6 节会展开。

#### 1.4.4 Secret 敏感配置注入

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_Secret引用方式-57f55a6c.png" width=80%></div>

&emsp;&emsp;<b>Secret 是 K8s 的敏感配置注入对象</b>——API key、数据库密码、TLS 证书等敏感数据走这里。写法跟 ConfigMap 几乎一致,区别只在 Secret 做 <b>base64 编码</b>存储。

&emsp;&emsp;<b>关键概念澄清</b>:<b>base64 不是加密,只是编码,谁都能 decode</b>——`echo "c2VjcmV0" | base64 -d` 就还原了。Secret 的"安全"来自 K8s 的 <b>RBAC 权限控制</b>（Role-Based Access Control,K8s 内置权限模型——给账号绑定 Role 控制能操作哪些资源,1.3 节 api-server 路径里简短提过）,不是编码本身:默认 namespace 用户能 `kubectl get configmap` 但不能 `kubectl get secret`,集群管理员可以精确控制"谁能查 secret"。<b>ConfigMap 和 Secret 的真正差别在"谁能访问",不在写法</b>——这也是二者拆成两个对象的根因。

&emsp;&emsp;<b>Secret 的使用场景</b>:文搜图项目里走 Secret 的环境变量是 <b>OPENROUTER_API_KEY</b>——本课用 OpenRouter 代理 OpenAI 模型,这个 key 泄露出去就直接被刷账单。其余非敏感配置都走 ConfigMap。<b>生产加密方案</b>有外挂 <b>HashiCorp Vault</b> 或 <b>Bitnami sealed-secrets</b>,具体配置查官方文档。第三章 3.6 节我们会动手写文搜图项目的 Secret yaml。

### 1.5 无状态服务 vs 有状态服务

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_StatefulSet对比Deployment-aa89c775.png" width=80%></div>

&emsp;&emsp;K8s 把工作负载分成两条独立路径。一条是<b>无状态服务</b>,每个副本完全可互换,挂一个起一个无所谓;另一条是<b>有状态服务</b>,像数据库、向量库、对象存储,每个副本有自己的身份和数据,不能随便互换。两条路径在 yaml 上对应两个不同对象:<b>Deployment</b>（无状态）和 <b>StatefulSet + PVC + PV + StorageClass + headless Service + initContainer</b>（有状态）。本节把这两条路径一次性讲清,后面第三章 3.3-3.5 节实战有状态（StatefulSet）,3.7/3.8 节实战无状态（Deployment）,1.5 节讲完概念后不再重复对象本身。

> <font size=2><b>【名词解释】</b><font color=red><b>Deployment</b></font>（部署）:K8s 工作负载对象,声明式管理一组 Pod 副本的生命周期,无状态服务标配。底层用 ReplicaSet 维持副本数 + rolling update 滚动更新策略——日常用户操作 Deployment 字段即可,ReplicaSet 是实现细节不直接动。HPA（水平自动扩缩）是另一个 controller 按 CPU/内存自动调副本数,本课 GPU 推理场景用不上,第四章 4.1 节详细讲为什么。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>StatefulSet</b></font>（有状态集）:K8s 工作负载对象,给 Pod 分配稳定序号 + 稳定 DNS 名 + 持久卷绑定。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>PVC</b></font>（PersistentVolumeClaim,持久卷申领）:声明"我要一块持久存储"。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>PV</b></font>（PersistentVolume,持久卷）:真实的存储卷资源（NFS / 云盘 / 本地盘）,通常由 <b>StorageClass</b> 按规则自动 provision 出来。本课用 k3s 默认 `local-path` StorageClass,PV 落在节点本地盘 `/var/lib/rancher/k3s/storage/`。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>headless Service</b></font>（无头服务）:字段 `clusterIP: None` 的 Service。跟普通 ClusterIP Service 的差别在于<b>没有虚拟 IP</b>——普通 Service 把请求转给一个虚拟 IP 再由 kube-proxy 做四层负载;headless Service 不分配虚拟 IP,DNS 直接解析到背后所有 Pod 的真实 IP。这样 StatefulSet 每个 Pod 都拿到一个稳定 DNS 名,格式是 `{pod-name}.{service-name}.{namespace}.svc.cluster.local`。<br><br>举例:本课第三章 wensoutu namespace 里 etcd StatefulSet 配的 headless Service 叫 `etcd`,本课用单副本（`replicas: 1`）,Pod 拿到稳定名 `etcd-0.etcd.wensoutu.svc.cluster.local`——milvus 启动时直接连这个 DNS 名,Pod 重建后 IP 可以变、DNS 名不变,milvus 不需要改配置。多副本场景（例如生产 etcd 3 副本）更能体现价值:`etcd-0.etcd` / `etcd-1.etcd` / `etcd-2.etcd` 三个成员按这套稳定名互找,做 Raft 选主和数据同步,headless Service 是这类集群成员互联的底座。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>volumeClaimTemplates</b></font>（卷申领模板）:StatefulSet 字段,为每个 Pod 自动生成独立 PVC。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>initContainer</b></font>（初始化容器）:Pod 主容器之前先跑的辅助容器,常用来等依赖就绪、做一次性初始化。</font>

#### 1.5.1 无状态 Deployment 的 yaml 写法

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_Deployment三件套-57a4b102.png" width=80%></div>

&emsp;&emsp;<b>无状态服务</b>的特征:每个副本完全可互换,挂掉一个 controller 立刻起一个新的（新名字、新 IP）无所谓。本课文搜图项目里 <b>frontend / backend</b> 都是无状态——前端跑 nginx 静态资源,后端是无状态 HTTP API。

&emsp;&emsp;<b>Deployment yaml 关键字段</b>:`replicas: 3` 声明期望副本数,`selector.matchLabels` 跟 Pod template 的 `labels` 必须对得上,`template.spec.containers` 写镜像、端口、资源限额。1.3 节我们讲了 Deployment Controller 的工作机制——Deployment 创建 ReplicaSet,ReplicaSet 维持目标 Pod 数;这里关心的是<b>用户视角的写法</b>。第三章 frontend 编排会从零写一份完整 Deployment yaml,先记住三件事:① 副本数声明在 `spec.replicas`;② Pod 模板放在 `spec.template`;③ selector 跟 labels 必须匹配,否则 Deployment 找不到自己管的 Pod。

&emsp;&emsp;<b>命令式 vs 声明式</b>:Compose 里我们写 `docker run nginx`,这是命令式——"现在启动一个 nginx 容器";K8s 里我们写 `replicas: 3`,这是声明式——"在任何时候我都期望系统里有 3 个 nginx Pod"。两者差别在于,K8s 会在节点故障、Pod 退出、进程被手动删除时,基于"我期望有 3 个"这个声明把缺的那个补回来,我们不用写"如果 Pod 挂了请重启"——这就是<b>自愈</b>的来源。要警惕的一点是:刚从 Compose 切过来容易把 `kubectl apply` 当 `docker run` 用——`apply` 是"声明期望",不是"立刻执行",Pod 没起来的时候要看 Events 和 controller-manager 日志,不是再 apply 一次。

&emsp;&emsp;<b>HPA（水平自动扩缩）是 Deployment 的天然搭档</b>。我们写 `replicas: 3` 是固定值;HPA 让我们声明"<b>frontend 副本数 1-5,目标 CPU 50%</b>",HPA 每 15 秒从 metrics-server 拉一次 Pod CPU 数据,按公式 `desiredReplicas = ceil(currentReplicas × currentMetric / desiredMetric)` 算出期望副本数,调 Deployment 改 replicas——副本数从"我们写死"升级为"集群按负载自动调"。

&emsp;&emsp;<b>两个关键前提</b>:① <b>k3s 已内置 metrics-server</b>,`kubectl top` 直接可用,不要再额外 apply 一套 metrics-server,否则会冲突;② HPA 目标 Deployment 必须配 `resources.requests.cpu`,否则 HPA 永远算不出比例。第四章 4.1 节会展开 HPA 在 GPU 推理服务上"很难发挥作用"的实际结论——模型加载分钟级,HPA 的采样和扩容节奏追不上短时峰值,所以本课主要通过并发请求观察多副本的负载分发,滚动升级零停机部分讲清原理,HPA 这部分留作概念理解。

#### 1.5.2 有状态 StatefulSet 与两件强配套

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L1_StatefulSet两件强配套-387c5c28.png" width=80%></div>

&emsp;&emsp;<b>有状态服务</b>的特点:每个副本有自己的身份（`etcd-0` 跟 `etcd-1` 各自承担不同角色）、自己的磁盘（数据跟 Pod 名绑定）、可控的启动顺序（默认按序号 0/1/2 起来）。

&emsp;&emsp;<b>StatefulSet 跟 Deployment 的核心区别</b>:Deployment 的 Pod 名是随机 hash（`frontend-abc123-xyz`）,重建后名字会变,磁盘可共享;StatefulSet 的 Pod 名是固定有序的（`etcd-0` / `etcd-1` / `etcd-2`）,重建后保持原名,独立 PVC 跟着 Pod 名绑定。对有状态服务,这意味着集群成员可以通过稳定 DNS 名互相找到,磁盘数据不会因"换 Pod"丢失。

&emsp;&emsp;<b>K8s 官方文档对 StatefulSet 只点名两件强配套</b>——其余 PVC / PV / StorageClass 是 K8s 通用存储机制,任何工作负载都能用,不是 StatefulSet 专属:


<div align=center>

| 配套对象 | 作用 |
|---|---|
| <b>headless Service</b>（无头服务） | `clusterIP: None`,给 StatefulSet Pod 提供稳定 DNS 名 `{pod-name}.{service-name}.{namespace}.svc.cluster.local`;StatefulSet 用 `serviceName` 字段绑定它 |
| <b>volumeClaimTemplates</b>（卷申领模板） | StatefulSet 专属字段,为每个 Pod 自动生成独立 PVC,命名 `{template}-{sts}-{ordinal}`;PVC 再去找 StorageClass 自动 provision PV |

</div>

&emsp;&emsp;<b>选型边界</b>:StatefulSet 主要适合<b>集群成员要按名字互相找到对方</b>的场景（etcd 集群成员选主 / milvus 集群分片 / minio 分布式存储分片）。如果是单副本服务,即使有持久化需求,用 Deployment + 独立 PVC 也能跑——StatefulSet 的真正价值在"稳定 DNS 名 + 每 Pod 独立 PVC + 顺序启停",单副本三个优势都用不上。本课实际部署的形态在第三章会确定。

### 1.6 Compose vs K8s

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/Compose对比K8s五差异-38d5b1b4.png" width=80%></div>

&emsp;&emsp;讲完控制面 + 工作节点 + 资源对象 + 工作负载,最后我们用一张图把 Compose 和 K8s 的<b>五个根本差异</b>并排展示,作为本章收束。这五个差异每一个都对应 K8s 多出来的复杂度,但每一个也都对应单机 Compose 给不了的能力。

<p align="center"><font face="黑体" size=4>Compose 与 K8s 的五点根本差异</font></p>
<div align=center>


<div align=center>

| 差异维度 | docker-compose | Kubernetes |
|----------|----------------|------------|
| 部署范围 | 单机 | 多机集群（任意规模） |
| 调度单位 | 容器 | Pod（可包含 1 到 N 个紧耦合容器,共享网络与存储） |
| 操作范式 | 命令式（`docker run`） | 声明式（YAML + reconcile loop） |
| 扩缩能力 | 无自动扩缩 | HPA 根据 CPU / 自定义指标自动扩缩 |
| GPU 调度 | 无 GPU 资源声明 | nvidia.com/gpu 作为可调度资源 |

</div>

</div>

&emsp;&emsp;<b>K8s 的复杂度对应生产能力</b>。生产环境的可靠性、可扩展性、AI 场景的 GPU 调度,这三件事单机 Compose 给不了。yaml 字段多是为了把"集群级期望"完整表达出来,复杂度的本质不是工程师炫技,而是真实生产场景的复杂度被诚实展开了。学完本课我们对 K8s 的复杂度会有真实判断——它的复杂度配得上它解决的问题。

&emsp;&emsp;<b>本章地基已经就位</b>。下一章我们把实操环境搭起来——在一台 4× RTX 3090 24GB 的 GPU 服务器上装单节点 k3s,装好 helm / k9s 工具栈（kubectl 由 k3s 自带）,配好国内 mirror,装上 NVIDIA Device Plugin 把 GPU 注册成 K8s 可调度资源。下一章末尾会给一段简短指引,讲清单节点扩展到多机集群应该怎么走——k3s 多机 join 本身只是一行命令的事,真正复杂的是网络 / 存储 / CNI 等生产配套,这些属于 K8s 集群管理话题,查 kubernetes.io 官方文档的集群管理章节即可。

---

## <center>第二章 在 GPU 服务器上装一套 K8s 集群</center>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L2_第二章学习路径-31ae78b8.png" width=80%></div>

&emsp;&emsp;第一章讲清了 K8s 的基本模型,这一章开始搭环境。我们会在一台 4× RTX 3090 24GB 的 GPU 服务器上 ssh 实操,装单节点 k3s、配置 mirror、装 helm / k9s（kubectl 由 k3s 自带）,再用 NVIDIA Device Plugin 把 GPU 注册成可调度资源。之所以不用 minikube 或 Docker Desktop,是因为本课后面要真正观察 GPU 调度和推理服务编排;走完本章,我们会得到一套<b>能跑 GPU 推理 pod 的 K8s 集群</b>。本章末尾会给一段简短的"<b>多节点扩展说明</b>"讲清单节点 → 多机集群的迁移路径。

### 2.1 单终端实操约定

&emsp;&emsp;本机开一个终端,`ssh 4GPU24G` 连到 GPU 服务器（192.168.110.131）,本章所有命令都在这一个终端里跑。后续 3.9 节单 pod 并发压测、4.5/4.6 节多副本观察会同时开 3 个终端联动观察（k9s 看 K8s 状态 + shell `&` 并发 curl 制造负载 + `kubectl logs -f` 看请求落点）,那是观察实验需要,跟本章装环境无关。本章只用一个终端,所有命令的预期效果就是这一个会话里看到的输出。

### 2.2 装 k3s server 节点（单节点）

#### 2.2.1 server 节点角色:single binary 设计

&emsp;&emsp;<b>k3s server 节点就是 K8s 控制面那台机器</b>——第一章 1.3 节讲过控制面四件套（api-server / etcd / scheduler / controller-manager）,这一节我们把这四件套以 k3s 形态在 server 节点上装起来。k3s 是 Rancher / SUSE 维护、经 CNCF 认证的 K8s 发行版,把官方 K8s 几个独立组件打包成一个单 Go 二进制文件（70MB）,一行安装命令即可启动,systemd 接管生命周期,跟标准 K8s 行为对写 yaml / 用 kubectl 完全透明。

&emsp;&emsp;<b>为什么单节点只装 server 就够了</b>。标准 K8s（kubeadm）的 主节点默认带 `node-role.kubernetes.io/control-plane:NoSchedule` taint 阻止业务 pod 调度到控制面,所以必须额外装 worker 节点。<b>k3s server 节点不带这个 taint</b>——k3s 二进制文件本身就把 api-server / scheduler / controller-manager（控制面）和 kubelet / kube-proxy / 嵌入式 containerd（工作面）全部塞在同一个 Go binary 里,server 进程启动时这两层组件一起跑。

&emsp;&emsp;所以单节点 k3s 集群的形态是:<b>k3s-server 这一个节点同时承担"决策"（control plane）和"执行"（worker）两个角色,业务 pod 直接调度到它身上跑,无需另装 worker 节点</b>。验证方式:`kubectl describe node k3s-server | grep -E 'Roles|Taints'` 看到 `Roles: control-plane` + `Taints: <none>`——`Taints` 为空就是"业务 pod 能直接调度过来"的硬证据。

> <font size=2><b>【名词解释】</b><font color=red><b>k3s</b></font>（读 kates,体量减半的 K8s）:Rancher 出品的轻量级 CNCF 认证 K8s 发行版,把 api-server / scheduler / controller-manager / kubelet / kube-proxy / 嵌入式 containerd 全部编译进同一个二进制文件,靠 `k3s server` / `k3s agent` 子命令切换角色——`server` 子命令同时启动控制面 + 工作面组件,所以单 server 节点就是个完整可用集群,生产可用。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>kubeconfig</b></font>（K8s 客户端凭证文件）:一份 YAML 文件,里头存着集群地址 + 证书 + 用户身份,kubectl 拿它跟 api-server 握手认证。k3s 装完后默认放在 `/etc/rancher/k3s/k3s.yaml`。</font>

#### 2.2.2 k3s 生态:边缘生产事实标准,不是模拟玩具

&emsp;&emsp;装命令之前先讲清 k3s 的真实定位——它<b>不是</b> minikube / kind / Docker Desktop K8s 这种"开发测试模拟玩具",而是 <b>CNCF Certified Kubernetes</b>、企业在生产真实跑的 K8s 发行版。本课用它是因为单机 + GPU 实验场景天然匹配,但要知道:这套 yaml 拿到生产环境一样能跑。

&emsp;&emsp;<b>k3s 在生产真实跑在哪</b>:


<div align=center>

| 场景 | 典型案例 |
|---|---|
| 零售门店 / POS | 沃尔玛 / Apple Store 等连锁,每店一个 k3s 跑收银 / 库存 / 监控 |
| 5G MEC 边缘节点 | 运营商 5G 边缘计算节点 |
| 工业 IoT | 工厂产线 / 油田传感器 / 自动驾驶车队 |
| 多分支机构 | 银行分行 / 物流仓 / 加油站等分布式独立节点 |
| 中小企业自建 | 创业公司 / 内部 SaaS 系统 |

</div>

&emsp;&emsp;这些场景共同点:<b>资源受限 + 分布式独立节点 + 不愿运维大型 K8s 但要 K8s 能力</b>。SUSE Rancher Prime 给企业级订阅,帮企业管理千级 k3s 边缘集群,这块是 k3s 的真实商业落地。

&emsp;&emsp;<b>跟模拟玩具 vs 大集群企业 K8s 的边界</b>:


<div align=center>

| 类别 | 典型工具 | 定位 | 能上生产? |
|---|---|---|---|
| K8s 模拟工具 | minikube / kind / k3d / Docker Desktop K8s | 本机跑 yaml 验证语法 | ❌ 性能 / 稳定性 / 跨节点都不行 |
| <b>k3s</b> | k3s 单 / 多节点 | <b>边缘 / IoT / 中小企业生产 + 学习</b> | ✅ 边缘 + 中小场景 |
| 企业大集群 K8s | EKS / GKE / AKS / OpenShift / RKE2 | 大型核心生产 | ✅ 大集群核心 |

</div>

&emsp;&emsp;<b>k3s 跟标准 K8s 100% 兼容</b>:yaml / kubectl / 资源对象行为完全一致——差别只在<b>组件部署形态</b>（super-binary vs 5 个独立进程）和<b>规模适用性</b>（k3s 适合 1 到几十节点;大集群走 EKS / RKE2 / OpenShift）。本课写的所有 yaml 拿到任何 CNCF 认证集群上 `kubectl apply` 都能跑,不需要改一行。

#### 2.2.3 一行命令装通 server

&emsp;&emsp;ssh 上 GPU 服务器,跑下面这条命令——一行装通 server,加几个 flag 让节点名固定为 `k3s-server` 且 kubeconfig 文件对当前用户可读:

```bash
# 国内服务器用 Rancher 中国镜像 (rancher-mirror.rancher.cn) 拉脚本和二进制,
# 比 get.k3s.io → GitHub release 快得多, 也避免 SSL EOF / DNS 超时等不稳定问题
curl -sfL https://rancher-mirror.rancher.cn/k3s/k3s-install.sh | \
    INSTALL_K3S_MIRROR=cn \
    INSTALL_K3S_EXEC='server --node-name=k3s-server --write-kubeconfig-mode=644' \
    sh -
```

&emsp;&emsp;<b>这条命令做三件事</b>:

- ① <b>拉脚本 + 拉二进制文件全走国内 CDN</b>:`rancher-mirror.rancher.cn` 拉脚本,`INSTALL_K3S_MIRROR=cn` 让脚本接着从国内拉 k3s 二进制文件——两个国内入口缺一不可,只配前者会"脚本是国内拉的,k3s 二进制文件还回 GitHub release CDN 超时"。

- ② <b>装成 systemd 服务</b>:把 k3s 二进制文件落到 `/usr/local/bin/k3s`,生成 `/etc/systemd/system/k3s.service`,`systemctl enable --now` 启动,自此控制面四件套（api-server / etcd / scheduler / controller-manager）+ kubelet 全在一个 `k3s` 进程里跑。

- ③ <b>固定节点名 + 放开 kubeconfig 权限</b>:`--node-name=k3s-server` 把节点名钉死（默认用 hostname,跨机器引用不统一）,`--write-kubeconfig-mode=644` 让 `/etc/rancher/k3s/k3s.yaml` 非 root 也可读（默认 600 只 root）。

&emsp;&emsp;<b>境内 vs 境外的取舍</b>:海外服务器用官方 `curl -sfL https://get.k3s.io | sh -` 一行装通;国内服务器必走上面那条 Rancher 中国镜像版本——否则 `get.k3s.io` 在国内反复 SSL EOF + GitHub release CDN 拉二进制文件超时,装一次半小时是常态。

&emsp;&emsp;<b>kubectl 不用单独装</b>——装 k3s 时顺手做了个软链 `/usr/local/bin/kubectl` 指向 `/usr/local/bin/k3s` 自己,<b>敲 `kubectl` 实际跑的是 k3s</b>,只是 k3s 看到自己被叫"kubectl"就表现成 kubectl（同套还自带 `crictl` / `ctr` 两件软链,后面排错用）。设置 KUBECONFIG 让它找到 kubeconfig 凭证:

```bash
echo 'export KUBECONFIG=/etc/rancher/k3s/k3s.yaml' >> /root/.bashrc
export KUBECONFIG=/etc/rancher/k3s/k3s.yaml
kubectl get nodes
```

&emsp;&emsp;期望看到的实测输出:

In [ ]:
NAME         STATUS   ROLES           AGE   VERSION
k3s-server   Ready    control-plane   19s   v1.35.5+k3s1

&emsp;&emsp;<b>三条关键路径要记住</b>（后面 mirror 配置和 Device Plugin 排错反复用到）:① kubeconfig 在 `/etc/rancher/k3s/k3s.yaml`（kubectl 凭证）;② 嵌入式 containerd socket 在 `/run/k3s/containerd/containerd.sock`（后面 ctr 命令直接拉镜像用到）;③ 镜像预导入目录在 `/var/lib/rancher/k3s/agent/images/`（镜像 tar 放这里,k3s 启动自动导入到 containerd,无需走 registry）。

&emsp;&emsp;<b>k3s 嵌入式 containerd 跟 docker daemon 是两套独立运行时</b>——Docker 里配过 mirror 不代表 k3s 能用,2.3 节会统一配置 `/etc/rancher/k3s/registries.yaml`。

#### 2.2.4 装完看 5 个维度的组件痕迹

&emsp;&emsp;<b>装完看不到独立组件进程?这才是 k3s single binary 设计的特征</b>。前面讲过 k3s 把控制面 + 工作面打成同一个 Go 二进制文件,这会儿在终端跑 `ps aux | grep kube` 看不到 kube-apiserver / kube-scheduler / kube-controller-manager / kubelet / kube-proxy 这 5 个标准 K8s 独立进程,可能会有"是不是没装完"的困惑。下面 5 条命令从不同维度证明组件全装好了——只是被打包进一个 `k3s server` 进程里跑:


<div align=center>

| 维度 | 命令 | 看到什么 |
|---|---|---|
| ① systemd 层 | `systemctl status k3s` | 一个 k3s.service unit + 一个 `k3s server` 主进程承载所有组件 |
| ② 进程层 | `ps -ef \| grep k3s` | 找不到 kube-apiserver / kube-scheduler 等独立进程 |
| ③ K8s 控制面 | `kubectl cluster-info` | 控制面在 `https://127.0.0.1:6443` |
| ④ 组件健康 | `kubectl get componentstatuses` | etcd / scheduler / controller-manager 三件套全 Healthy |
| ⑤ 端口分布 | `ss -tlnp \| grep k3s` | 5 个端口都属于 `k3s server` 一个进程 |

</div>

```bash
# 装完 k3s 看组件痕迹 5 条命令 (整段 copy-paste 跑)

# ① systemd 层: 一个 k3s.service unit + 一个 k3s server 主进程承载所有组件
systemctl status k3s --no-pager | head -15

# ② 进程层: 找不到 kube-apiserver / kube-scheduler / kubelet 等独立进程
#         它们都是 k3s server 进程内的 goroutine, 不是独立进程
ps -ef | grep -E 'k3s.*server' | grep -v grep | head -3

# ③ K8s 控制面入口 (6443 API server) 是否就绪 + 关键 addon 状态
kubectl cluster-info

# ④ 控制面三件套健康状态 (老 API 但仍能看到 etcd / scheduler / controller-manager)
kubectl get componentstatuses

# ⑤ k3s server 进程内嵌组件监听的所有端口
ss -tlnp | grep -E ':(6443|10250|10256|10257|10259)'
```

&emsp;&emsp;<b>5 个端口对应 5 个传统 K8s 组件</b>（这张对照表后面 2.7 节集群验证还会再用一次,先眼熟）:


<div align=center>

| 端口 | 组件 | 作用 |
|---|---|---|
| `6443` | kube-apiserver | K8s 集群总入口,kubectl 全走这里 |
| `10250` | kubelet | 节点上的"容器管理员",api-server 通过这个端口让 kubelet 起容器 |
| `10256` | kube-proxy | 节点 iptables/ipvs 规则录入器（1.2.4 节展开过:流量实际走 Linux 内核 netfilter,不经过 kube-proxy 用户态） |
| `10257` | controller-manager | 控制器健康检查端口 |
| `10259` | scheduler | 调度器健康检查端口 |

</div>

&emsp;&emsp;<b>single binary 设计是 k3s 省资源的关键</b>——传统 K8s 装到这里至少要 5 个独立进程 + 5 套日志输出 + 5 套 systemd unit。k3s 把它们打成一个 70MB 二进制文件 + 一个 systemd unit + 一个进程统一日志,内存占用 600-700 MB,这是 k3s 适合开发机 / 边缘节点 / 小集群的根本原因。

#### 2.2.5 多节点扩展:agent join + 生产配套四件事

&emsp;&emsp;本课用单节点是因为我们只有一台 GPU 服务器,机制原理跟多机集群完全一样。如果我们有多台 Linux 服务器要组多机 k3s 集群,核心步骤就两条命令:

```bash
# server 机器上拿 join token
sudo cat /var/lib/rancher/k3s/server/node-token
# 输出形如: K10xxxxxxxxxxxxxxxxxxxxxx::server:yyyyyyy

# 每台 agent 机器跑 (K3S_URL 指向 server 节点的内网 IP, 不是 127.0.0.1)
# 【提醒】跟 server 节点保持一致走 Rancher 中国镜像, 国内服务器必须, 否则 SSL EOF
curl -sfL https://rancher-mirror.rancher.cn/k3s/k3s-install.sh | \
    INSTALL_K3S_MIRROR=cn \
    INSTALL_K3S_EXEC='agent' \
    K3S_URL='https://<server-机器-内网-IP>:6443' \
    K3S_TOKEN='<上一步拿到的 token>' sh -
```

&emsp;&emsp;<b>join 本身一行命令,真正复杂的是生产配套</b>:① 网络打通——agent 机器到 server 的 6443 端口要通（防火墙 / VPC / 安全组）;② GPU 一致性——每台 GPU 节点的 NVIDIA driver / Container Toolkit 版本要对齐,RuntimeClass `nvidia` 在每个节点的 containerd 上都得有;③ 共享存储——本课用 hostPath + k3s 自带的 local-path-provisioner（单机够用）,多机生产要换 Ceph / Longhorn / NFS / 云厂商共享盘,否则跨节点迁移 pod 数据会丢;④ CNI 网络方案——k3s 自带 flannel 跨节点能跑通,生产规模大可能换 Calico / Cilium 拿到 NetworkPolicy / eBPF 等更细粒度能力。这些都不是 k3s 本身的事,是分布式 K8s 集群的通用话题,查 kubernetes.io 官方文档"Cluster Administration"章节即可。

### 2.3 国内拉镜像必备的 mirror 配置

&emsp;&emsp;<b>mirror 配置就是给 K8s 集群配镜像加速</b>——集群里绝大多数 pod 都得拉镜像跑（pause / coredns / traefik / metrics-server / device plugin / 业务镜像）,默认从 docker.io / nvcr.io / registry.k8s.io / quay.io / gcr.io 这些境外 registry 拉,国内网络直连数分钟超时是常态。mirror（镜像加速代理）的本质是在国内放一份代理 registry,让 containerd 拉镜像时优先走代理,代理拿不到再回源——我们这一节配的就是这套优先级路由。

> <font size=2><b>【名词解释】</b><font color=red><b>mirror registry</b></font>（镜像加速 registry）:一个公开访问的代理仓库,从境外 upstream registry 拉镜像缓存到国内 CDN,客户端拉镜像时配置成 mirror 后会优先走代理。本课用 DaoCloud 的 m.daocloud.io 域名簇覆盖 5 个主流 upstream。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>containerd hosts.toml</b></font>（containerd 镜像主机配置）:containerd 1.5+ 引入的"按 host 配置 mirror"格式,文件路径 `/var/lib/rancher/k3s/agent/etc/containerd/certs.d/{registry}/hosts.toml`,k3s 启动时会从 `/etc/rancher/k3s/registries.yaml` 自动翻译生成,无需我们手写。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>ImagePullBackOff</b></font>（K8s pod 状态）:kubelet 尝试拉镜像失败后进入指数退避重试的状态（第一次失败等 10 秒重试,再失败等 20 秒、40 秒……）,`kubectl get pod` 这一列就显示 ImagePullBackOff。典型成因:① 境外 registry 直连超时（本节配 mirror 解决）;② 镜像名/tag 写错;③ 私有仓库缺 imagePullSecrets。诊断手法:`kubectl describe pod <名字>` 看 Events 段最后几行的真实错误。</font>

&emsp;&emsp;<b>配 mirror 一气呵成四步</b>。<b>装 Device Plugin / 跑业务 pod 之前必须先配 mirror</b>——国内直连 `nvcr.io` / `registry.k8s.io` / `gcr.io` 等境外 registry 会 i/o timeout,pod 拉不下镜像就卡 ImagePullBackOff。在 GPU 服务器粘贴下面这段一次跑完:

```bash
# 一键生成 registries.yaml 并让 k3s 重读
# 步骤 1: tee heredoc 写文件 (sudo tee 而不是 sudo cat >, 因为 > 重定向是当前 shell 处理的,
#         不是 sudo;tee 命令自己以 sudo 跑负责写文件,权限才对)
sudo tee /etc/rancher/k3s/registries.yaml > /dev/null <<'EOF'
# 【提醒】这份文件只在 server 节点写一份, k3s 启动时会自动同步给 agent,
#       每个节点的 containerd 都会重新生成 hosts.toml, 不用手抄到每个节点
mirrors:
  "docker.io":         { endpoint: ["https://docker.m.daocloud.io"] }
  "nvcr.io":           { endpoint: ["https://nvcr.m.daocloud.io"] }
  "registry.k8s.io":   { endpoint: ["https://k8s.m.daocloud.io"] }
  "quay.io":           { endpoint: ["https://quay.m.daocloud.io"] }
  "gcr.io":            { endpoint: ["https://gcr.m.daocloud.io"] }
EOF

# 步骤 2: 验证文件已写入 (cat 看一眼, 没内容会立即暴露问题)
cat /etc/rancher/k3s/registries.yaml

# 步骤 3: 重启 k3s service (containerd 会读 registries.yaml 翻译成 hosts.toml)
systemctl restart k3s

# 步骤 4: 验证节点 Ready (重启后等 10 秒)
sleep 10 && kubectl get nodes -o wide
```

&emsp;&emsp;期望看到的实测输出:

In [ ]:
NAME         STATUS   ROLES           AGE   VERSION        INTERNAL-IP     EXTERNAL-IP   OS-IMAGE             KERNEL-VERSION      CONTAINER-RUNTIME
k3s-server   Ready    control-plane   62m   v1.35.5+k3s1   192.168.130.4   <none>        Ubuntu 22.04.5 LTS   6.8.0-107-generic   containerd://2.2.3-k3s1

&emsp;&emsp;关键看两件事:① <b>STATUS=Ready</b> 表示重启 k3s 后节点重新就绪（重启大约 5-10 秒,所以前面 `sleep 10`）;② <b>CONTAINER-RUNTIME=containerd://X.X.X-k3s1</b> 表示 k3s 嵌入式 containerd 重启后重新读取了 hosts.toml,mirror 配置生效。<b>AGE / IP / kernel</b> 等字段跟环境相关,数值不一致不影响判断。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L2-mirror-routing-d7ff5f5a.png" width=80%></div>

&emsp;&emsp;<b>验证 mirror 落地</b>:k3s 重启后会自动把 registries.yaml 翻译成 containerd 的 hosts.toml 文件,落在 `/var/lib/rancher/k3s/agent/etc/containerd/certs.d/{registry}/hosts.toml`。可以 `cat` 看一眼实测内容——里面除了我们写的 DaoCloud,还能看到 k3s 兜底加上的 `https://docker.1panel.live/v2`、`https://hub-mirror.c.163.com/v2` 备用 mirror,多一层冗余更稳。配前 vs 配后的拉镜像耗时对比:device plugin 镜像从数分钟超时降到 < 1 分钟成功——这个数量级差异就是 mirror 必须放在装 Device Plugin 之前的理由。

### 2.4 装工具栈

&emsp;&emsp;<b>装好 server 之后还需要工具操作集群</b>。本课涉及 3 件工具:<b>kubectl</b>（命令行入口）、<b>helm</b>（包管理器）、<b>k9s</b>（TUI 观察台）。kubectl 跟 k9s 每节都用,helm 在第三章打包文搜图服务用。<b>kubectl 已经在 2.2.3 节装 k3s 时 symlink 自带</b>,本节只装 helm 和 k9s 两件。

> <font size=2><b>【名词解释】</b><font color=red><b>kubectl</b></font>（Kubernetes CLI,K8s 命令行客户端）:K8s 集群的官方命令行操作工具,所有 yaml apply / 资源查询 / 调试都走这一个二进制文件。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>helm</b></font>（K8s 包管理器）:K8s 上的 "yum/apt",把多个 yaml 打包成 Chart 模板,一行 `helm install` 部署整套服务,支持 install / upgrade / rollback 生命周期管理。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>k9s</b></font>（K8s TUI 客户端）:基于 kubectl 的终端 TUI 界面,光标 + 快捷键即可浏览 Pod / 看日志 / 进容器,比 kubectl 命令行节省 80% 击键。</font>

&emsp;&emsp;<b>GPU 服务器是 Ubuntu 22.04（linux/amd64）</b>,本课走直接装 Linux 二进制文件的路径。在 GPU 服务器依次跑:

```bash
# 先确认 k3s 自带的 kubectl 能用 (装 k3s 时自动建的 symlink)
which kubectl                   # 期望: /usr/local/bin/kubectl
ls -la /usr/local/bin/kubectl   # 期望: lrwxrwxrwx ... /usr/local/bin/kubectl -> /usr/local/bin/k3s
kubectl version --client        # 期望: Client Version: vX.Y.Z+k3s1, 跟 k3s server 版本一致

# 【提醒】如果 ls -la 显示 -rwxr-xr-x (普通文件, 不是 symlink), 说明之前装过独立 kubectl
#       覆盖了 k3s 的 symlink, 跑下面两条修复:
#   sudo rm /usr/local/bin/kubectl
#   sudo ln -s /usr/local/bin/k3s /usr/local/bin/kubectl

# helm v3.21.0 (官方 install 脚本)
curl -fsSL https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

# k9s 最新版 (版本号从 GitHub API 查，安装包从 GitHub Release 拉，只是下载包这一步经过了 gh-proxy.com 代理)
GITHUB_PROXY="https://gh-proxy.com/"

K9S_VER=$(curl -s https://api.github.com/repos/derailed/k9s/releases/latest \
  | grep '"tag_name"' \
  | cut -d'"' -f4)

K9S_URL="https://github.com/derailed/k9s/releases/download/${K9S_VER}/k9s_Linux_amd64.tar.gz"

curl -L "${GITHUB_PROXY}${K9S_URL}" -o /tmp/k9s.tar.gz

tar xzf /tmp/k9s.tar.gz -C /tmp
install /tmp/k9s /usr/local/bin/k9s
```

&emsp;&emsp;装完跑一遍版本验证,确认 3 件工具都能执行:

```bash
kubectl version --client      # 跟 k3s server 版本一致 (k3s super-binary)
helm version --short          # v3.21.0+...
k9s version                   # 最新 stable (本课实测 v0.50.18)
```

### 2.5 K8s yaml 通用结构

&emsp;&emsp;k3s 装好 + mirror 配齐 + 工具栈装完,下一节就要 apply 整门课的第一份 yaml（NVIDIA Device Plugin 的 RuntimeClass）。动手写 yaml 之前先把<b>共通骨架</b>认清:K8s 所有<b>资源对象</b>（用 yaml 声明的 K8s 对象,如 Pod / Service / Deployment / Ingress / ConfigMap 等,1.4 节展开过）都用同一套 yaml 表达——`apiVersion` / `kind` / `metadata` / `spec` 四件套。无论是 5 行的 RuntimeClass 还是 200 行的 milvus StatefulSet,顶层结构完全一样,差别只在 `spec` 里写什么。本节把骨架讲清,下一节装 Device Plugin 第一份 yaml 套上去就是。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L2_5_yaml四件套结构-0078fb2f.png" width=80%></div>

&emsp;&emsp;<b>四件套各自的职责</b>:


<div align=center>

| 字段 | 干什么 | 第一次接触时常踩的坑 |
|---|---|---|
| `apiVersion` | 声明对象由 K8s 哪一组 API 管 | 不同 kind 用不同 apiVersion:Pod / Service / ConfigMap / Secret 是 `v1`,Deployment / StatefulSet / DaemonSet 是 `apps/v1`,Ingress 是 `networking.k8s.io/v1`（注:Gateway API `gateway.networking.k8s.io/v1` 是 Ingress 的继任者,2026 年生产新集群官方推荐）,RuntimeClass 是 `node.k8s.io/v1`——写错就报 `no matches for kind` |
| `kind` | 声明对象类型 | 常见 kind 按职责分:<b>工作负载</b> `Pod` / `Deployment` / `StatefulSet` / `DaemonSet` / `Job` / `CronJob`;<b>网络</b> `Service` / `Ingress`;<b>配置</b> `ConfigMap` / `Secret`;<b>存储</b> `PersistentVolume` / `PersistentVolumeClaim` / `StorageClass`;<b>基础设施</b> `Namespace` / `RuntimeClass`。<b>大小写敏感</b>——必须是 `Deployment` 不是 `deployment`,`StatefulSet` 不是 `statefulset` |
| `metadata` | 对象身份信息 | 常见子字段:<b>必填</b> `name`（对象名,DNS-1123 合法,小写字母+数字+`-`,如 `etcd-0` / `wensoutu-backend`）;<b>选填</b> `namespace`（对象所在命名空间,默认 `default`,本课业务全放 `wensoutu`）、`labels`（`key: value` 键值对,如 `app: backend` / `tier: storage`,Service selector / Deployment selector 找 Pod 靠这个,1.4.1 节讲过）、`annotations`（不参与 selector 的元数据,如 `kubectl.kubernetes.io/last-applied-configuration` / k3s 自动加的 `objectset.rio.cattle.io/owner-gvk`）。<b>常踩的坑</b>——name 大写或带 `_` 直接 apply 报 `invalid value`,namespace 漏写就跟系统对象混在 `default` |
| `spec` | 对象的规约,真正的变量部分 | 每种 kind 的 `spec` 字段完全不同（K8s 学习曲线最陡的一段）,本课用到的:<b>Pod</b> `containers` / `volumes` / `nodeSelector` / `runtimeClassName` / `restartPolicy`;<b>Deployment</b> `replicas` / `selector` / `template`（里面套一层 Pod spec）;<b>StatefulSet</b> 多 `serviceName` / `volumeClaimTemplates`;<b>DaemonSet</b> 不用 `replicas`（每节点一个 pod）;<b>Service</b> `type`（ClusterIP / NodePort / LoadBalancer）/ `selector` / `ports`;<b>Ingress</b> `ingressClassName` / `rules`（host + paths）。<b>特例</b>:ConfigMap / Secret 用 `data` 代替 `spec`,RuntimeClass 用顶层 `handler` 代替 |

</div>

&emsp;&emsp;<b>最简 yaml 长什么样</b>——用最常见的 Pod 举例,7 行带完整四件套:

```yaml
apiVersion: v1                 # Pod 属于核心组, 直接写 v1
kind: Pod                       # 对象类型
metadata:
    name: hello                # 对象叫 hello
    namespace: default          # 落到 default namespace
spec:                           # spec 是真正的变量部分
    containers:                 # Pod 的 spec 核心字段是 containers
    - name: hello
      image: nginx:1.27-alpine
```

&emsp;&emsp;<b>四件套之外还有两个常见但选择性出现的字段</b>:`status`（K8s 控制面自动写入的运行时状态,我们写 yaml 时不填,只在 `kubectl get pod -o yaml` 看实际状态时出现）和 `data`（ConfigMap / Secret 的载荷字段,代替 `spec`,因为这两类对象的"规约"就是数据本身）。看到 `data:` 就能反应过来"这是 ConfigMap 或 Secret 类对象"。

&emsp;&emsp;<b>这一节的关键</b>:K8s yaml 不是无穷无尽的字段海洋,而是<b>一套固定四件套骨架 + 每种 kind 各自的 spec 字段集</b>（少数 kind 像 RuntimeClass / ConfigMap / Secret 用 `handler` / `data` 代替 spec,但骨架前三件不变）。学新 kind 时前三件不用重学,只需要查"这个 kind 的 spec / data / handler 里能写啥"。下一节装 Device Plugin、第三章 3.3 到 3.8 节部署 5 个服务、配好 ConfigMap/Secret,所有 yaml 都是套这个骨架填字段。

### 2.6 装 NVIDIA Device Plugin 上 GPU

&emsp;&emsp;<b>前置一:把配套部署文件传到服务器</b>。从这一节开始我们要 apply 一批写好的 yaml（本节的 Device Plugin,以及第三章的 etcd / minio / milvus / backend / frontend 等），这些文件都在课件配套代码包里。先在本机解压代码包,然后在服务器建一个项目目录、用 scp 把文件整目录传上去:

```bash
# 下面这几条在【本机】跑(不是 ssh 到服务器之后)。4GPU24G 换成你自己的 ssh 别名或 user@ip
# 目标用绝对路径 /home/XiaoYangWorkSpace/wensoutu-k8s, 跟后面所有 apply 命令的路径保持一致
# (别图省事写成 ~/wensoutu-k8s: 本课 ssh 登录的是 root, ~ 展开成 /root, 跟 apply 路径对不上, 文件会传错地方)

# 1. 服务器上建项目目录(换成你自己服务器上打算放文件的绝对路径)
ssh 4GPU24G "mkdir -p /home/XiaoYangWorkSpace/wensoutu-k8s"

# 2. 把配套代码包里的 manifests 和 helm Chart 整目录传上去
#    (本机源路径换成你自己的代码包解压路径)
scp -r ~/code/k8s部署所需文件/manifests      4GPU24G:/home/XiaoYangWorkSpace/wensoutu-k8s/
scp -r ~/code/k8s部署所需文件/wensoutu-chart 4GPU24G:/home/XiaoYangWorkSpace/wensoutu-k8s/

# 3. 传完上服务器确认文件都到位
ssh 4GPU24G "ls /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/"
# 期望列出 runtime-class.yaml / nvidia-device-plugin-k3s.yaml / etcd-statefulset.yaml ... 等一批 yaml
```

&emsp;&emsp;传完之后,本课正文里所有 `kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/xxx.yaml` 就都能跑了。详细的文件清单和项目结构见第三章 3.2 节。<b>另外 backend 还依赖宿主机上的模型权重目录</b>（hostPath 挂载,约 8 GB），这部分怎么准备见配套代码包的 README 和第三章 3.7 节。

&emsp;&emsp;<b>前置二:宿主机 NVIDIA Driver + Container Toolkit 必须先装好</b>（机器层装的,跟 K8s 是两层）。在 GPU 服务器跑:

```bash
# 检查 1: NVIDIA Driver 已装且能看到 GPU
nvidia-smi
# 期望输出: Driver Version: 580.x (或更高), CUDA Version: 13.x, 列出 4× RTX 3090

# 检查 2: NVIDIA Container Toolkit 已装(让 containerd / docker 能起 GPU 容器)
nvidia-ctk --version
# 期望输出: NVIDIA Container Toolkit CLI version 1.x.x
```

&emsp;&emsp;两条命令都跑通才能继续本节。如果是新装的 Ubuntu 服务器,这两件事的标准做法:① 用 `sudo ubuntu-drivers install` 装 Driver（或从 NVIDIA 官网下 `.run` 包手装）;② 跟着 NVIDIA 官方 `docs.nvidia.com/datacenter/cloud-native/container-toolkit/` 装 Container Toolkit。两件都不在本课范围,本课假设宿主机已经能跑 GPU docker 容器（前一门课的产物）。

&emsp;&emsp;<b>NVIDIA Device Plugin 是一个 DaemonSet 形态的 K8s pod</b>——每个 GPU 节点上跑一份,工作是把节点上的物理 GPU 硬件注册成 K8s 调度器能看到的资源（资源名固定叫 `nvidia.com/gpu`,值是节点上的 GPU 卡数）。一旦注册成功,我们就能在 pod yaml 里写 `resources.limits.nvidia.com/gpu: 1` 来申请一块 GPU,scheduler 会自动找到有可用 GPU 的节点把 pod 调过去。没有 Device Plugin 这一层,K8s 是"看不见"GPU 的——这是 K8s 识别和调度 GPU 资源的官方标准机制。

> <font size=2><b>【名词解释】</b><font color=red><b>容器运行时</b></font>（Container Runtime）:负责真正把容器跑起来的底层组件。K8s 本身不直接启动容器,而是由节点上的 kubelet 调用 containerd 这类运行时完成镜像拉取、容器创建和生命周期管理;更底层还会调用 runc、crun、nvidia-container-runtime 等组件,真正创建 namespace、cgroup 并启动容器进程。简单说:K8s 负责调度和管理,容器运行时负责把容器真正跑起来。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>RuntimeClass</b></font>（运行时类）:K8s 的集群级对象,不属于任何 namespace。它用来声明一种可被 Pod 选择的容器运行时配置。Pod 通过 `runtimeClassName` 指定自己要用哪种 runtime 启动——普通容器默认走 runc,GPU 容器可以指定 nvidia,强隔离场景可以指定 kata。注意:RuntimeClass 只是 K8s 层面的入口声明,不代表底层 runtime 一定已经装好,真正能不能用还要看 containerd 配置和节点上的运行时二进制文件是否存在。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>runtimeClassName</b></font>（Pod 的运行时类引用字段）:Pod yaml 里的字段,指定这个 Pod 使用哪个 RuntimeClass。比如写 `spec.runtimeClassName: nvidia`,表示这个 Pod 启动容器时走 NVIDIA runtime。GPU Pod 通常还要同时申请 GPU 资源 `resources.limits.nvidia.com/gpu: 1`。这两个配置作用不同:`runtimeClassName` 决定"用哪种运行时启动容器",`nvidia.com/gpu` 决定"这个 Pod 申请几张 GPU"。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>handler</b></font>（RuntimeClass 的运行时匹配名）:RuntimeClass 里连接 K8s 和 containerd 的名字。Pod 里写 `runtimeClassName: nvidia` 后,K8s 找到名为 `nvidia` 的 RuntimeClass,再读里面的 `handler: nvidia`;随后 containerd 去自己的配置文件里找 `runtimes.'nvidia'` 段,最终调用 `/usr/bin/nvidia-container-runtime` 这类真实二进制文件。简单说:handler 不是程序文件,而是 RuntimeClass 指向底层 runtime 配置的匹配名。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>NVIDIA Device Plugin</b></font>（GPU 设备插件）:让 K8s 识别和分配 NVIDIA GPU 的组件。K8s 默认不认识 GPU,只认识 CPU、内存这类基础资源。Device Plugin 跑在 GPU 节点上,检查本机有几张 GPU,把 `nvidia.com/gpu` 这种资源上报给 kubelet;上报成功后,Pod 才能通过 `resources.limits.nvidia.com/gpu: 1` 申请 GPU。它通常以 DaemonSet 方式部署,因为每个 GPU 节点都需要运行一份。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>NVML</b></font>（NVIDIA Management Library）:NVIDIA 提供的 GPU 管理库,常见文件名 `libnvidia-ml.so.1`。它可以查询 GPU 数量、显存占用、利用率、温度、功耗等信息,`nvidia-smi` 底层也用它。NVIDIA Device Plugin 需要通过 NVML 识别本机 GPU;如果找不到 NVML,插件可能无法枚举 GPU,K8s 也就看不到 `nvidia.com/gpu` 资源。</font>

&emsp;&emsp;<b>装 Device Plugin 必须配齐三件事</b>:① 给 pod spec 加 `runtimeClassName: nvidia`,让 containerd 用 nvidia-container-runtime 启动（这个 runtime 会把宿主机 `/usr/lib/x86_64-linux-gnu/libnvidia-ml.so.1` 自动挂进容器）;② 集群里必须存在名为 `nvidia` 的 RuntimeClass 对象,pod 才能引用它;③ 容器镜像走 mirror（上一节配过）,否则拉不下来。第三件已经齐备;前两件 k3s 已经自动建好,我们先验证一下。

&emsp;&emsp;<b>第一步:确认 k3s 已经把 RuntimeClass 一整套准备好</b>。k3s install 自带一份 `/var/lib/rancher/k3s/server/manifests/runtimes.yaml`（<b>10 个 RuntimeClass 的预定义清单</b>）,k3s 启动后通过内置 Addon 控制器异步 apply 这份 manifest——所以装完 k3s 立刻跑 `kubectl get runtimeclass` 可能会看到 `No resources found`,<b>等 30 秒</b>再跑就有了:

```bash
# 装完 k3s 等 30 秒让 manifest auto-apply
sleep 30 && kubectl get runtimeclass
```

&emsp;&emsp;期望看到 10 个左右:

In [ ]:
NAME                  HANDLER               AGE
crun                  crun                  1m
lunatic               lunatic               1m
nvidia                nvidia                1m
nvidia-experimental   nvidia-experimental   1m
slight                slight                1m
spin                  spin                  1m
wasmedge              wasmedge              1m
wasmer                wasmer                1m
wasmtime              wasmtime              1m
wws                   wws                   1m

&emsp;&emsp;<b>为啥这么多</b>:k3s 默认就把这一整套准备好——`nvidia` 系列给 GPU 容器、`crun` 是轻量 OCI runtime 替代 runc、剩下 7 个是 WebAssembly / Serverless runtime（`wasmedge` / `wasmtime` / `wasmer` / `spin` / `slight` / `lunatic` / `wws`）。这些 RuntimeClass 仅仅是 K8s 声明,<b>能不能真正用要看 containerd 配置 + 节点上有没有对应 runtime 二进制文件</b>:


<div align=center>

| RuntimeClass | 配套二进制文件 | 本课能用吗? |
|---|---|---|
| `nvidia` / `nvidia-experimental` | `/usr/bin/nvidia-container-runtime`（NVIDIA Container Toolkit 装的） | ✅ k3s 装好 + Toolkit 装好后 k3s 自动在 containerd config 加 `nvidia` runtime 段,可用 |
| `crun` | `crun` 二进制文件（系统包） | ⚠️ 节点没装就用不了,pod 报 `unknown runtime` |
| `wasmedge` / `wasmtime` / `wasmer` / `spin` / `slight` / `lunatic` / `wws` | 对应的 containerd-shim-X-v1 二进制文件 | ⚠️ 节点没装就用不了,纯占位声明 |

</div>

&emsp;&emsp;<b>那为啥列表里没有 runc</b>:大部分 pod（coredns / traefik / etcd / minio / milvus / frontend 等）都跑在 runc 上,但 `kubectl get runtimeclass` 列表里没 `runc` 这一行。<b>因为 runc 是 K8s 的隐式默认 runtime,不需要 RuntimeClass 对象就能用</b>——`RuntimeClass` 对象只是给"非默认 runtime"做路由凭证,默认 runc 不需要这层路由。具体路径分两层:


<div align=center>

| 层 | 实际形态 | 怎么验证 |
|---|---|---|
| <b>containerd 层</b>（节点级） | containerd config.toml 里有 `[plugins.containerd.runtimes.runc]` 段定义 runc | `sudo grep runc /var/lib/rancher/k3s/agent/etc/containerd/config.toml` 能看到 |
| <b>K8s 层</b>（集群级） | <b>没有 runc 这个 RuntimeClass 对象</b>（因为是默认 runtime,不需要显式声明） | `kubectl get runtimeclass` 看不到 runc |

</div>

&emsp;&emsp;具体到 pod 行为:① pod yaml <b>不写</b> `runtimeClassName` 字段 → kubelet → containerd 默认 runtime（runc）,整个过程 K8s 层完全不查 RuntimeClass 对象;② pod yaml <b>写</b> `runtimeClassName: nvidia` → kubelet 查 K8s RuntimeClass 对象 `nvidia` → 拿到 `handler: nvidia` → containerd 找 `runtimes.nvidia` 段 → 起 nvidia-container-runtime。这是 K8s 的<b>显式才需要声明</b>设计——大部分容器走 runc 兜底,少数特殊需求（GPU / 强隔离 / WASM）才用 RuntimeClass 显式声明。

&emsp;&emsp;<b>本课只关心 `nvidia`</b>——其他 8 个有声明但 containerd 不认,本课不动它们。<b>前置二我们已经用 `nvidia-ctk --version` 确认过 NVIDIA Container Toolkit 装好了</b>——k3s 启动时会自动检测到 Toolkit、在 containerd config 里注册一个 `nvidia` runtime handler(上面表格那行说的就是这件事),所以这台机器的 `nvidia` RuntimeClass 大概率是真能用的。下面 grep 一下 containerd 配置坐实它:

```bash
# 看 containerd 实际配置, 应该有 nvidia runtime 段
sudo grep -A3 "containerd.runtimes.'nvidia'" /var/lib/rancher/k3s/agent/etc/containerd/config.toml
# 期望输出:
# [plugins.'...'.containerd.runtimes.'nvidia']
#   runtime_type = "io.containerd.runc.v2"
# [plugins.'...'.containerd.runtimes.'nvidia'.options]
#   BinaryName = "/usr/bin/nvidia-container-runtime"
```

&emsp;&emsp;看到 `BinaryName = /usr/bin/nvidia-container-runtime` 这一行就说明 k3s 检测到了 NVIDIA Container Toolkit 装的 runtime 二进制文件,把它自动加进 containerd config——这是 k3s "约定优于配置" 设计:**装好 Toolkit + 重启 k3s 后,nvidia runtime 自动 wire 上**,我们不用手动改 containerd 配置。

&emsp;&emsp;<b>有了 RuntimeClass,直接装 Device Plugin DaemonSet</b>。用自定义 DaemonSet yaml,关键三处:① pod spec 加 `runtimeClassName: nvidia`（引用 k3s 自动建好的 RuntimeClass）;② 容器镜像写 `nvcr.io/nvidia/k8s-device-plugin:v0.14.5`;③ 挂载宿主机 `/var/lib/kubelet/device-plugins` 目录（这是 k3s server kubelet 注册 device plugin socket 的路径,<b>不是</b> `/var/lib/rancher/...`）:

```yaml
# /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/nvidia-device-plugin-k3s.yaml(核心字段)
spec:
  template:
    spec:
      runtimeClassName: nvidia       # 关键 1:切到 nvidia 运行时
      containers:
      - name: nvidia-device-plugin
        image: nvcr.io/nvidia/k8s-device-plugin:v0.14.5
        volumeMounts:
        - name: device-plugin
          mountPath: /var/lib/kubelet/device-plugins
      volumes:
      - name: device-plugin
        hostPath:
          path: /var/lib/kubelet/device-plugins   # 关键 2:k3s kubelet 路径
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L2-device-plugin-flow-4fbd0905.png" width=80%></div>

&emsp;&emsp;<b>apply DaemonSet yaml</b>（本课配套包路径 `/home/XiaoYangWorkSpace/wensoutu-k8s/manifests/nvidia-device-plugin-k3s.yaml`）:

```bash
# 1. apply DaemonSet yaml
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/nvidia-device-plugin-k3s.yaml

# 2. 等 30 秒让 DaemonSet 把 pod 调度起来 + 拉镜像 + 注册 GPU 资源
sleep 30 && kubectl get pod -n kube-system -l name=nvidia-device-plugin-ds

# 3. 验证 GPU 资源是否成功注册到 server 节点
kubectl get node k3s-server -o jsonpath='{.status.capacity.nvidia\.com/gpu}'
# 输出: 4
```

&emsp;&emsp;看到 `4` 就说明 device plugin 已经把 4 块 RTX 3090 GPU 注册成功——`Capacity` 和 `Allocatable` 字段同时变成 `nvidia.com/gpu: 4`。<b>跑一个 cuda 测试 pod 实证 GPU 真的能用</b>:

```yaml
# /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/test-gpu-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: gpu-test
spec:
  runtimeClassName: nvidia          # pod 也要切 nvidia 运行时
  restartPolicy: Never
  nodeSelector:
    kubernetes.io/hostname: k3s-server   # 本课单节点拓扑,nodeSelector 是为生产多机扩展铺垫
  containers:
  - name: cuda
    image: nvcr.io/nvidia/cuda:12.0.0-base-ubuntu22.04   # 注意完整三段版本号
    command: ["nvidia-smi"]
    resources:
      limits:
        nvidia.com/gpu: 1            # 申请 1 块 GPU
```

```bash
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/test-gpu-pod.yaml
kubectl logs gpu-test
```

&emsp;&emsp;实测 `kubectl logs gpu-test` 输出 nvidia-smi 表格,关键信息包括 <b>Driver Version: 580.126.09</b>、<b>CUDA Version: 13.0</b>、看到 1 张 NVIDIA GeForce RTX 3090（BusId 00:AF:00.0）、显存 15MiB / 24576MiB——这块 GPU 是 device plugin 自动分配的 GPU 。

&emsp;&emsp;<b>两个反例要记住</b>:① 不加 `runtimeClassName: nvidia` → device plugin pod 报 NVML not found,Pod 状态变 CrashLoopBackOff;② 镜像 tag 写成 `nvcr.io/nvidia/cuda:12.0-base-ubuntu22.04`（两段版本号） → not found,nvcr.io 这个仓库要求完整三段 `12.0.0-base-ubuntu22.04`（NVIDIA 的镜像 tag 规范跟 Docker Hub 不同,Docker Hub 上 `nvidia/cuda:12.0-base-ubuntu22.04` 短 tag 是有的,nvcr.io 上没有,要拉对仓库的对应 tag）。

&emsp;&emsp;<b>建议:消费卡跑 K8s 多副本前先限功耗</b>。在 GPU 服务器上跑一次性配置:

```bash
sudo nvidia-smi -pm 1                # 开 persistence mode,防卡闲时低功耗态被瞬间唤醒打到峰值
sudo nvidia-smi -pl 220              # 把单卡功耗上限从 350W 限到 220W(himeshp.blogspot 4×3090 vLLM 实测最优点)
```

&emsp;&emsp;限功耗的作用是降低多卡并发推理时 PSU 瞬时压力和整机发热（第四章 4.5 节会讨论这台消费卡机器遇到的硬件稳定性问题）。本课所有实测都默认这套限功耗配置已生效。

&emsp;&emsp;<b>消费卡 vs 数据中心卡的硬件边界预告</b>（诚实划界）:本课硬件 4× RTX 3090 是<b>消费卡</b>,设计目标是单卡游戏 / 个人推理,不是为"K8s 多副本数据并行 + 持续高频推理"设计的。本课在第四章 4.5 节实测发现:这台机器在多副本 GPU 部署下会偶发整机 ssh 失联、需物理重启——挂机前内核日志没有 Xid / thermal / panic 记录,属于内核级硬挂,真因未能在驱动层确证,但跟请求并发量没有清晰因果（详见 4.5 节末尾"真实工程反例"段）。

&emsp;&emsp;<b>这不是 K8s 的问题</b>——是消费卡的物理边界:RTX 3090 不在 NVIDIA GPU Operator 官方支持矩阵里,没有 NVLink 4-way fully connected、没有 ECC 显存、没有多副本并发推理的官方压力测试覆盖。生产环境跑多副本 GPU 推理服务,硬件应该用<b>数据中心卡</b>（A100 / H100 / L40S / RTX A6000）,这些卡有 NVLink + ECC + 多年多副本压力测试,本课同一份 yaml 部署上去完全稳定。

&emsp;&emsp;<b>生产方式预告</b>:本节装的是<b>裸 NVIDIA Device Plugin DaemonSet</b>——这是 K8s 接 GPU 的最小依赖方案,适合入门和单机实验。生产环境推荐用 <b>NVIDIA GPU Operator</b>（Operator 模式统一管理 driver / container-toolkit / device-plugin / DCGM-exporter / NFD / GPU Feature Discovery / Validator 全套组件,带 operator 级自愈循环 + DCGM 健康检查 + 完整监控指标）。GPU Operator 的具体安装查 NVIDIA 官方文档 `docs.nvidia.com/datacenter/cloud-native/gpu-operator/`。

### 2.7 集群验证 + k9s 首次开箱

&emsp;&emsp;经过 2.2 到 2.5,我们手上有一套单节点 k3s 集群,server 节点上跑着 coredns / traefik / metrics-server / local-path-provisioner / nvidia-device-plugin 全部系统 pod,GPU 资源已注册成 `nvidia.com/gpu: 4`,工具栈齐全（kubectl 由 k3s 自带 + helm + k9s）,mirror 配置就位。这一节做一次完整的集群验证,并把 k9s 这个 TUI 观察台首次打开。

> <font size=2><b>【名词解释】</b><font color=red><b>TUI</b></font>（Text User Interface,终端图形界面）:用 ANSI 字符在终端里画窗口 / 表格 / 高亮的交互界面形态,比 GUI 轻、比纯命令行直观。</font>

&emsp;&emsp;<b>第一步用 kubectl 看节点</b>:

```bash
kubectl get nodes -o wide
```

&emsp;&emsp;期望看到一行 `k3s-server Ready control-plane`——单节点 k3s 集群的标准输出。

&emsp;&emsp;<b>第二步用 kubectl 看系统 pod</b>:

```bash
kubectl get pods -A
```

&emsp;&emsp;期望看到 kube-system 命名空间下 <b>8 个 pod</b>（6 个常驻 Running + 2 个 helm-install Job Completed）:

In [ ]:
NAMESPACE     NAME                                      READY   STATUS      RESTARTS
kube-system   coredns-8db54c48d-rkgvh                   1/1     Running     0
kube-system   helm-install-traefik-crd-xxxxx            0/1     Completed   0
kube-system   helm-install-traefik-yyyyy                0/1     Completed   2
kube-system   local-path-provisioner-5d9d9885bc-t4j77   1/1     Running     0
kube-system   metrics-server-786d997795-7vsh2           1/1     Running     0
kube-system   nvidia-device-plugin-daemonset-phng2      1/1     Running     0
kube-system   svclb-traefik-xxxxxxxx-zzzzz              2/2     Running     0
kube-system   traefik-9bcdbbd9-wwwww                    1/1     Running     0
default       gpu-test                                  0/1     Completed

&emsp;&emsp;<b>8 个 pod 各自的角色</b>:


<div align=center>

| Pod | 状态 | 角色 |
|---|---|---|
| `coredns` | Running | 集群 DNS,Service DNS 名（`milvus.wensoutu.svc.cluster.local` 等）解析靠它 |
| `local-path-provisioner` | Running | k3s 自带的本地存储动态 provisioner,PVC 触发它在节点本地分配 PV |
| `metrics-server` | Running | `kubectl top pod` / `kubectl top node` 的数据源 |
| `nvidia-device-plugin` | Running | 上一节装的,把 GPU 注册成 K8s 可调度资源 |
| `traefik` | Running | k3s 默认的 Ingress Controller（3.8 节部署 Ingress 时它来执行路由规则） |
| `svclb-traefik` | Running（2/2） | k3s 自带的 Klipper LoadBalancer DaemonSet,给 LoadBalancer Service 提供本地端口转发 |
| `helm-install-traefik-crd` | <b>Completed</b> | k3s 启动时跑的 Helm 安装 Job（装 Traefik CRD）,Job 跑完容器退出但 pod 记录保留 |
| `helm-install-traefik` | <b>Completed</b> | k3s 启动时跑的 Helm 安装 Job（装 Traefik 本体）,同上 |

</div>

&emsp;&emsp;<b>为啥 Completed 的 pod READY 是 `0/1` 不是 `1/1`</b>:`READY` 列显示的是<b>当前还活着的容器数 / pod 内总容器数</b>,Job 容器跑完任务以 `exit 0` 退出后,容器已经不在内存里跑了,所以 ready 数 = 0;但任务正常完成（exit code = 0）,所以 STATUS = `Completed`。<b>READY=0/1 + STATUS=Completed 配套出现是正常状态,不是 bug</b>。Running pod 显示 `1/1` 是因为容器还在跑、健康检查通过。

&emsp;&emsp;<b>Completed 跟 Running 平级</b>,都是 K8s 正常状态:前者表示"容器跑完任务正常退出"（适用 Job / CronJob 一次性任务）,后者表示"容器在持续运行"（适用 Deployment / StatefulSet / DaemonSet 常驻服务）。Completed 的 pod 不占算力,只是 etcd 里的元数据记录,留着方便排错查日志。

&emsp;&emsp;`gpu-test` 是 2.6 节我们跑的 cuda 测试 pod,状态 Completed（命令跑完正常退出）,不是失败状态。

&emsp;&emsp;<b>第三步用 k9s 把集群可视化打开</b>。直接敲:

```bash
k9s
```

&emsp;&emsp;k9s 启动后默认进入 default 命名空间的 pods 视图。常用导航命令（都是冒号开头的命令模式）:

In [ ]:
:no                   # 看 nodes 列表
:po --all-namespaces  # 看所有命名空间的 pods
:svc                  # 看 services
:ing                  # 看 ingress
:cm                   # 看 configmaps
:secret               # 看 secrets
:pv                   # 看 PV(持久卷)
:pvc                  # 看 PVC(持久卷声明)
:events               # 看集群事件流

&emsp;&emsp;在任何一个资源视图里,光标移到一行用快捷键操作:`d` describe、`e` 编辑、`l` 看日志、`s` 操作（在 pod 视图是进容器 shell,在 deployment 视图是弹出 scale 输入框改副本数——k9s 快捷键是上下文相关的）、`y` 看 yaml、`ctrl-d` 删除。退出 k9s 按 `:q` 或 `ctrl-c`。

&emsp;&emsp;<b>k9s 不只是只读观察台,也能改运行中的配置</b>。如果只是把挂掉、处于错误状态的 pod 清理掉,刚才的 `ctrl-d` 删掉就行,不用碰 yaml（controller 会自动按期望副本数补一个新的）。但如果要改的是<b>业务配置</b>（副本数 1 改 3、改环境变量、升级镜像 tag）,有两条路:

&emsp;&emsp;<b>路一:改 yaml 重新 apply（声明式,生产正道）</b>。改本地 `deployment.yaml` 里的 `replicas` / `image` / `env`,再 `kubectl apply -f deployment.yaml`。K8s 会 diff 新 yaml 跟集群现状,只动变了的部分做平滑滚动更新,不会把整套推倒重来。

&emsp;&emsp;<b>路二:k9s 里热编辑（临时救急）</b>。比如第三章部署 backend 之后,进 `:deploy` 选中 backend 按 `e`,k9s 调起 `$EDITOR`（默认 vim）弹出这个 Deployment 的完整 yaml,把 `replicas: 1` 改成 `3` 存盘 `:wq`,保存的瞬间 K8s 就收到变更开始拉新 pod;只改副本数的话更快的是按 `s`（scale）直接填数字。

&emsp;&emsp;<b>但热编辑有个工程纪律必须知道——配置漂移</b>:k9s 的 `e` / `s` 是<b>直接改集群里运行的对象</b>,不会动本地的 yaml 文件。这意味着热编辑改完集群后,本地 `deployment.yaml` 还是旧的——下次谁 `kubectl apply -f deployment.yaml` 一次,就把热改默默覆盖回去了;如果这套服务是 helm 装的（第三章 3.10 节）,热编辑还会跟 helm 记录的 release 状态对不上,下次 `helm upgrade` 同样覆盖。<b>所以 `e` / `s` 适合临时救急和调试,真要改业务配置,改 yaml 重新 apply（helm 管理的就改 values.yaml 跑 helm upgrade）才是不会丢的做法</b>。

&emsp;&emsp;<b>澄清一个常见误解</b>:k9s 不是只看节点 CPU/MEM 负载的工具——上面那些 `:no / :po / :svc / :ing / :cm / :secret / :pv / :pvc / :events` 命令展示的,是 K8s 资源对象的完整光谱（节点 / pod / 工作负载 / 服务 / 配置 / 卷 / 事件全在）。再加上实时 CPU/MEM 用量、pod 日志流、进 pod shell,k9s 是一个"用一个 TUI 终端覆盖 80% 集群运维操作"的工具。本课从这里到第五章每节实操,我们都会留一个常驻 k9s 终端看 pod 状态变化（3.9 节开始的多终端联动观察,这个常驻 k9s 会变成"终端 1 观察台"）。

&emsp;&emsp;<b>本章回顾</b>:我们现在手上有一套能跑 GPU 推理 pod 的单节点 K8s 集群、一份从零开始可复刻的命令清单、一份能让国内拉得动镜像的 mirror 配置、一份工具栈（kubectl 由 k3s 自带 + helm + k9s）,以及一段"单节点扩展到多机集群"的指引。下一章我们正式开始动 yaml,把第一门课跑通的文搜图 5 个服务（etcd / minio / milvus / backend / frontend）从 docker-compose 翻译成 K8s 工作负载,搬到这套集群上跑通。

---

## <center>第三章 把文搜图项目 5 个服务搬上 K8s 集群</center>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L3_第三章学习路径-d3b265df.png" width=80%></div>

&emsp;&emsp;第二章把集群和 GPU 底座搭好了,这一章正式动 yaml:先用 docker compose 看一眼活的文搜图应用,再把 5 个服务逐个翻译成 K8s 工作负载并搬上集群。我们会按 etcd / minio / milvus / backend / frontend 的顺序推进,中途用单副本压测看清 backend 的并发瓶颈,再把环境变量抽成 ConfigMap（yaml）和 Secret（命令行建）,最后把 7 个 manifest 打包成 1 个 helm Chart。走完本章,手上会有一份<b>完整的 K8s 部署清单</b>,下面这张拓扑图就是它最终拼出来的样子。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L3-five-services-topology-f48e5bf2.png" width=80%></div>

### 3.1 先用 docker compose 跑通文搜图应用

&emsp;&emsp;动 yaml 之前先用 docker-compose 把文搜图跑起来,确认要搬上 K8s 的是一个活着的、可访问的应用。先看见项目的具体形态,再看 yaml 字段,后面每个资源对象才有落点。

&emsp;&emsp;<b>启动命令一行跑齐 5 服务</b>:

```bash
cd /home/XiaoYangWorkSpace/wensoutu              # 项目根目录

# 拉起 prod 配置 5 服务 (up -d 是 detached, 后台跑, 立刻返回 shell)
docker compose -f docker-compose.prod.yml \
                --env-file .env.prod up -d

# 实时 follow backend 日志, 看模型 lazy load 跑到哪一步了
# 期望看到: "🚀 初始化 Qwen3-VL-Embedding..." → "加载 18 张图片" → "✅ 应用启动成功"
# Ctrl-C 退出 follow, backend 容器继续在后台跑(只是不再 tail 日志)
docker compose -f docker-compose.prod.yml logs -f backend
```

&emsp;&emsp;实测 5 个容器全部 healthy 后,`docker ps` 会看到:`wensoutu-etcd` / `wensoutu-minio` / `wensoutu-milvus` / `wensoutu-backend` / `wensoutu-frontend` 五个容器并排跑着。<b>端口分配</b>:frontend 映射宿主机 3000 端口、backend 映射 3011 端口（本来想用 3001 但被宿主机另一个 `fufanspace-backend.service` 占着、所以换 3011）、minio 内网 9000 不对外暴露、etcd / milvus 也都在 docker network 内网通信。

&emsp;&emsp;<b>backend 模型加载有等待</b>:backend 容器起来后,FastAPI lifespan 会从 ModelScope 缓存目录加载 Qwen3-VL-Embedding-2B 模型,<b>实测耗时约 80-97 秒</b>（模型已缓存到磁盘 + caption_cache 命中时）。这个 1-2 分钟级的数字是后面我们讨论"K8s 副本扩容快不快"时反复回到的关键——AI 推理服务跟普通 web 服务在副本启动速度上的根本差异就在这里。<b>注意</b>:K8s 场景下如果 caption_cache 走 emptyDir（每 pod 启动需重建）,时间会拉到 <b>130-180 秒（约 2-3 分钟）</b>;后面第三章 3.7 节、第四章 4.4 / 4.5 节用到这个数字时会注明是哪一档。

&emsp;&emsp;<b>用 curl 验证 backend 健康</b>:

```bash
curl http://127.0.0.1:3011/api/health
# {"status":"ok","service":"multimodal-rag-backend"}
```

&emsp;&emsp;<b>用浏览器跑一次文搜图</b>。前置课程的 `docker-compose.prod.yml` 出于<b>安全考虑把 frontend / backend 端口绑死在 `127.0.0.1` 而不是 `0.0.0.0`</b>——服务器本地能访问,从本机浏览器直接打 `http://<服务器 IP>:3000` 会撞 `ERR_EMPTY_RESPONSE`（docker-proxy 只接受 localhost 来源的连接）。验证一下端口确实是 localhost 绑定:

```bash
# 服务器上看 frontend / backend 容器的端口映射, 注意 PORTS 列前面带 127.0.0.1
docker ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}' | grep -E 'frontend|backend'
# wensoutu-frontend   Up X (healthy)   127.0.0.1:3000->80/tcp     ← 关键: 绑死 localhost
# wensoutu-backend    Up X (healthy)   127.0.0.1:3011->3001/tcp   ← 同上
```

&emsp;&emsp;<b>用 ssh 隧道把本机 localhost:3000 转发到服务器 127.0.0.1:3000</b>——在<b>本机</b>另开一个终端跑,不是 ssh 上服务器后跑:

```bash
# 本机敲: -L 选项是 local port forwarding (本地端口转发)
ssh -L 3000:127.0.0.1:3000 -L 3011:127.0.0.1:3011 4GPU24G
# 隧道建好后保持这个终端开着, 本机就能用 http://localhost:3000 访问
# (4GPU24G 是本机 ~/.ssh/config 里配的 Host 别名, 换成自己的 ssh 连接配置即可)
```

&emsp;&emsp;隧道开着的情况下,<b>本机浏览器打开 `http://localhost:3000`</b>,在搜索框里输中文关键词（譬如"猫"）,后端把文本编码成向量、在 milvus 里检索图片向量、minio 里返回图片 URL,前端把图列出来——五个服务协同工作,我们看到一个活的多模态检索系统。

> 💡 <b>不用 ssh 隧道的话还可以直接服务器上 curl 测</b>:`curl http://127.0.0.1:3011/api/health` 看 backend 健康,或者发个 `curl -X POST http://127.0.0.1:3011/api/search ...` 直接看 JSON 返回——本节末尾"端到端 curl 验证"段就是用这种方式。但浏览器跑一次能直观看到图,推荐 ssh 隧道方案。

&emsp;&emsp;<b>看完之后 docker compose 怎么处理</b>:浏览器跑完文搜图,5 个 docker 容器还在后台跑着。<b>3.2 到 3.5 节我们写 K8s yaml,不动服务,docker compose 一直跑没关系</b>——只是 backend 容器占 GPU 0 的 12.6 GiB 显存。真正必须停 docker compose 是 <b>3.7 节部署 K8s backend 之前</b>（那时候 K8s 扩 3 副本会把 GPU 0/1/2 全用上,留 GPU 0 给 docker 会冲突）;3.7 节我们会显式跑一次 `docker compose down` 释放。<b>如果浏览器看完之后立刻想停 compose 释放显存,也直接跑下面这条</b>:

```bash
cd /home/XiaoYangWorkSpace/wensoutu
docker compose -f docker-compose.prod.yml down     # 停 5 容器, 释放 GPU + 端口
```

&emsp;&emsp;<b>先看 compose 再写 K8s 的原因</b>:先看见要迁移的应用和服务关系,再看 yaml 字段才有锚点。如果上来直接写 etcd StatefulSet 的 volumeClaimTemplates,etcd 在文搜图里承担元数据存储的画面还没建立,字段也难以理解。

&emsp;&emsp;<b>本课配套 yaml 文件位置</b>:第三章 3.3 到 3.10 节要 apply 的所有 yaml 文件（`etcd-statefulset.yaml` / `minio-statefulset.yaml` / `milvus-statefulset.yaml` / `backend-deployment.yaml` / `frontend-deployment.yaml` / `ingress.yaml` / `configmap.yaml` 共 7 个 manifest）+ 3.10 节用的 helm Chart 都预先放在<b>课件配套代码包</b>的 `wensoutu-k8s/` 目录里（跟前置课程 `wensoutu/` 项目<b>同级目录</b>）。Secret 用 secret.yaml apply,但真实 key 不入仓——配套代码包只给占位的 `secret.example.yaml`,你 cp 成 secret.yaml 填真实 key（见 3.6 节）:

```bash
# 把课件配套代码包解压到 $HOME 后, 项目结构是:
/home/XiaoYangWorkSpace/
├── wensoutu/                    # 前置课程产出, docker compose 跑这套
│   ├── docker-compose.prod.yml
│   ├── backend/
│   ├── frontend/
│   └── volumes/                 # models / images / uploads / caption_cache 数据卷
└── wensoutu-k8s/                # 本课配套, K8s yaml + Chart + 脚本
    ├── manifests/               # K8s yaml(第二章 3 个 + 第三章 8 个, 逐个 apply)
    │   ├── etcd-statefulset.yaml
    │   ├── minio-statefulset.yaml
    │   ├── milvus-statefulset.yaml
    │   ├── backend-deployment.yaml
    │   ├── frontend-deployment.yaml
    │   ├── ingress.yaml
    │   ├── configmap.yaml
    │   └── secret.example.yaml  # 占位模板(真实 secret.yaml 你 cp 填 key, 不入仓)
    ├── wensoutu-chart/          # helm Chart, 3.10 节打包用
    └── reset.sh                 # 一键清场脚本
```

&emsp;&emsp;课件正文也给出了每份 yaml 的完整内容,我们可以选:① 直接 apply 配套包里的 yaml 路径（快,推荐第一遍跑通）;② 照课件正文 yaml 自己 `vim` 创建到任意路径再 apply（深度学习字段时推荐）。

### 3.2 5 服务编排选型

&emsp;&emsp;动 yaml 前最关键的一次决策,是把 5 个服务一一对应到 K8s 资源类型。选错了——比如有状态服务用 Deployment——后面重启 Pod 就丢数据,milvus 的索引全没。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L3-compose-vs-k8s-04e58fed.png" width=80%></div>

&emsp;&emsp;<b>5 服务对照表</b>:


<div align=center>

| 服务 | docker-compose 形态 | K8s 资源类型 | 选型理由 |
|---|---|---|---|
| etcd | 单容器 + volume | <b>StatefulSet + PVC</b> | 有状态:存元数据,重启 pod 要复用同一份磁盘,否则数据全丢 |
| minio | 单容器 + volume | <b>StatefulSet + PVC</b> | 有状态:对象存储,重启复用同一份磁盘 |
| milvus | 单容器 + 依赖 etcd/minio | <b>StatefulSet + PVC + initContainer</b> | 有状态:向量索引;启动顺序敏感（必须等 etcd / minio Ready） |
| backend | 单容器 + GPU | <b>Deployment + `nvidia.com/gpu`</b> | 无状态:模型存镜像里, 副本数可以随便扩 |
| frontend | 单容器 + 静态站 | <b>Deployment</b> | 无状态:nginx 服静态文件 |

</div>

&emsp;&emsp;表里 6 个核心术语（StatefulSet / Deployment / PVC / PV / headless Service / initContainer）1.5 节已经名词解释过,这里不再重复。

&emsp;&emsp;<b>本课所有镜像版本统一说明</b>:本课 yaml 里用的镜像（`etcd:v3.5.5` / `minio:RELEASE.2023-03-20` / `milvus:v2.3.21` / `nginx:1.27-alpine` / `busybox:1.36` / `cuda:12.0.0` / `device-plugin:v0.14.5`）都是前置《文搜图项目》课程产出的本机已有镜像 + 已实测兼容性的选型,直接复用避免重新拉。<b>生产新部署不要原封不动用本课版本</b>——尤其几条已知风险:


<div align=center>

| 镜像 | 本课版本 | 生产建议 + 风险 |
|---|---|---|
| **etcd** | v3.5.5（2023 初） | 升 3.5.28+ patch 版,**v3.5.5 有 CVE-2026-33343 / CVE-2026-33413 认证绕过漏洞**（milvus 用的元数据 etcd 受影响,K8s 控制面那个 etcd 不受影响） |
| **MinIO** | RELEASE.2023-03-20 | **MinIO 开源 CE 2026-04 上游归档 EOL**,不再适合生产。生产新部署考虑 AIStor（MinIO 付费版） / Ceph / SeaweedFS / Garage 等替代 |
| **milvus** | v2.3.21（2024 初） | 升 v2.5.27+（含 CVE-2026-26190 critical fix）或 v2.6.x（2.6 系列重构架构、降低资源消耗） |
| nginx-alpine / busybox / cuda / device-plugin | 见上 | 跟随各项目当前 stable 即可,不是关键风险点 |

</div>

&emsp;&emsp;所有跨大版本升级要看官方迁移指南（<https://etcd.io/docs/v3.5/upgrades/> / <https://milvus.io/docs/release_notes.md>）,涉及索引格式 / 配置参数变更,不是改一行 `image:` 就完事。

&emsp;&emsp;<b>有状态服务必须用 StatefulSet</b>:Deployment 的 Pod 名是 `backend-7d4f6c-xxx` 这种带随机哈希的,重启后哈希变了 Pod IP 也变了,而且所有副本共享同一个 PVC——如果给 milvus 用 Deployment,3 个副本想读同一份索引,容器写冲突就崩了。StatefulSet 给每个 Pod 一个稳定名（`milvus-0` / `milvus-1` / `milvus-2`）和独立 PVC,重启后名字不变、磁盘还是原来那块,这才是有状态服务的标准做法。

&emsp;&emsp;<b>容易踩的坑</b>:5 服务全用 Deployment 看起来"简单",但 milvus 重启后向量索引全丢、minio 重启后图片全丢、etcd 重启后元数据全丢——本课文搜图就只剩个空壳。这是 Compose 课跨进 K8s 时最常见的认知翻车,后面 3.3-3.5 节我们就按这个表逐个落地。

### 3.3 部署 etcd StatefulSet

&emsp;&emsp;<b>前置一步:先创建 namespace</b>。本课所有 yaml 都声明了 `namespace: wensoutu`,K8s 不会自动创建 namespace,所以在动第一份 yaml 之前必须显式建好,否则 `kubectl apply` 会报 `namespaces "wensoutu" not found`:

> <font size=2><b>【名词解释】</b><font color=red><b>namespace</b></font>（K8s 命名空间）:K8s 集群里对资源做<b>逻辑隔离</b>的一层标签,同一 namespace 内资源名必须唯一,跨 namespace 的同名资源互不冲突。本课所有业务资源（StatefulSet / Deployment / Service / ConfigMap / Secret / PVC）都统一落在 `wensoutu` 这个 namespace 下,好处是:① `kubectl get pods -n wensoutu` 一条命令看全部业务 pod 不混系统组件;② 一键清理整套服务直接 `kubectl delete namespace wensoutu` 全删;③ 跨项目部署到同一集群时各自占一个 namespace 不打架。K8s 默认 namespace 叫 `default`,系统组件落在 `kube-system`。</font>

```bash
# 先看当前集群已经有哪些 namespace, 确认 wensoutu 还不存在
kubectl get namespace
# NAME              STATUS   AGE
# default           Active   2h     ← K8s 默认 namespace, 没指定 namespace 的对象都落这里
# kube-node-lease   Active   2h     ← 存节点心跳租约
# kube-public       Active   2h     ← 集群公开信息, 任何身份都可读
# kube-system       Active   2h     ← K8s 系统组件 (coredns / traefik / metrics-server ...)

# 新建 wensoutu namespace, 本课业务 5 服务全放在这个 namespace 里
kubectl create namespace wensoutu
# namespace/wensoutu created

# 再看一次, 确认 wensoutu 已经在列表里
kubectl get namespace
# 期望多出一行: wensoutu          Active   1s
```

&emsp;&emsp;etcd 是 milvus 的元数据存储,我们从 etcd 起头,把<b>StatefulSet + headless Service + PVC</b>这套有状态服务的最小完整 yaml 模板建立起来——后面 minio / milvus 都是按这套模板填字段。先看一张图,把 StatefulSet apply 之后 K8s 内部发生的 5 步串起来:

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L3-statefulset-pvc-flow-549a2d79.png" width=80%></div>

&emsp;&emsp;<b>etcd 完整 yaml</b>（保存在 `/home/XiaoYangWorkSpace/wensoutu-k8s/manifests/etcd-statefulset.yaml`）:

```yaml
# headless Service: 给每个 etcd Pod 提供稳定 DNS 名
apiVersion: v1
kind: Service
metadata:
    name: etcd
    namespace: wensoutu
spec:
    clusterIP: None                         # headless: 不要 ClusterIP
    selector:
        app: etcd
    ports:
        - port: 2379
          name: client
        - port: 2380
          name: peer
---
# StatefulSet 本体
apiVersion: apps/v1
kind: StatefulSet
metadata:
    name: etcd
    namespace: wensoutu
spec:
    serviceName: etcd                       # 绑定上面那个 headless Service
    replicas: 1                             # 本课单副本即可
    selector:
        matchLabels:
            app: etcd
    template:
        metadata:
            labels:
                app: etcd
        spec:
            nodeSelector:
                kubernetes.io/hostname: k3s-server   # 第二章决定的硬约束:全部钉到 server
            containers:
                - name: etcd
                  image: quay.io/coreos/etcd:v3.5.5
                  command:
                      - etcd
                      - --data-dir=/var/lib/etcd
                      - --listen-client-urls=http://0.0.0.0:2379
                      - --advertise-client-urls=http://etcd-0.etcd:2379
                      - --listen-peer-urls=http://0.0.0.0:2380
                      - --initial-advertise-peer-urls=http://etcd-0.etcd:2380   # 必填: 否则 etcd 启动 fatal
                      - --initial-cluster=default=http://etcd-0.etcd:2380       # 单副本本课用 default 名字
                  ports:
                      - containerPort: 2379
                        name: client
                      - containerPort: 2380
                        name: peer
                  volumeMounts:
                      - name: etcd-data
                        mountPath: /var/lib/etcd     # etcd 数据盘
    volumeClaimTemplates:                    # 自动建 PVC 的模板
        - metadata:
              name: etcd-data
          spec:
              accessModes: ["ReadWriteOnce"]
              resources:
                  requests:
                      storage: 10Gi
```

&emsp;&emsp;<b>这份 yaml 的关键看点</b>:① `kind: Service` 加 `clusterIP: None` 是 headless Service,K8s 不分配 ClusterIP,只为 Pod 注册 DNS 名 `etcd-0.etcd.wensoutu.svc.cluster.local`——后面 milvus 连 etcd 就走这个 DNS。② `kind: StatefulSet` 的 `serviceName: etcd` 必须跟前面那个 headless Service 名字一致,这是 StatefulSet 跟 headless Service 的强绑定。③ `volumeClaimTemplates` 是 StatefulSet 独有字段,等同于"每个 Pod 起来时自动 `kubectl create pvc`"——本课单副本就只生成一个 `etcd-data-etcd-0` 的 PVC,3 副本时会自动生成 3 个独立 PVC。

&emsp;&emsp;<b>apply 之后 k9s 看到的过程</b>:

```bash
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/etcd-statefulset.yaml
k9s                                       # :sts 看 StatefulSet, :po 看 Pod, :pvc 看 PVC
```

&emsp;&emsp;在 k9s 里我们会看到 PVC 状态从 `Pending` → `Bound`（等本地路径 PV 落盘）,Pod 状态从 `Pending` → `ContainerCreating` → `Running`,整个过程几秒钟。<b>验证 etcd 真的活着</b>:

```bash
kubectl exec -n wensoutu etcd-0 -- etcdctl endpoint health
# 127.0.0.1:2379 is healthy: successfully committed proposal: took = 1.2ms
```

&emsp;&emsp;<b>最常见的错误形态</b>:headless Service 的 `clusterIP: None` 漏写,K8s 会给 etcd 分一个 ClusterIP,这时候 milvus 连 `etcd-0.etcd` 的 DNS 名解析不出 Pod IP——会一直连不上 etcd 看似"网络不通",其实是 Service 类型错了。下一节我们用同一份模板再写一份 minio StatefulSet,主要看 Service 类型上的差异（minio 是普通 ClusterIP 不是 headless）。

### 3.4 部署 minio StatefulSet

&emsp;&emsp;minio 是文搜图项目的对象存储,存的是图片二进制文件。yaml 跟 etcd 大同小异,本节重点看<b>跟 etcd 不一样的两点</b>:一是 Service 类型——minio 用普通 ClusterIP,不需要 headless;二是端口暴露策略——minio 只内网访问,不对外开 host 端口。

&emsp;&emsp;<b>minio 完整 yaml</b>（`/home/XiaoYangWorkSpace/wensoutu-k8s/manifests/minio-statefulset.yaml`）:

```yaml
# 普通 ClusterIP Service: 集群内 backend 通过这个名字访问 minio
apiVersion: v1
kind: Service
metadata:
    name: minio
    namespace: wensoutu
spec:
    type: ClusterIP                         # 注意: 不是 headless,有 ClusterIP 做 L4 负载均衡
    selector:
        app: minio
    ports:
        - port: 9000
          name: api
        - port: 9001
          name: console
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
    name: minio
    namespace: wensoutu
spec:
    serviceName: minio                       # 即使是 ClusterIP, StatefulSet 仍然要绑 Service
    replicas: 1
    selector:
        matchLabels:
            app: minio
    template:
        metadata:
            labels:
                app: minio
        spec:
            nodeSelector:
                kubernetes.io/hostname: k3s-server
            containers:
                - name: minio
                  image: minio/minio:RELEASE.2023-03-20T20-16-18Z
                  command: ["minio", "server", "/data", "--console-address", ":9001"]
                  env:
                      - name: MINIO_ROOT_USER
                        value: minioadmin
                      - name: MINIO_ROOT_PASSWORD
                        value: minioadmin                # minio 服务端 root 密码; backend 连它时用 3.6 节 Secret 里的同名 key, 两边要一致
                  ports:
                      - containerPort: 9000
                      - containerPort: 9001
                  volumeMounts:
                      - name: minio-data
                        mountPath: /data
    volumeClaimTemplates:
        - metadata:
              name: minio-data
          spec:
              accessModes: ["ReadWriteOnce"]
              resources:
                  requests:
                      storage: 20Gi
```

&emsp;&emsp;<b>跟 etcd 的两点不同</b>:① Service `type: ClusterIP` 取代了 etcd 的 `clusterIP: None`——minio 只需要"集群内被 backend 一个名字访问",有 ClusterIP 做 L4 负载均衡完全够用,不需要每个 Pod 独立 DNS。② 没有对外 NodePort——minio 在文搜图里只跟 backend 通信（backend 调 minio 存图片 / 取图片）,不需要从浏览器直接访问,所以内网访问足够。

&emsp;&emsp;<b>apply 之后验证</b>:

```bash
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/minio-statefulset.yaml
# 在常驻 k9s 终端 :po wensoutu 看 minio-0 状态序列:
#   Pending → ContainerCreating → Running, 整个过程 50-90 秒
# 看到 READY=1/1 STATUS=Running 再跑下面验证命令

# minio 镜像是精简版, 容器内不带 mc / curl / wget — 直接从集群外 curl Service ClusterIP 验证
MINIO_IP=$(kubectl get svc minio -n wensoutu -o jsonpath='{.spec.clusterIP}')
curl -sf -w 'HTTP %{http_code}\n' http://$MINIO_IP:9000/minio/health/live
# 期望: HTTP 200, 说明 minio 服务起来且能响应健康检查
```

### 3.5 部署 milvus + initContainer 等依赖

&emsp;&emsp;milvus 是 5 服务里最复杂的一个——既要 StatefulSet + PVC 装索引,又依赖 etcd 和 minio,启动顺序敏感。本节用 K8s 的 <b>initContainer</b> 机制确保 milvus 主容器启动前 etcd 和 minio 都已经 Ready,避免出现"milvus 起得太快、etcd 还没 Ready、milvus 连不上 etcd 立刻退出"的崩溃循环。

&emsp;&emsp;<b>initContainer 在 milvus yaml 里的位置</b>（只列关键字段,完整 yaml 较长）:

```yaml
# ClusterIP Service: backend 通过 milvus:19530 访问 milvus
apiVersion: v1
kind: Service
metadata:
    name: milvus
    namespace: wensoutu
spec:
    type: ClusterIP
    selector:
        app: milvus
    ports:
        - port: 19530
          name: grpc
        - port: 9091
          name: metrics
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
    name: milvus
    namespace: wensoutu
spec:
    serviceName: milvus                            # serviceName 引用上面的 milvus Service, 本课单副本不依赖 Pod DNS
    replicas: 1
    selector:
        matchLabels:
            app: milvus
    template:
        metadata:
            labels:
                app: milvus
        spec:
            nodeSelector:
                kubernetes.io/hostname: k3s-server
            initContainers:                         # 主容器之前按顺序跑
                - name: wait-for-etcd                # 第一个 initContainer: 等 etcd
                  image: busybox:1.36
                  command:
                      - sh
                      - -c
                      - until nc -z etcd 2379; do echo "waiting for etcd..."; sleep 2; done
                - name: wait-for-minio               # 第二个 initContainer: 等 minio
                  image: busybox:1.36
                  command:
                      - sh
                      - -c
                      - until nc -z minio 9000; do echo "waiting for minio..."; sleep 2; done
            containers:
                - name: milvus
                  image: milvusdb/milvus:v2.3.21
                  command: ["milvus", "run", "standalone"]
                  env:
                      - name: ETCD_ENDPOINTS
                        value: etcd-0.etcd:2379       # 用 etcd 的 headless DNS 名
                      - name: MINIO_ADDRESS
                        value: minio:9000             # 用 minio 的 ClusterIP DNS 名
                  ports:
                      - containerPort: 19530
                        name: grpc
                      - containerPort: 9091
                        name: metrics
                  livenessProbe:                      # K8s 自动健康探针
                      httpGet:
                          path: /healthz
                          port: 9091
                      initialDelaySeconds: 30
                      periodSeconds: 10
                  readinessProbe:                     # 就绪后才接流量
                      httpGet:
                          path: /healthz
                          port: 9091
                      initialDelaySeconds: 15
                      periodSeconds: 5
                  volumeMounts:
                      - name: milvus-data
                        mountPath: /var/lib/milvus
    volumeClaimTemplates:
        - metadata:
              name: milvus-data
          spec:
              accessModes: ["ReadWriteOnce"]
              resources:
                  requests:
                      storage: 50Gi
```

> <font size=2><b>【名词解释】</b><font color=red><b>Probe</b></font>（健康探针,liveness/readiness）:K8s 内置的容器健康检查机制,分 liveness / readiness / startup 三档,两两独立运行,各管各的。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>livenessProbe</b></font>（存活探针）:用于检测"卡死了"——失败时 kubelet 自动重启容器。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>readinessProbe</b></font>（就绪探针）:用于检测"暂时没准备好接流量"——失败时 Service 把这个 Pod 从负载均衡里临时摘除。</font>

&emsp;&emsp;<b>apply 之后看 initContainer 等依赖 + milvus 主容器起来</b>:

```bash
# 1. apply milvus StatefulSet (含 Service + initContainer 2 个 + 主容器 + PVC)
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/milvus-statefulset.yaml

# 2. 看 initContainer 顺序: wait-for-etcd → wait-for-minio → milvus 主容器
kubectl get pod milvus-0 -n wensoutu -w
# 期望状态序列: Init:0/2 → Init:1/2 → PodInitializing → Running

# 3. 看 milvus 启动完成日志 (约 2-3 分钟)
kubectl logs milvus-0 -n wensoutu -f --tail=10
# 期望最后看到: "[INFO] [indexnode/indexnode.go:276] [get IndexNode components states ...]"
```

&emsp;&emsp;<b>两个 initContainer 顺序执行</b>:K8s 保证 `wait-for-etcd` 跑完（exit 0）才跑 `wait-for-minio`,后者跑完才启动 `milvus` 主容器。这是<b>声明式依赖等待</b>的标准用法——比在主容器里写 `sleep 30` 强百倍,因为它真的等 etcd 9000 端口响应才往下走。

&emsp;&emsp;<b>livenessProbe + readinessProbe 的分工</b>:livenessProbe 是"容器还活着吗"（失败就 kill 重启）,readinessProbe 是"容器准备好接流量了吗"（失败就从 Service 后端摘除但不重启）。milvus 启动初期 readinessProbe 还失败、Service 不会把流量发给它;启动完成后 readinessProbe 通过、Service 开始把流量打进来;一旦 milvus 死锁导致 livenessProbe 失败、kubelet 直接重启容器。两层探针配合 = K8s 自愈机制的底层武器。

&emsp;&emsp;<b>容易踩的坑</b>:不用 initContainer 直接起 milvus 主容器,milvus 几秒内启动、发现 etcd 还没 Ready、立刻退出 → kubelet 检测到容器 Exit、按 Deployment/StatefulSet 策略立刻重启 → 再次启动太快又退出 → 进入 `CrashLoopBackOff` 状态,K8s 按指数退避重试越来越久。这种"看似 yaml 正确但服务起不来"的崩溃循环,是 Compose 玩家上 K8s 最容易翻车的场景。三件有状态服务(etcd / minio / milvus)到这就搭完了。下一节先把 backend 要用的配置和密钥建成 ConfigMap / Secret,再正式部署 backend 这个无状态 GPU 推理服务。

### 3.6 ConfigMap / Secret 准备配置与密钥

&emsp;&emsp;三件有状态服务搭完了,下一节就要部署 backend。在那之前,这一节先把 backend 要用的配置建好。<b>ConfigMap</b> 是 K8s 用来存非敏感配置的对象,内容是明文 key=value 的 yaml,可以 git 提交;<b>Secret</b> 是用来存敏感数据的对象,value 用 base64 编码,生产环境配合 RBAC 限制只有授权 SA 能读。前面 etcd / minio / milvus 的 yaml 里配置都直接写在 yaml 里(比如 minio 的 `MINIO_ROOT_USER: "minioadmin"`)——默认值这样够用,但 backend 的 OpenRouter API key 这种敏感数据不能明文写进 git。

> <font size=2><b>【名词解释】</b><font color=red><b>ConfigMap</b></font>（K8s 配置对象）:存非敏感配置（端口、地址、特性开关）,yaml 明文可 git 提交,改了不用重新 build 镜像,pod 重启就生效。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>Secret</b></font>（K8s 敏感数据对象）:存账号密码、API key、证书私钥,value base64 编码（注意 base64<b>不是加密</b>只是编码,生产环境靠 RBAC + Sealed Secret / Vault 等方案做真实加密）。</font>

&emsp;&emsp;<b>为什么 backend 之前先建这两个对象</b>:下一节要部署的 backend 需要一批环境变量——既有 `MILVUS_URI: http://milvus:19530` 这类内部 Service 地址(非敏感),也有 OpenRouter API Key 这类敏感数据。敏感字段如果直接写进 backend-deployment.yaml 提交到 git,任何人 clone 仓库都能拿到。所以我们<b>先把这些变量按"敏感 / 非敏感"分别建成 Secret 和 ConfigMap</b>,3.7 节部署 backend 时用 `envFrom` 一行引用——敏感 key 不进 backend yaml、不进 git。

&emsp;&emsp;<b>ConfigMap yaml</b>（`/home/XiaoYangWorkSpace/wensoutu-k8s/manifests/configmap.yaml`,非敏感字段,跟代码一起提交 git）:

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
    name: backend-config
    namespace: wensoutu
data:
    MILVUS_URI: "http://milvus:19530"
    OPENAI_BASE_URL: "https://openrouter.ai/api/v1"
    BACKEND_HOST_PORT: "3001"
```

&emsp;&emsp;ConfigMap 不含敏感信息,直接 apply:

```bash
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/configmap.yaml
# configmap/backend-config created
```

&emsp;&emsp;<b>Secret 用 secret.yaml apply 建,但真实 key 不进 git</b>。配套代码包里给的是 <b>secret.example.yaml</b>（占位模板),你复制成 secret.yaml、填上自己的真实 key 再 apply:

```bash
# 1. 先去 OpenRouter (https://openrouter.ai/) 注册账号 + 创建一个 API key
#    key 长这样: sk-or-v1-xxxxxxxx... (字符串前缀 sk-or-v1)

# 2. 复制模板成 secret.yaml(真实 key 写进这份, 这份不进 git)
cd /home/XiaoYangWorkSpace/wensoutu-k8s/manifests
cp secret.example.yaml secret.yaml

# 3. 编辑 secret.yaml, 把占位的 xxxxxxxx 换成你自己的真实 key
vim secret.yaml

# 4. apply
kubectl apply -f secret.yaml
# secret/backend-secret created

# 5. 验证 (能看到 key 数量, Opaque 类型默认不显示 value)
kubectl get secret backend-secret -n wensoutu
# NAME             TYPE     DATA   AGE
# backend-secret   Opaque   3      5s
```

&emsp;&emsp;secret.yaml 的内容长这样——`stringData` 直接写明文 key,`kubectl apply` 时 K8s 自动做 base64 编码存进 etcd:

```yaml
apiVersion: v1
kind: Secret
metadata:
    name: backend-secret
    namespace: wensoutu
type: Opaque                                # 默认通用 Secret 类型
stringData:                                 # 写明文, apply 时自动 base64
    OPENROUTER_API_KEY: "sk-or-v1-你的真实key"
    OPENAI_API_KEY: "sk-or-v1-你的真实key"     # backend 走 openai SDK 读这个, 跟上面同值
    MINIO_ROOT_PASSWORD: "你的minio密码"
```

> <font size=2><b>【为什么给 example 模板】</b>真实 key 写在 secret.yaml 里,这份文件<b>不能提交 git</b>（`.gitignore` 已排除 secret.yaml）。所以配套代码包里只放占位的 <b>secret.example.yaml</b>,你 `cp` 成 secret.yaml 填真实值即可。这是工程上分发"含密钥配置"的常见做法——模板进 git、真实值留本地,既能让别人知道"要配哪几个 key",又不泄露密钥。</font>

&emsp;&emsp;<b>生产环境 3 种 Secret 管理方案</b>（本课用方案 1,方案 2/3 是生产常见进阶）:


<div align=center>

| 方案 | 工作机制 | 适用场景 |
|---|---|---|
| <b>1. secret.yaml apply + .gitignore</b>（本课用） | 真实 key 写 secret.yaml（不进 git）,配套只发占位 secret.example.yaml | 小团队 / 单一环境,密钥手动维护 |
| <b>2. Sealed Secrets</b>（Bitnami 出品） | 用集群公钥加密 Secret yaml,加密后的密文 yaml 可放心提交 git;集群里的 Sealed Secrets Controller 解密成普通 Secret | 多人协作 + GitOps 工作流（Argo CD / Flux） |
| <b>3. External Secrets Operator + Vault</b>（企业级） | Secret 存在外部 secrets 管理系统（HashiCorp Vault / AWS Secrets Manager / Azure Key Vault）,ESO 拉取并同步成 K8s Secret | 跨集群 / 跨环境 / 审计合规要求高 |

</div>

&emsp;&emsp;<b>backend Deployment 用 envFrom 一行引用全部</b>:

```yaml
spec:
    template:
        spec:
            containers:
                - name: backend
                  image: yanggggg/wensoutu-backend:0.2
                  envFrom:
                      - configMapRef:
                            name: backend-config       # 一次性注入 ConfigMap 全部 key
                      - secretRef:
                            name: backend-secret       # 一次性注入 Secret 全部 key
```

&emsp;&emsp;<b>envFrom 的好处</b>:不用一个一个 `env: name/value` 写,ConfigMap 加新字段 backend 自动拿到。改 ConfigMap 或 Secret 之后 pod 不会自动重启（K8s 这点跟 docker-compose 的 `env_file` 行为一样）,需要手动 `kubectl rollout restart deployment/backend` 触发滚动重启。

&emsp;&emsp;<b>确认两个对象都建好了</b>(ConfigMap 和 Secret 上面已经各自 apply 过):

```bash
kubectl get configmap,secret -n wensoutu | grep backend-
# configmap/backend-config   3      ...
# secret/backend-secret      Opaque   3   ...
```

&emsp;&emsp;建好之后,下一节部署 backend 时,它的 yaml 里已经写好 `envFrom` 引用这两个对象,`kubectl apply` 一上去就把全部环境变量注入,敏感 key 也不出现在 backend yaml 里。<b>一个常错点</b>:之后改了 ConfigMap / Secret,运行中的 pod <b>不会自动重载</b>(跟 docker-compose `env_file` 一样),要 `kubectl rollout restart deployment/backend` 才让新值生效。

### 3.7 部署 backend（单副本单卡）

&emsp;&emsp;上一节 ConfigMap 和 Secret 都建好了,这一节正式部署 backend。backend 是文搜图项目的核心——FastAPI 服务跑 Qwen3-VL-Embedding-2B 多模态模型,负责把文本和图片编码成向量、调 milvus 检索。<b>它是无状态的</b>——向量数据存在 milvus 不在 backend 本身,模型权重和图床数据沿用宿主机已有目录,所以用 Deployment;但它申请 GPU,所以 yaml 里有<b>第二章 2.6 节</b>建好的 `runtimeClassName: nvidia` + `nvidia.com/gpu: 1` 这一套。这是整门课"<b>K8s 编排 GPU 推理服务</b>"主题在 yaml 层面的第一次具象。

&emsp;&emsp;<b>前置:准备 backend / frontend 镜像到 k3s containerd</b>。本课 backend / frontend 镜像发布在 Docker Hub 个人 namespace:`yanggggg/wensoutu-backend:0.2`、`yanggggg/wensoutu-frontend:0.2`。

&emsp;&emsp;远程 k3s 节点创建 Pod 时,实际是由 k3s 内置的 containerd 拉取镜像,它<b>不会直接使用 Docker daemon 里的本地镜像</b>。因此有两种方式:

&emsp;&emsp;<b>方式 A:让 k3s 从远程镜像仓库拉取</b>。如果 Docker Hub / TCR / Harbor 网络可用,Pod 可以直接拉镜像。

&emsp;&emsp;<b>方式 B:提前把镜像导入 k3s containerd</b>。如果镜像较大,或者远程拉取不稳定,推荐先在服务器上通过 `docker pull` / `docker save` 获取镜像,再导入 k3s containerd 的 `k8s.io` namespace:

```bash
docker images | grep yanggggg

sudo apt install -y pv

docker save yanggggg/wensoutu-backend:0.2 | \
    pv -s $(docker image inspect yanggggg/wensoutu-backend:0.2 --format='{{.Size}}') | \
    sudo k3s ctr -n k8s.io images import -

docker save yanggggg/wensoutu-frontend:0.2 | \
    sudo k3s ctr -n k8s.io images import -

sudo k3s ctr -n k8s.io images ls | grep yanggggg
```

&emsp;&emsp;这里<b>必须带 `-n k8s.io`</b>。因为 containerd 支持 namespace 隔离,`ctr` 默认操作的是 `default` namespace,而 K8s 的 kubelet 使用 `k8s.io` namespace。镜像只有导入 `k8s.io` namespace,K8s 创建 Pod 时才看得到。

&emsp;&emsp;<b>动 yaml 前先停 docker compose 的 backend</b>:第 3.1 节我们用 docker compose 跑起来的 `wensoutu-backend` 容器现在还在宿主机上占着 <b>GPU 0 的 12.6 GiB 显存</b>——实测它会把 Embedding + Reranker 两个 Qwen3-VL 模型一直常驻显存。K8s backend Pod apply 之后,K8s 调度器会按 `nvidia.com/gpu: 1` 找一块"显存空闲"的 GPU 分配,GPU 0 已被占的话会调度到 GPU 1/2/3 之一,所以理论上不冲突。<b>但如果第四章 4.5 节扩到 3 副本</b>,4 块 GPU 减去 docker 占的 GPU 0,只剩 3 块够用,刚好压满——再扩到 4 副本时 GPU 0 抢不到,新 Pod 直接 OOM 启动失败。最干净的做法是动 K8s yaml 之前先把 docker compose 整套停掉:

```bash
cd /home/XiaoYangWorkSpace/wensoutu
docker compose -f docker-compose.prod.yml down     # 停掉 5 个 docker 容器, 释放 GPU 0
nvidia-smi                                          # 确认 GPU 0 显存回到 ~15 MiB 接近全空
```

&emsp;&emsp;<b>backend 完整 yaml</b>（`/home/XiaoYangWorkSpace/wensoutu-k8s/manifests/backend-deployment.yaml`）:

```yaml
apiVersion: v1
kind: Service
metadata:
    name: backend
    namespace: wensoutu
spec:
    type: ClusterIP
    selector:
        app: backend
    ports:
        - port: 3001                          # 集群内 service 端口
          targetPort: 3001                    # 容器内 FastAPI 端口
---
apiVersion: apps/v1
kind: Deployment
metadata:
    name: backend
    namespace: wensoutu
spec:
    replicas: 1                                # 这里先 1 副本; 4.5 节会扩到 3
    selector:
        matchLabels:
            app: backend
    template:
        metadata:
            labels:
                app: backend
        spec:
            runtimeClassName: nvidia            # 用 NVIDIA Container Runtime
            nodeSelector:
                kubernetes.io/hostname: k3s-server   # 本课单节点拓扑,nodeSelector 是为生产多机扩展铺垫
            initContainers:
                - name: wait-for-milvus              # 等 milvus 起来再启动 backend
                  image: busybox:1.36
                  command:
                      - sh
                      - -c
                      - until nc -z milvus 19530; do echo "waiting for milvus..."; sleep 2; done
            volumes:                                 # 复用宿主机 docker compose 留下的数据目录
                - name: models                       # 8 GB 模型权重: Embedding + Reranker 两份
                  hostPath:
                      path: /home/XiaoYangWorkSpace/wensoutu/volumes/models
                      type: Directory
                - name: images                       # 图床: 18 张待检索 PNG
                  hostPath:
                      path: /home/XiaoYangWorkSpace/wensoutu/volumes/images
                      type: Directory
                - name: uploads                      # 用户上传的查询图临时目录
                  hostPath:
                      path: /home/XiaoYangWorkSpace/wensoutu/volumes/uploads
                      type: DirectoryOrCreate
                - name: caption-cache                # caption 文本缓存
                  hostPath:
                      path: /home/XiaoYangWorkSpace/wensoutu/volumes/caption_cache
                      type: DirectoryOrCreate
            containers:
                - name: backend
                  image: yanggggg/wensoutu-backend:0.2
                  ports:
                      - containerPort: 3001
                  env:                                  # 容器内固定值, 不随环境变, 留在这
                      - name: MODELSCOPE_CACHE             # 让 modelscope SDK 直接读本地缓存
                        value: /app/backend/data/models
                      - name: HF_HOME                       # transformers / huggingface 缓存指向同目录
                        value: /app/backend/data/models/.hf
                      - name: HF_HUB_OFFLINE               # 离线模式: 不联网拉权重
                        value: "1"
                      - name: TRANSFORMERS_OFFLINE         # 同上, transformers 的开关
                        value: "1"
                      - name: MINIO_ROOT_USER               # minio 用户名(非敏感, 默认 minioadmin)
                        value: "minioadmin"
                  envFrom:                                # 其余配置从 3.6 节建好的 ConfigMap + Secret 注入
                      - configMapRef:
                            name: backend-config           # MILVUS_URI / OPENAI_BASE_URL / BACKEND_HOST_PORT
                      - secretRef:
                            name: backend-secret           # OPENROUTER_API_KEY / OPENAI_API_KEY / MINIO_ROOT_PASSWORD
                  volumeMounts:
                      - { name: models,        mountPath: /app/backend/data/models }
                      - { name: images,        mountPath: /app/backend/data/images }
                      - { name: uploads,       mountPath: /app/backend/data/uploads }
                      - { name: caption-cache, mountPath: /app/backend/data/caption_cache }
                  resources:
                      limits:
                          nvidia.com/gpu: 1                # 申请 1 块 GPU
                          memory: 16Gi
                          cpu: "4"
                      requests:
                          memory: 12Gi
                          cpu: "2"
```

&emsp;&emsp;<b>三个关键字段一次说清</b>:① `runtimeClassName: nvidia` 告诉 K8s 这个 Pod 要用 NVIDIA Container Runtime 起容器,这是第二章 2.6 节装 NVIDIA Device Plugin 时同步注册的 RuntimeClass。② `nodeSelector: kubernetes.io/hostname: k3s-server` 在本课单节点环境下其实只有 k3s-server 一个候选,scheduler 自动调度过去——保留这个字段是因为生产多机集群时它会发挥真正作用（把 GPU pod 钉到带卡节点）,本课的 yaml 跟生产保持一致便于将来扩展。③ `resources.limits.nvidia.com/gpu: 1` 申请 1 块 GPU,K8s 调度器会从 server 节点上 4 块 GPU 里挑一块没被占用的分配给这个 Pod。

&emsp;&emsp;<b>配置从哪来</b>:`env` 里只留 4 个容器固定值 + minio 用户名;milvus 地址、OpenRouter API key、minio 密码这些走 `envFrom` 引用 3.6 节建好的 `backend-config` ConfigMap 和 `backend-secret` Secret。所以本节 apply backend 之前,3.6 节那两个对象必须已经建好,否则 pod 会报 `CreateContainerConfigError`——好处是敏感 key 一行都不出现在这份 backend yaml 里。

> <font size=2><b>【名词解释】</b><font color=red><b>nvidia.com/gpu</b></font>（K8s 扩展资源名）:NVIDIA Device Plugin 注册到 K8s 的自定义资源,pod 用 `resources.limits.nvidia.com/gpu: 1` 申请一块 GPU（下一节 4.2 节会专门展开这条强约束的工程含义）。</font>

&emsp;&emsp;<b>hostPath 复用宿主机已有数据目录的设计</b>:yaml 里 4 个 `hostPath` 卷指向 `/home/XiaoYangWorkSpace/wensoutu/volumes/` 下的 `models` / `images` / `uploads` / `caption_cache`——这恰好是 docker compose 时代 backend 容器挂载的同一组目录。我们沿用这套数据有两个收益:① <b>模型权重不用重下</b>（`models` 目录里已经放着 Qwen3-VL-Embedding-2B 和 Reranker 两份共 8 GB 权重,从 modelscope 下完整一遍要 10+ 分钟,k8s 这边直接复用）;② <b>图床和检索缓存沿用 docker compose 时已有的样本</b>（`images` 目录 18 张待检索 PNG、`caption_cache` 里的文本缓存）,省掉重新生成。

> <font size=2><b>【名词解释】</b><font color=red><b>hostPath</b></font>（K8s 宿主机路径卷）:把宿主机上的目录直接挂载到 Pod 容器里,K8s 不负责管理,只做一层 bind mount。优点是数据跟 Pod 解耦（Pod 删了数据还在）,缺点是 Pod 必须钉死在挂载源所在的节点（本课所有业务 pod 都钉到 k3s-server,刚好规避这个限制）。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>caption_cache</b></font>（本课项目特有目录）:reranker 用的图描述文本缓存,18 张图各对应一个 `.txt` 文件,首次启动需要给每张图调模型生成 caption,从零跑约 60-120 秒;复用 hostPath 缓存可秒级跳过。</font>

&emsp;&emsp;<b>配套的 4 个 offline env 是关键</b>:`MODELSCOPE_CACHE` 把 modelscope SDK 的下载目录指到挂载点,`HF_HOME` 同步指过去,然后 `HF_HUB_OFFLINE=1` + `TRANSFORMERS_OFFLINE=1` 把所有"找不到就联网下载"的回落路径关掉,强制 SDK 只读本地。少配任何一个 env,modelscope 都会"看不见本地权重,以为没下载",触发重新下载 8 GB,在国内网速下要等 10+ 分钟,整个实操节奏被拖垮。

> <font size=2><b>【名词解释】</b><font color=red><b>4 个 offline env</b></font>（`MODELSCOPE_CACHE` / `HF_HOME` / `HF_HUB_OFFLINE` / `TRANSFORMERS_OFFLINE`）:告诉 modelscope 和 transformers SDK 只读本地权重不联网下载;少配任何一个,SDK 都会"看不见本地权重",触发重新下载 8 GB Qwen3-VL 模型,国内网速下要等 10+ 分钟。</font>

&emsp;&emsp;<b>关于 MILVUS_URI 这个细节</b>:lab-records 实测过——这里必须用 K8s 的 Service DNS 名 `http://milvus:19530`,<b>不能用 host:port 形式（如 `http://192.168.110.131:19530`）</b>。Pod 跟 host 之间走的不是同一个网络,host IP 在 Pod 里不通;但 Service 名 `milvus` 在 wensoutu namespace 内任何 Pod 都能解析。这是 Compose 跨 K8s 时最容易卡的网络认知。

&emsp;&emsp;<b>apply 之后看 backend 起来</b>:

```bash
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/backend-deployment.yaml
k9s                                          # :po 看 backend Pod, 选中按 l 看 logs
```

&emsp;&emsp;在 k9s 的 logs 视图里我们能看到 FastAPI lifespan 从本地 `data/models` 加载 Qwen3-VL-Embedding-2B,因为 4 个 offline env 都配了,modelscope 不会重新下载;18 张图 embedding 1.5 秒跑完,然后 Reranker 模型加载完成,<b>实测约 80 秒后 Pod 进 Running 状态</b>——本节用 hostPath 挂载 caption_cache,缓存命中所以快。

&emsp;&emsp;但要注意:本节 yaml 还没配 readinessProbe,K8s 看进程在就标 Ready=1/1,Service 立刻开始打流量进来——但 FastAPI lifespan 可能还在加载模型,前几秒请求会撞 5xx。这是第三章入门版的简化,等到 4.6 节我们会演进出 readinessProbe + startupProbe 的完整探针方案,把"应用真就绪"这件事告诉 K8s。Pod 跑起来后,进 Pod 跑 `nvidia-smi`:

```bash
kubectl exec -n wensoutu deploy/backend -- nvidia-smi
# 看到一块 RTX 3090 24GB, GPU 0/1/2/3 中的一块被分配, 约 12 GiB 显存被 Embedding+Reranker 两个模型占用
```

&emsp;&emsp;<b>backend 单跑验证留到 3.8 节做完 Ingress 之后再统一端到端 curl</b>——本节聚焦"GPU pod 编排起来"这件事,验证标准就是 pod READY=1/1 + `kubectl exec ... nvidia-smi` 看到 GPU 0 被占。下一节装 Ingress 把外部 HTTP 请求路由到 backend / frontend,届时一次 curl 测完整条链路。

&emsp;&emsp;<b>这一节的关键</b>:GPU 推理服务的 K8s 编排比 web 服务多两件事——`runtimeClassName: nvidia`（指定容器运行时）、`nvidia.com/gpu: 1`（申请 GPU 资源）,其它跟普通 Deployment 完全一样。这两件事配齐,K8s 就能像调度 CPU / 内存一样调度 GPU。<b>叠加 hostPath 数据复用 + 4 个 offline env 这套组合</b>,我们就把"模型权重不用打进镜像"和"docker compose 历史数据无缝迁移"两件事一次解决——这是把传统 docker compose 项目搬上 K8s 时,普遍会踩的两个坑。

### 3.8 部署 frontend Deployment + 配 Ingress

&emsp;&emsp;frontend 是 nginx 服一个静态网站,无状态、无依赖、不挂卷,用 Deployment 就行。本节重点是<b>Ingress</b>——我们要把"集群外的 HTTP 请求"按 URL 路径分发给 frontend（`/`）和 backend（`/api/*`）,这是 K8s 暴露服务给外部访问的标准做法。本课用 Ingress + k3s 默认 Traefik 这条入门路径——Traefik 同时支持 Ingress 和 Gateway API,学完后切换到 Gateway API yaml 是平滑升级（1.4.2 节展开过两者关系）。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L3-ingress-routing-62e24def.png" width=80%></div>

&emsp;&emsp;<b>frontend Deployment + Service yaml</b>（`/home/XiaoYangWorkSpace/wensoutu-k8s/manifests/frontend-deployment.yaml`）:

```yaml
apiVersion: v1
kind: Service
metadata:
    name: frontend
    namespace: wensoutu
spec:
    type: ClusterIP
    selector:
        app: frontend
    ports:
        - port: 80
          targetPort: 80
---
apiVersion: apps/v1
kind: Deployment
metadata:
    name: frontend
    namespace: wensoutu
spec:
    replicas: 1
    selector:
        matchLabels:
            app: frontend
    template:
        metadata:
            labels:
                app: frontend
        spec:
            nodeSelector:
                kubernetes.io/hostname: k3s-server
            containers:
                - name: frontend
                  image: yanggggg/wensoutu-frontend:0.2     # nginx:1.27-alpine 基础镜像
                  ports:
                      - containerPort: 80
```

&emsp;&emsp;<b>Ingress yaml</b>（`/home/XiaoYangWorkSpace/wensoutu-k8s/manifests/ingress.yaml`）:

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
    name: wensoutu
    namespace: wensoutu
spec:
    ingressClassName: traefik                  # 用 k3s 内置 Traefik, 不是 nginx-ingress
    rules:
        - http:
              paths:
                  - path: /api                  # 所有 /api/* 走 backend
                    pathType: Prefix
                    backend:
                        service:
                            name: backend
                            port:
                                number: 3001
                  - path: /                     # 其它路径走 frontend
                    pathType: Prefix
                    backend:
                        service:
                            name: frontend
                            port:
                                number: 80
```

> <font size=2><b>【名词解释】</b><font color=red><b>Ingress</b></font>（K8s 入口对象）: K8s 内置 kind,声明"按域名/路径把外部 HTTP 请求路由到内部 Service"。Ingress 自身不实现路由,需要 Ingress Controller（Traefik / nginx-ingress / 等）实际执行规则。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>ingressClassName</b></font>（Ingress 资源字段）:告诉 K8s 这条 Ingress 规则由哪个 Controller 负责;k3s 上必须填 `traefik` 而不是 `nginx`（填错了规则不生效）。</font>

&emsp;&emsp;<b>apply 之后浏览器访问验证</b>:

```bash
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/frontend-deployment.yaml
kubectl apply -f /home/XiaoYangWorkSpace/wensoutu-k8s/manifests/ingress.yaml
kubectl get ingress -n wensoutu
# NAME       CLASS     HOSTS   ADDRESS                ...
# wensoutu   traefik   *       192.168.110.131        ...
```

&emsp;&emsp;<b>用浏览器打开 `http://192.168.110.131/`</b>——看到文搜图首页;打开 `http://192.168.110.131/api/health`——看到 backend 健康 JSON。Traefik 按 path 把请求分发到对应 Service,Service 再 L4 负载均衡到 Pod,链路就通了。

&emsp;&emsp;<b>容易踩的坑</b>:`ingressClassName: nginx` 是 K8s 通用文档里出现频率很高的写法,但<b>k3s 上必须改成 `traefik`</b>,因为 k3s 内置的是 Traefik 而不是 nginx-ingress。直接照搬 nginx 示例,apply 之后 Ingress 看起来 Status 正常,但 80 端口没有对应的 nginx-ingress 接管请求 → 浏览器一直 ECONNREFUSED,很容易误判成网络问题。下一节我们离开 yaml 编写,做整门课<b>第一次三个终端联动观察</b>——先看单 backend pod 在并发请求下的串行瓶颈,为第四章多副本对照打基线。

### 3.9 三个终端联动首演:看见 backend 单 pod 的并发瓶颈

&emsp;&emsp;前面几节我们把 5 个服务的 yaml 写完、apply 完,服务在跑了。但 K8s 的关键能力不只在"写 yaml apply",还在于它后续如何调度、分发和恢复服务。本节用 <b>shell `&` 并发 curl</b> 模拟"几个用户同时点搜索"的真实瞬时流量,单副本 backend,用<b>三个终端联动</b>的方式看见一件事:单 pod 处理并发请求时被完全串行化,N 个并发被 GIL + 单 CUDA stream 排队成一个流。这套观察方式整门课会反复用,后面 4.5 / 4.6 节的 GPU 测试全部沿用。

&emsp;&emsp;<b>为啥用 shell `&` 并发 curl 模拟</b>:推理服务的真实流量是 <b>open-loop with think time</b>——用户点一下、看结果、思考几秒、再点,是离散事件,RPS 大概 0.01-0.1。shell `&` 并发 curl 是最接近"几个用户碰巧同时点击"的瞬时模拟,不依赖外部工具,跟推理服务的真实流量模型对齐。

&emsp;&emsp;<b>先压单副本的价值</b>:单副本压测的信息密度更大,能看到 backend `async def search` 调 sync 函数阻塞 event loop 的硬数据,也能为后续多副本对照留下基线。本节不做 `kubectl scale --replicas=3`,这个动作留到第四章 4.5 节,届时配合 "1 副本 vs N 副本" 对比,K8s 多副本的真实价值才显得清晰。

&emsp;&emsp;<b>三个终端联动方式</b>的核心:一个屏幕上同时盯三件事——K8s 在做什么（终端 1）、谁在制造负载（终端 2）、效果落到哪（终端 3）。本节按单副本剧本分工:


<div align=center>

| 终端 | 命令 | 看的事 |
|---|---|---|
| 终端 1 观察台 | `k9s -n wensoutu`,按 `:po` 进 pod 视图 | 1 个 backend pod 的状态（READY / Running）、CPU 跑满情况（`%CPU/R` 列） |
| 终端 2 施压器 | `for i in 1 2 3; do curl ... & done; wait` 然后改 `1 2 3 4 5` 再跑一次 | 制造并发负载,看每个请求响应时间 + 总耗时 |
| 终端 3 处理证据 | `kubectl logs -f -n wensoutu -l app=backend` | 观察单 pod 处理 3 并发的串行过程 |

</div>

&emsp;&emsp;<b>关于 GPU 占用</b>:k9s 默认 `:po` 视图只显示 CPU / MEM 列（K8s 标准资源）,<b>不带 GPU 列</b>（`nvidia.com/gpu` 是 K8s 扩展资源,不在默认 metrics 里）。想看 GPU 显存 / util 实时变化,另开一个终端跑 `nvidia-smi` watch,跟上面三个终端不抢空间:

```bash
# 1 秒刷新看 GPU 0/1/2/3 显存 + util 实时变化
watch -n 1 nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv,noheader
# 期望: 推理时 GPU 0 显存 ~12.6 GiB(Embedding+Reranker 常驻), util 飙到 80-100%
```

&emsp;&emsp;<b>怎么制造并发最直观</b>:开 3 个浏览器窗口对着文搜图页面同时点搜索,你会看到它们一起转圈、再陆续返回结果——这就是"几个用户同时用"的真实场景。下面我们把同一件事用 shell `&` 并发 curl 脚本化,好处是能精确看到每个请求的返回时刻、看清单个 pod 怎么把并发请求排成一列串行处理。curl 发 HTTP POST 到 Ingress 入口 `/api/search`,body 走 `SearchRequest` schema（`textQuery` / `searchMode` / `recallTopK` / `rerankTopK` / `threshold` 5 个必填字段）,用 shell `&` 并发 + `wait` 等所有返回:

```bash
# 并发 3 curl(模拟 3 个用户同时点搜索):
START=$(date +%s.%N)
for i in 1 2 3; do
    (curl -s -X POST http://192.168.110.131/api/search \
        -H 'Content-Type: application/json' \
        -d "{\"textQuery\":\"query $i\",\"searchMode\":\"文搜图\",\"recallTopK\":10,\"rerankTopK\":3,\"threshold\":0}" \
        -o /dev/null -w "req$i: HTTP %{http_code}, %{time_total}s\n") &
done
wait
END=$(date +%s.%N)
echo "总耗时: $(echo "$END - $START" | bc)s"
```

&emsp;&emsp;<b>这份 payload schema 是从源码读出来的实测结果, 不是凭印象</b>。第一次跑的时候我们用 `{"query":"cat"}` 一直 500, 后来 `kubectl exec` 进 backend pod 看 `/app/backend/schemas.py` 里的 `SearchRequest` 类才发现真字段叫 `textQuery`——这个实测教训的意思是: <b>curl 请求里的 payload 字段名必须跟 backend Pydantic schema 完全对齐</b>, 错一个字母就 422/500, 跟 K8s 没关系。

&emsp;&emsp;<b>跑下来会看到一个清晰的规律</b>:单个请求本身就要花上十秒上下（Embedding 编码 + Milvus 召回 + Reranker 重排完整 pipeline 的物理耗时,GPU 推理就是这么重）;3 个并发一起发,它们<b>不是同时返回,而是响应时间呈阶梯式递增</b>——第一个最先回,第二个要多等大约一个单请求的时间,第三个再多等一个,<b>总耗时大致等于"并发数 × 单请求耗时"</b>,而且全部成功不丢请求。

&emsp;&emsp;这就是 3 个请求被 <b>GIL + 单 CUDA stream 串行化、依次占用 GPU</b> 的现象——后到的请求只能在 uvicorn 连接队列里干等前面的跑完。<b>单 pod 吞吐是物理封顶,加并发只让排队等待变长,不增加 RPS（每秒请求数）</b>。具体秒数每台机器、每次跑都会浮动,但"阶梯式递增、总耗时随并发数成倍涨"这个规律是稳定的——你在自己环境跑上面的脚本,看 `time_total` 输出就能复现这个阶梯。

&emsp;&emsp;<b>单 pod 的并发瓶颈</b>——回到 backend 的 search router 源码:

In [ ]:
# backend/routers/search.py
@router.post("", response_model=SearchResponse)
async def search(request: SearchRequest):
    return retrieval_engine.search(...)        # sync 调用,阻塞 event loop

&emsp;&emsp;`async def` 在 FastAPI 里看似异步, 但 `retrieval_engine.search(...)` 是 sync 函数, 内部串起 Qwen3-VL Embedding 编码 + Milvus 向量召回 + Reranker 重排, 每一步都是 CPU/GPU 同步阻塞。一旦执行, uvicorn 的 event loop 整个被卡住, 新进来的请求只能在 uvicorn 内部的连接队列里等。叠加 Python GIL + 单 CUDA stream + 单例模型（backend 启动时只加载一份 Embedding/Reranker 权重）, 单 pod 处理 N 并发请求 ≈ 串行处理 N 个请求, p95 ≈ N × 单请求耗时。

> <font size=2><b>【名词解释】</b><font color=red><b>async def 调 sync 函数</b></font>（FastAPI 反模式）:FastAPI 里 `async def` 路由函数如果内部调用 sync 阻塞函数（没 `await`）, event loop 会被这个调用整个卡住,期间其它请求只能排队。正确做法是用 `def`（starlette 会自动放到 threadpool 跑）,或者把 sync 函数包成 `run_in_executor`。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>GIL</b></font>（Global Interpreter Lock, Python 全局解释器锁）:CPython 同一时刻只允许一个线程执行 Python 字节码,多线程压不动 CPU 密集型工作。GPU 推理虽然主算力在 CUDA,但 PyTorch 前后处理 / Numpy 数组操作 / Pydantic 序列化都吃 GIL。</font>

&emsp;&emsp;<b>K8s 在这里的价值</b>:K8s <b>不能让单 pod 变快</b>——单 pod 内部的 async/sync 改造、worker 数量优化都跟 K8s 无关;K8s 的价值是<b>横向扩</b>,通过 Service + N 副本,把 N 个并发请求分到 N 个 pod 各自串行处理,总 RPS 提升 N 倍。这就是第四章 4.5 节要展开的"多副本对延迟的实际价值"——届时我们会扩到 3 副本重跑同样的并发 curl,用实测对比看 K8s 横向扩的效果。

&emsp;&emsp;本课单节点拓扑下,4.5 / 4.6 节扩到 3 副本时, 3 个 backend pod 全部跑在 k3s-server 这<b>一台机器</b>上（`nodeSelector: kubernetes.io/hostname: k3s-server`）, K8s 的调度发生在"分 GPU 卡"层面而不是"分机器"层面——它把 3 个 pod 分配到同节点的 3 块不同 GPU 上。生产多机集群下 K8s 自动把副本分散到不同节点, 机制完全一样, 只是我们这台单机服务器演不出节点间分散。

### 3.10 helm Chart 打包整套服务

&emsp;&emsp;<b>helm</b> 是 K8s 上的<b>包管理工具</b>（类比 Ubuntu 的 apt、Python 的 pip）,它把一组相关的 K8s yaml 打包成一个<b>Chart</b>,通过 `helm install` 一行命令完成整套服务的部署。前面 7 节我们手工写了 <b>7 个 manifest 文件</b>（etcd / minio / milvus 各一个 StatefulSet + Service,backend / frontend 各一个 Deployment + Service,加上 Ingress + ConfigMap——合计 7 个 yaml 文件,内含约 12 个 K8s 资源对象;Secret 由 3.6 节 apply secret.yaml 建,真实 key 不进 git）,`kubectl apply -f` 要敲 7 次,改副本数或镜像 tag 要在多个文件里找。本节我们把这 7 个 manifest 抽成一个 helm Chart 统一管理。

> <font size=2><b>【名词解释】</b><font color=red><b>Chart</b></font>（helm 的包格式）:一个目录,内含 Chart.yaml（包元数据）、values.yaml（可配置参数）、templates/（用 Go template 渲染的 yaml 模板）,`helm install` 时把 values 注入 templates 渲染出最终 yaml 再 apply。</font>

&emsp;&emsp;<b>从无到有创建 Chart 目录</b>（本课配套包里 `wensoutu-chart/` 已经准备好,直接用即可;下面一起跑一遍,理解 Chart 怎么从 7 个 manifest 长出来）:

```bash
# 1. 进 wensoutu-k8s 目录(跟 manifests/ 同级建 Chart 目录)
cd /home/XiaoYangWorkSpace/wensoutu-k8s

# 2. helm create 一行生成 Chart 骨架(标准目录结构)
helm create my-chart
# 自动建出: my-chart/Chart.yaml + values.yaml + templates/ (含一堆 nginx 假数据示例)

# 3. 删 helm create 生成的 nginx 假数据示例(都跟本课无关)
rm -rf my-chart/templates/* my-chart/charts my-chart/.helmignore

# 4. 把前面 7 个 manifest 拷进 templates/ (Secret 不拷, 见下面注)
cp manifests/{etcd,minio,milvus,backend,frontend,ingress,configmap}*.yaml my-chart/templates/

# 5. 用 sed 把 backend / frontend 两个 yaml 的硬编码 replicas / image 换成 {{ .Values.xxx }} 占位符
#    (cp 来的 yaml 全是硬编码定量, 必须改成变量后 --set 命令才能生效)
sed -i \
    -e 's|replicas: 1|replicas: {{ .Values.backend.replicas }}|' \
    -e 's|image: yanggggg/wensoutu-backend:0.2|image: "{{ .Values.backend.image }}:{{ .Values.backend.tag }}"|' \
    my-chart/templates/backend-deployment.yaml

sed -i \
    -e 's|replicas: 1|replicas: {{ .Values.frontend.replicas }}|' \
    -e 's|image: yanggggg/wensoutu-frontend:0.2|image: "{{ .Values.frontend.image }}:{{ .Values.frontend.tag }}"|' \
    my-chart/templates/frontend-deployment.yaml

# 6. 改 values.yaml 把上面 4 个占位符的实际值集中写进去 (覆盖 helm create 默认 nginx 假数据)
cat > my-chart/values.yaml <<'EOF'
backend:
    image: yanggggg/wensoutu-backend
    tag: "0.2"
    replicas: 1
frontend:
    image: yanggggg/wensoutu-frontend
    tag: "0.2"
    replicas: 1
EOF

# 7. 验证占位符在 yaml 里, 跟 values.yaml 对应得上
grep -E 'replicas:|image:' my-chart/templates/backend-deployment.yaml
# 期望: replicas: {{ .Values.backend.replicas }}
#       image: "{{ .Values.backend.image }}:{{ .Values.backend.tag }}"
```

&emsp;&emsp;<b>步骤 1-7 是一次性制作 Chart 的过程</b>——尤其步骤 5 的 `sed` 把硬编码换成占位符,只在建 Chart 时做这一次。Chart 做好之后,<b>日常改配置(换镜像 tag、扩缩副本)只改 `values.yaml` 那几行就够了,templates 和 sed 都不用再碰</b>。

&emsp;&emsp;<b>注:Secret 不进 Chart</b>（跟 3.6 节做法一致）,由 `helm install` 之前 `kubectl apply -f secret.yaml` 单独建好,key 不进 git 也不进 Chart。

&emsp;&emsp;<b>Chart 目录结构</b>（`/home/XiaoYangWorkSpace/wensoutu-k8s/wensoutu-chart/`）:

In [ ]:
wensoutu-chart/
├── Chart.yaml                       # 包元数据(name / version / description)
├── values.yaml                      # 可配置参数(副本数 / 镜像 tag)
└── templates/
    ├── etcd.yaml                    # etcd StatefulSet + headless Service
    ├── minio.yaml                   # minio StatefulSet + Service
    ├── milvus.yaml                  # milvus StatefulSet + Service + initContainer
    ├── backend.yaml                 # backend Deployment + Service
    ├── frontend.yaml                # frontend Deployment + Service
    ├── ingress.yaml                 # Traefik Ingress 路由
    └── configmap.yaml               # 非敏感配置
# 注: Secret 不放 Chart, 由 helm install 之前 kubectl apply -f secret.yaml 单独建好
#    (跟 3.6 节的做法一致, key 不进 Chart 也不进 git)

&emsp;&emsp;<b>values.yaml 把可变参数抽出来</b>:

```yaml
backend:
    image: yanggggg/wensoutu-backend
    tag: "0.2"                            # 改这一行就能换镜像版本
    replicas: 1                           # 改这一行就能扩缩副本
frontend:
    image: yanggggg/wensoutu-frontend
    tag: "0.2"
    replicas: 1
```

&emsp;&emsp;<b>templates/backend.yaml 用模板语法引用</b>:

```yaml
spec:
    replicas: {{ .Values.backend.replicas }}
    template:
        spec:
            containers:
                - name: backend
                  image: "{{ .Values.backend.image }}:{{ .Values.backend.tag }}"
```

&emsp;&emsp;<b>Chart 跟原本 manifests/ 的 yaml 区别在哪</b>:核心就两件事——<b>把值从结构里抽出来</b>,以及<b>把一组 yaml 当一个整体来管理</b>。前面 3.3-3.8 节我们写的 manifests/ 是<b>静态 yaml</b>:`replicas: 1`、`image: ...:0.2` 这些值直接写死在文件里,要改副本数就得进对应文件找到那行改。Chart 把这些<b>会变的值</b>抽进 `values.yaml`,模板里只留 `{{ .Values.backend.replicas }}` 这样的占位符——结构（templates/）和值（values.yaml）分开,改值不动结构。

<div align=center>


<div align=center>

| 维度 | 原本 manifests/ 静态 yaml | helm Chart |
|---|---|---|
| 可变值写在哪 | 写死在每个 yaml 文件里 | 集中抽到 `values.yaml`,模板用 `{{ .Values.xxx }}` 占位 |
| 改一个参数 | 进对应文件找到那行改 | 改 `values.yaml` 一处,或 `helm install` 时 `--set` 临时覆盖 |
| 部署方式 | `kubectl apply -f` 逐个文件敲 | `helm install` 一行装整套 |
| 多环境 | 每个环境复制一份 yaml 各自改值 | 一份模板 + `values-dev.yaml` / `values-prod.yaml` 切换 |
| 版本管理 | apply 后 K8s 不记得"这批是一组" | helm 记录 release 版本,支持 `upgrade` / `rollback` / `uninstall` 整套 |

</div>

</div>

&emsp;&emsp;<b>一个容易误解的点</b>:templates/ 里<b>不是每个文件都得模板化</b>。本课只有 backend / frontend 抽了 `replicas` / `image` 两个可变参数,etcd / minio / milvus / ingress / configmap 这 5 个是<b>原样拷进 templates/ 没动一个字</b>——它们没有需要按环境变的参数,写死就行。helm 对"纯静态 yaml"和"带 `{{ }}` 占位的模板"一视同仁地渲染,静态的渲染完还是原样。<b>所以 Chart 不要求把每个值都变量化,只把真正会变的（镜像 tag、副本数）抽出来就够了</b>——过度变量化反而让 Chart 难读。

&emsp;&emsp;<b>实操命令</b>（<b>⚠️ 前置必读</b>:跑 `helm install` 前必须先清空 namespace,否则 helm 会跟前面 3.3-3.8 节手工 apply 的对象冲突,报 `Secret/ConfigMap exists` 拒绝接管。最好直接用 `kubectl delete namespace wensoutu --wait` 整体清场,清完再跑下面命令）:

```bash
# 0. 前置清场(只在跑 helm install 时执行一次)
kubectl delete namespace wensoutu --wait   # 等 Terminating 完成,约 30-60 秒

# 前置: Secret 先用 3.6 节的 kubectl apply -f secret.yaml 建好 (Chart 不管 Secret)
# 1. 装整套服务 (7 个 manifest 一次性 apply, 等价于 3.3-3.8 节全部手工 apply)
# helm 默认读 Chart 目录下的 values.yaml 填占位符, 跑出来就是 image:0.2 + replicas:1
# (跟原版 manifests/ 等价); 想换值见下面 --set 或 -f my-values.yaml
helm install wensoutu ./wensoutu-chart -n wensoutu --create-namespace

# 2. 看已装的 release
helm list -n wensoutu
# NAME       NAMESPACE   REVISION   STATUS     CHART
# wensoutu   wensoutu    1          deployed   wensoutu-0.1.0

# 3. 改 backend 副本数 1 -> 3 (--set 命令行直接覆盖 values, 不动 values.yaml 文件)
helm upgrade wensoutu ./wensoutu-chart -n wensoutu --set backend.replicas=3
# Release "wensoutu" has been upgraded. Happy Helming!

# 看 backend pod 是不是真变成 3 个 (k9s :po wensoutu 也能直观看到)
kubectl get pod -n wensoutu -l app=backend
# NAME                       READY   STATUS              AGE
# backend-xxx-aaa            1/1     Running             5m       <- 老 pod 保留
# backend-xxx-bbb            0/1     ContainerCreating   3s       <- 新拉的 2 个 pod
# backend-xxx-ccc            0/1     ContainerCreating   3s
# (等 ~90 秒新 pod 模型加载完, 3 个都 1/1 Running)

# 想改其他参数同理: --set backend.tag="0.3" / --set frontend.replicas=2 / ...
# 多环境部署用 -f my-values.yaml 替代 --set(不同环境各一份 values)

# 4. 回滚到上一版本 (本课不需要跑, 只是展示 helm 能回滚)
# helm rollback wensoutu 1 -n wensoutu

# 5. 全部卸载 (本课不需要跑, 跑了整套服务都没了, 后面第四章 GPU 编排实验全跑不通)
# helm uninstall wensoutu -n wensoutu
```

&emsp;&emsp;<b>helm 在工程上的实际价值</b>:同一个 Chart 可以装到不同环境（开发 / 测试 / 生产）,只用一份 `values-dev.yaml` / `values-prod.yaml` 切换;升级镜像 tag 改一行 `--set` 就能完成;出问题一行 `helm rollback` 回滚整套。<b>helm 放在本章最后的原因</b>:必须先理解每个 yaml 的作用、scheduler 的调度过程、reconcile loop 的收敛方式——这些都看明白了再学打包,顺序不能反,否则 helm Chart 出问题会很难 debug。

### 3.11 本章回顾

&emsp;&emsp;<b>本章完成的内容</b>:文搜图项目的 5 个服务（etcd / minio / milvus / backend / frontend）全部跑在 K8s 集群上,浏览器访问 `http://192.168.110.131/` 能看到文搜图首页,真实搜索请求能从 frontend 走 Ingress 走到 backend（<b>本章止于 backend 单副本</b>,3.9 节用并发 curl 看清单 pod 的串行瓶颈,扩到 3 副本对比留到第四章 4.5 节）、做 embedding 后落到 milvus 检索。
&emsp;&emsp;第二章我们把 K8s 集群装好,第三章我们把文搜图 5 服务搬上集群跑通。接下来第四章我们专门看 K8s 编排 GPU 推理服务时的<b>关键能力</b>——这是很多 K8s 入门内容较少系统展开、但 AI 工程师在真实项目里很容易遇到的方向。

---

## <center>第四章 K8s 编排 GPU 推理服务的关键能力</center>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L4_第四章学习路径-d1e1190c.png" width=80%></div>

&emsp;&emsp;第三章已经把文搜图 5 服务搬上 K8s 跑通,这一章不再写新的 yaml,而是在现有 backend Deployment 上反复扩、缩、杀、滚动升级,专门观察 K8s 编排 GPU 推理服务时的关键能力。GPU 推理和普通 web 服务不一样:单请求更慢、副本启动更久、吞吐受 GPU 算力限制,所以我们会从 GPU 强约束和第 5 副本 Pending 讲起,再看共享方案、自愈、多副本压测和滚动升级零停机,最后沉淀出一套<b>GPU 推理服务的 K8s 编排决策框架</b>。

### 4.1 GPU 推理跟 web 服务的根本差异

&emsp;&emsp;<b>GPU 推理服务</b>指的是把深度学习模型加载到 GPU 显存里、用 GPU 算力对请求做前向推理（embedding / 生成 / 分类等）的服务。文搜图 backend 就是典型代表——加载 Qwen3-VL-Embedding-2B 模型到 GPU 显存,每个搜索请求把文本切 token、过模型计算 embedding、用 embedding 在 milvus 里查相似图。

&emsp;&emsp;<b>GPU 推理跟 web 服务在五个维度上的对照</b>——这层差异决定了后面"1 卡 1 pod"、"HPA 自动扩缩容很难发挥作用"、"多副本必须预先开好"这些工程取舍:


<div align=center>

| 维度 | web 服务（如 nginx / 简单 FastAPI） | GPU 推理服务（如本课 backend） |
|---|---|---|
| 单请求响应时间 | 毫秒级（10-100 ms） | 秒级（实测 backend 单请求约 12-14 秒） |
| 副本启动时间 | 秒级（< 5 秒） | 分钟级（本课 backend 实测 80-180 秒,取决于 caption_cache 是否需重建,大模型 5-15 分钟） |
| 单副本吞吐瓶颈 | CPU / IO | GPU 算力（物理封顶） |
| 扩副本边际效益 | 接近线性 | 取决于 GPU 卡数（卡用完就不能再扩） |
| HPA 自动扩缩容适用性 | 适用 | <b>通常不适合</b>（模型加载太慢,HPA 难以及时接住峰值） |

</div>

&emsp;&emsp;<b>这张表里最反常识的是最后一行</b>。HPA（HorizontalPodAutoscaler,K8s 水平自动扩缩容）的设计假设是"看见负载上来,几秒内拉起新副本接住"。但 GPU 推理 pod 从创建到 Ready 要 80-180 秒——HPA 触发扩容时负载已经堆了 1-3 分钟,要么早就堆爆要么早就过去了,自动扩缩容这件事在 GPU 推理服务上<b>失效</b>。

> <font size=2><b>【名词解释】</b><font color=red><b>HPA</b></font>（HorizontalPodAutoscaler,K8s 水平自动扩缩容）:根据 CPU / 内存 / 自定义指标自动调整 Deployment 副本数的内置资源。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>冷启动</b></font>（模型加载到能接请求的间隔）:GPU 推理 pod 从创建到 readinessProbe 通过的时间窗口,本课 backend 实测 80-180 秒（取决于 caption_cache 是否需重建）。</font>

&emsp;&emsp;<b>结论先行</b>:GPU 推理服务的副本数<b>必须预先按峰值估算开好</b>（留足冗余）,不能指望临时扩。这也解释了生产 AI 推理团队普遍采用"预测式扩容"（根据时间 / 历史模式提前 5 分钟扩好）,而不是"反应式扩容"（看见负载上来才扩）——前者赶得上,后者赶不上。

&emsp;&emsp;<b>进阶:业界 4 种弹性方案对比</b>:

<div align=center>


<div align=center>

| 方案 | 工作原理 | 跟 GPU 推理的适配度 |
|---|---|---|
| <b>默认 HPA</b>（看 CPU/内存） | 每 15 秒采样 → 阈值触发扩容 | <b>几乎完全失效</b>。CPU 不打满（GPU 才是瓶颈）+ 副本冷启动 2-3 分钟 = 等扩好流量峰值早过 |
| <b>Custom Metrics HPA</b> | 看 DCGM GPU 利用率 / RPS / 队列长度等自定义指标 | 比默认强,但仍受"扩容 2-3 分钟"硬伤限制 |
| <b>KEDA</b>（K8s Event-Driven Autoscaling） | CNCF 项目,基于 Prometheus / Kafka / Redis 队列长度等事件触发,支持 scale-to-zero | Hugging Face Inference Endpoints 用类似机制做 scale-to-zero,但冷启动 1-3 分钟限制还在 |
| <b>预先开够 + vLLM / SGLang / TensorRT-LLM 内部 batching</b> | 静态副本数按峰值估算,单 pod 内推理框架自己做 continuous batching（也叫 inflight batching）把多请求拼一次前向计算 | <b>2026 年生产 LLM 推理事实标准</b>。开源三大主流引擎 vLLM / SGLang / TensorRT-LLM 都内置;Hugging Face 自家 TGI 也是这条路,但 2025-12 已进入 maintenance mode 推荐改用 vLLM / SGLang |

</div>

</div>

&emsp;&emsp;<b>关键认知</b>:<b>K8s 在大模型推理领域的主要定位不是"临时弹性扩缩",而是"编排和生命周期管理"</b>。弹性这件事可以拆成两层:① 跨副本横向扩 → K8s 负责调度和生命周期管理（`kubectl scale` 静态调整）;② 单 pod 内吸收瞬时峰值 → 靠 vLLM / Triton / TGI 这类推理框架的<b>continuous batching</b> 能力——单 pod 内多个请求自动拼成 batch、一次前向计算跑完所有,GPU 利用率从 30% 提升到 80-95%。本课 backend 用裸 FastAPI + sync 调用是<b>入门简化</b>（让我们看清 K8s 编排在做什么）,生产方式通常是 <b>K8s + vLLM</b> 配合:K8s 管副本调度 / 滚动升级 / 自愈,vLLM 管单 pod 内的请求 batching / KV cache / 显存利用率。

### 4.2 GPU 是 K8s 调度的资源

&emsp;&emsp;<b>4 张 RTX 3090 共 96 GiB 显存,backend 一个 pod 占 12 GiB,理论上能装 8 个 pod——但实际 K8s 只能装 4 个 pod（每卡 1 个）,第 5 个就 Pending</b>。这是 GPU 调度跟普通 CPU / 内存最大的差异,也是后面 GPU 共享方案（下一节）的存在理由。

&emsp;&emsp;<b>三个层面解释为啥不能"跨卡显存累加"</b>:


<div align=center>

| 层 | 真相 |
|---|---|
| <b>K8s 调度层</b> | `nvidia.com/gpu` 资源单位是 <b>integer 整数</b>（一整张物理 GPU）,不支持 `nvidia.com/gpu: 0.5` 小数申请。K8s scheduler 看到的是"GPU 占用 / 未占用"二元状态,不看显存利用率——4 块卡 = 4 个调度单位,装满即 Pending |
| <b>CUDA 运行时层</b> | 推理服务一个 pod = 一个 CUDA context,<b>只能跑在单张 GPU 上</b>。一个 pod 的 12 GiB 显存只能占 1 张卡的 12 GiB,这张卡剩下的 12 GiB 是<b>卡内部浪费</b>,不能"分给"另一张卡的另一个 pod 用 |
| <b>硬件互连层</b> | 跨卡显存聚合（unified memory across GPUs）需要 NVLink 高速互连 + 框架级支持（vLLM tensor parallel / DeepSpeed pipeline parallel 等）。RTX 3090 硬件上<b>支持双卡 NVLink</b>（NVLink 3.0,需另插 30 系专用桥接器;本课这台机器 4 卡都没插桥接器,`nvidia-smi nvlink -s` 显示 `all links are inActive`）,但它最多双卡桥接,不像数据中心卡那样有 NVSwitch 做多卡全互连;更关键的是——<b>NVLink 通了也不等于 K8s 能把多卡显存合成一个池子任意切给 pod</b>。跨卡用显存必须靠应用框架主动做模型并行 / 张量并行（训练场景为主）,不是 `nvidia.com/gpu` 调度层自动完成,也不是推理服务多 pod 部署的玩法 |

</div>

&emsp;&emsp;<b>"理论 96 GiB / 12 GiB = 8 个 pod" 跟 "K8s 实际只装 4 个 pod" 的差距</b>来自 K8s + CUDA + 硬件互连三层的联合限制——不是 K8s scheduler 设计不合理,是 GPU 这件物理资源跟 CPU / 内存的根本差异。想真的在 4 卡上装 8 个 pod,不能靠 `nvidia.com/gpu` 整数申请,必须走 GPU 共享方案（time-slicing / MPS / MIG）。

### 4.3 GPU 共享方案对比:time-slicing / MPS / MIG

&emsp;&emsp;前面 4.2 节讲清了 GPU 是 K8s 调度的原子资源,4 卡只能装 4 个 pod。如果硬件真的不够、又想让多个 pod 共用同一块卡,K8s 加 NVIDIA 一共给了 3 条路:time-slicing（时间片轮转）、MPS（多进程并发）、MIG（硬件分区）。

&emsp;&emsp;<b>三方案对照</b>:


<div align=center>

| 方案 | 原理 | 适用 | 不适用 |
|---|---|---|---|
| <b>time-slicing</b>（时间片轮转） | 多 pod 看似都能用 GPU,Device Plugin 在时间维度切片轮转分配 | 训练任务 / 离线推理 / 开发联调 | 在线推理（切片切换有延迟抖动,p95 会爆） |
| <b>MPS</b>（Multi-Process Service） | 多进程共享一块 GPU 算力,NVIDIA 官方多进程方案 | 多个小模型（显存 < 4GB）共享 1 块大卡 | 大模型（显存 > 8GB）占满显存就不能共享 |
| <b>MIG</b>（Multi-Instance GPU） | 硬件层把 1 块 GPU 切成多份（各份独立 SM / 显存 / 缓存）,真隔离 | A100 / H100 等数据中心卡 | 消费级卡（RTX 3090 不支持 MIG,只能 time-slicing / MPS） |

</div>

> <font size=2><b>【名词解释】</b><font color=red><b>time-slicing</b></font>（GPU 时间片轮转）:NVIDIA Device Plugin 的共享模式,允许 1 块 GPU 同时分给 N 个 pod,但 N 个 pod 实际是在时间维度上轮流用算力。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>MPS</b></font>（Multi-Process Service,NVIDIA 多进程服务）:让多个 CUDA 进程<b>真共享</b>1 块 GPU 的算力,延迟比 time-slicing 低,需要主机层启 nvidia-cuda-mps-control。</font>

> <font size=2><b>【名词解释】</b><font color=red><b>MIG</b></font>（Multi-Instance GPU,硬件多实例 GPU）:A100/H100 上把 1 块 GPU 物理切成 7 份,每份有独立的 SM / 显存 / L2 缓存,跨实例隔离。</font>

&emsp;&emsp;<b>本课 backend 的选型决策</b>:Qwen3-VL-Embedding-2B 加载到显存约 10-15GB,已经吃掉单卡 RTX 3090 24GB 显存的大半,<b>1 卡 1 pod 独占是最优解</b>——不用考虑共享,直接用 3.7 节 的 `nvidia.com/gpu: 1` 强约束。

&emsp;&emsp;<b>共享适用场景</b>:① 单模型显存只占用 2-4GB（小语言模型 / 图像分类小模型）,共享空间大;② 集群 GPU 紧张但负载不饱（白天忙晚上闲）,想榨干使用率;③ 开发环境多人共享一台 GPU 机调试。<b>共享不适用场景</b>:① 在线推理服务对 p95 延迟敏感（time-slicing 切换会抖）;② 单模型大、共享后跑不动。

&emsp;&emsp;<b>三方案速选口诀</b>:开发联调或离线训练用 <b>time-slicing</b>（配置最简单）、生产多小模型混部用 <b>MPS</b>（延迟更稳）、数据中心卡硬隔离用 <b>MIG</b>（最贵但最干净）。具体实施改 Device Plugin 的 ConfigMap 即可,文档地址 `github.com/NVIDIA/k8s-device-plugin`。

### 4.4 自愈:停 pod 看 K8s 自动重启

&emsp;&emsp;<b>自愈</b>在 K8s 语境里指的是<b>reconcile loop 自动收敛期望态和实际态</b>——我们声明 backend Deployment 期望 3 副本,K8s 会一直盯着这件事,任何时刻发现实际副本数少于 3（pod 被杀 / OOM / 节点宕机）,Deployment Controller 立刻新建一个 pod 把数补上。这跟 docker-compose 的行为完全不同:Compose 里 `docker rm` 杀掉一个容器就是杀掉了,要 `docker compose up` 才能再起;K8s 里杀不掉,杀一个起一个。

&emsp;&emsp;<b>实操</b>:删掉 1 个 backend pod,看 K8s 自动起一个新的。

```bash
# 当前 3 副本 Running
kubectl get pod -n wensoutu -l app=backend

# 挑一个删掉
kubectl delete pod -n wensoutu backend-7d4f-abc
# pod "backend-7d4f-abc" deleted

# 立刻再看
kubectl get pod -n wensoutu -l app=backend
# 旧 backend-7d4f-abc 状态变 Terminating
# 立刻出现新 backend-7d4f-newhash 状态 ContainerCreating
```

&emsp;&emsp;<b>同时打开 k9s 看完整过程</b>:终端 1 的 k9s 里选 `:po`,filter `backend`,能看到:① 被杀的 pod 进入 Terminating 状态、GPU 释放;② Deployment 立刻调谐:scheduler 挑节点 → kubelet 拉镜像（本地有不用拉） → 容器启动 → 模型 lazy load + caption_cache 重建;③ 大约 <b>2-3 分钟</b>新 pod 进 Ready 状态,3 副本回到目标态——emptyDir 卷场景下每次新 pod 都要重建 caption_cache,所以跟首次启动一样慢。

> <font size=2><b>【名词解释】</b><font color=red><b>Terminating</b></font>（K8s pod 状态）:pod 已被标记删除但还在做 graceful shutdown,默认给 30 秒让进程响应 SIGTERM（可通过 `terminationGracePeriodSeconds` 调整）,过期 SIGKILL 强杀。</font>

&emsp;&emsp;<b>这件事对 GPU 推理服务的意义</b>:有了自愈,我们写 backend Deployment 时不用自己写"进程崩溃后的拉起逻辑",K8s 会把副本补回来。但有一个代价:<b>2-3 分钟的冷启动时间</b>（模型重新加载 + caption_cache 重建）。这段时间这个 pod 没真就绪,但<b>本课 backend 第三章 3.7 节的入门版没配 readinessProbe,K8s 看进程在就标 Ready=1/1,Service 会把流量打到还没就绪的新 pod,造成 502</b>——4.6 节会讲清"配齐探针实现 0 停机"的原理。生产上常见方式是<b>多副本预先开好 + 配齐 readinessProbe/livenessProbe/startupProbe 三档探针</b>,自愈在这个基础上做兜底,不能指望自愈解决所有可用性问题。

&emsp;&emsp;<b>更进一步</b>:把整个节点 cordon 掉（`kubectl cordon k3s-server`）让 scheduler 不再往上分配新 pod,然后 drain（`kubectl drain k3s-server --ignore-daemonsets`）迁走所有 pod。这是生产换硬件 / 升级内核常用动作。本课单机集群没法真换节点,先理解概念即可。

### 4.5 多副本对延迟的真实价值

&emsp;&emsp;前面 4.1-4.4 讲清了 GPU 推理的特殊性、强约束、共享方案和自愈,这一节回到主线话题——K8s 多副本对 GPU 推理服务的真实价值。我们用真实请求对比 1 副本 vs 3 副本,本节给两组互补实验:<b>实验 A 串行 3 次 curl</b> 看 Service 分发机制（稳定,必跑）,<b>实验 B 并发 3 curl</b> 看多副本延迟价值。实验 A 证明 Service round-robin 会把请求分发到不同 pod,实验 B 证明多副本能把总延迟压下来——不过并发 3 + 3 副本受 iptables round-robin 概率均摊影响,分发常常不均,达不到理想的"每 pod 各 1 个",但总体仍明显比 1 副本快。

&emsp;&emsp;<b>关于测试方法的关键说明</b>:本节用 shell `&` 并发 curl 制造负载,跟推理服务的真实流量（用户离散点击 + 思考间隔,RPS 0.01-0.1）匹配。消费卡在多副本高负载下有偶发的硬件稳定性边界,本课实测就遇到过整机失联（详见末尾"真实工程反例"段）,所以本节避开持续高压压测,只做"几个用户碰巧同时点击"的瞬时观察。

&emsp;&emsp;<b>前置:扩到 3 副本</b>

```bash
# 终端 1: 扩到 3 副本, 等 3 个 pod 都 ready
kubectl scale deployment backend -n wensoutu --replicas=3
kubectl get pod -n wensoutu -l app=backend -w
# 等 3 pod 都打印 "✅ 检索引擎和Agent就绪"(约 140-180 秒,新 pod 第一次启动会重建 caption_cache)
```

&emsp;&emsp;3.9 节我们在 1 副本 backend 上跑过 shell 并发 3 curl:3 个请求响应时间呈阶梯式递增,总耗时大致是单请求的 3 倍。这是单 pod 内 backend `async def` + sync 调用让 event loop 串行化的现象——N 个并发请求被 GIL + 单 CUDA stream 强制成一个队列,<b>总耗时 ≈ N × 单请求耗时</b>,RPS 物理封顶。

---

#### 实验 A: 串行 3 次 curl 看 Service 分发证据å

&emsp;&emsp;<b>这是本节的主线实验</b>,看 Service 的 L4 round-robin 把请求分发到不同 pod。串行发请求（每次发完等返回再发下一次）,3 副本环境下 K8s Service 会把 3 次请求按 iptables 概率规则分发到 3 个 pod。

```bash
# 终端 2: 串行 3 次 curl(每次发完等返回再发下一次), 输出每次单独耗时 + 总耗时
START=$(date +%s.%N)
for i in 1 2 3; do
    curl -s -X POST http://192.168.110.131/api/search \
        -H 'Content-Type: application/json' \
        -d "{\"textQuery\":\"query $i\",\"searchMode\":\"文搜图\",\"recallTopK\":10,\"rerankTopK\":3,\"threshold\":0}" \
        -o /dev/null -w "req$i: HTTP %{http_code}, %{time_total}s\n"
done
END=$(date +%s.%N)
echo "总耗时: $(echo "$END - $START" | bc)s"

# 终端 3: 实时看每个请求落在哪个 pod
kubectl logs -f -n wensoutu -l app=backend --tail=0 --prefix=true | grep 'POST /api/search'
```

&emsp;&emsp;<b>跑下来会看到</b>:3 个请求串行发出,每个都花十秒上下、全部成功;总耗时大致是单请求的 3 倍——<b>跟 1 副本串行处理一样慢</b>。重点不在耗时（串行本来就没指望快）,而在终端 3 的 kubectl logs:这 3 个请求被 Service 分发到了<b>不同的 pod</b>（具体落点由 iptables 概率决定,不承诺严格均匀）。

&emsp;&emsp;<b>实验 A 看的事</b>:终端 3 的 kubectl logs 输出形如:

In [ ]:
[pod/backend-xxx-pod1/backend] 🔍 文搜图: query 1 ...
[pod/backend-xxx-pod2/backend] 🔍 文搜图: query 2 ...
[pod/backend-xxx-pod3/backend] 🔍 文搜图: query 3 ...

&emsp;&emsp;<b>这就是 Service round-robin 真在工作的证据</b>:3 个独立 HTTP 请求,K8s Service 通过 iptables 规则分发到不同 pod。<b>注意实验 A 没利用多副本的延迟价值</b>——因为串行,每个时刻只有 1 个请求在跑,其它 pod 闲着,总耗时 ≈ 3 × 单请求,跟 1 副本一样慢。实验 A 只解决"分发机制对不对"这一件事,延迟优化是实验 B 的事。

---

#### 实验 B: 并发 3 curl 看多副本延迟价值

&emsp;&emsp;<b>这是本节的延伸实验</b>,展示多副本对延迟的工程价值。shell `&` 并发 3 个 curl 同时发,Service 把它们分发到 3 个 pod。理想情况每 pod 各 1 个、总耗时 ≈ 1×单请求;但实测会看到一个真实现象——iptables round-robin 是概率均摊,3 个请求里往往有 2 个落到同一个 pod,那个 pod 内 `async def + sync` 让 event loop 串行,第 2 个请求要等第 1 个跑完,所以总耗时 ≈ 2×单请求（而不是理想的 1×）。

```bash
# 终端 2: shell `&` 同时发 3 个 curl, wait 等所有返回, 输出每次单独耗时 + 总耗时
START=$(date +%s.%N)
for i in 1 2 3; do
    (curl -s -X POST http://192.168.110.131/api/search \
        -H 'Content-Type: application/json' \
        -d "{\"textQuery\":\"query $i\",\"searchMode\":\"文搜图\",\"recallTopK\":10,\"rerankTopK\":3,\"threshold\":0}" \
        -o /dev/null -w "req$i: HTTP %{http_code}, %{time_total}s\n") &
done
wait
END=$(date +%s.%N)
echo "总耗时: $(echo "$END - $START" | bc)s"
```

&emsp;&emsp;<b>跑下来会看到</b>:3 个并发请求被分发,但因为请求数 = 副本数,概率均摊下常常有 <b>2 个落到同一个 pod</b>——那个 pod 串行处理这 2 个,所以其中一个的响应时间明显更长（大约是别人的 2 倍）;另一个 pod 可能闲着没分到。总耗时大致是单请求的 2 倍,<b>明显比 1 副本并发 3（约 3 倍单请求）快,但达不到理想的 1 倍</b>。多轮跑下来,这个"2 个撞一起"的分布相当稳定——这正是 round-robin 概率均摊在请求数≈副本数时的典型表现。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L4-1vs3-latency-curve-be986b6a.png" width=80%></div>

&emsp;&emsp;<b>这组观察说了什么</b>:3 个并发请求被 Service round-robin 分发,但因为<b>请求数 = 副本数</b>,概率均摊下往往有 2 个落到同一个 pod——那个 pod 内 `async def search` 调 sync 让 event loop 串行,第 2 个请求得等第 1 个跑完才开始,响应时间约为别人的 2 倍。<b>总耗时大致是 2 倍单请求,比 1 副本并发 3（约 3 倍单请求）明显快</b>,但达不到理想的 1 倍。如果并发数远大于副本数（例如并发十几个 / 3 副本）,大数定律会让分发趋于均匀,提速更接近理论上限的 3 倍。<br><br>&emsp;&emsp;<b>这暴露了一个真实工程认知</b>:K8s Service 的 L4 负载均衡是 iptables（或 ipvs）<b>概率轮询</b>,不是严格意义的完美轮询——请求数越接近副本数,分发不均的"毛刺"越明显。生产里靠两件事消化:① 副本数 × 单 pod 承受能力按峰值预留足够余量;② 并发量通常远大于副本数,大数定律把分发拉平。

---

&emsp;&emsp;<b>K8s 多副本的真实价值定位</b>（综合两组实验）:K8s 多副本<b>不是"扛单 pod 内的并发"</b>——单 pod 内 GIL + 单 CUDA stream 永远是串行,有几个并发就有几倍延迟（实验 A 证明）。K8s 多副本的真实价值是<b>"把请求分到 N 个独立 pod / N 张独立 GPU"</b>,N 倍延迟变成 N 个 pod 各自 1 倍延迟,Service 的 L4 round-robin 负责均摊（实验 B 证明）。本质是<b>横向扩,不是单点变快</b>。

&emsp;&emsp;<b>分清两层很重要</b>:挂的是<b>硬件层</b>（消费卡的稳定性问题）,而 <b>K8s 的编排能力</b>（yaml / Service / 多副本调度 / 自愈 / 滚动升级）在本课实测里完全成立——多副本并发请求多轮跑下来分发正常、不丢请求。换句话说,硬件不稳是这批消费卡的问题,换数据中心卡就好,跟 K8s 这一层的编排逻辑无关。本课用消费卡是为了能真实跑通 GPU 调度的全流程,生产环境的硬件选型则是另一回事。

### 4.6 滚动升级零停机（原理说明）

&emsp;&emsp;<b>滚动升级</b>（rollout）是 K8s Deployment 的内置能力——更新镜像 / 改 env 时,K8s <b>不会一次性把所有旧 pod 杀掉</b>,而是按比例先起新 pod、等新 pod Ready 后再杀旧 pod,这样服务在升级窗口内始终有 pod 接请求,达成<b>零停机</b>。这跟 docker-compose 升级镜像必须先 stop 后 up 的中断窗口完全不同。

> <font size=2><b>【名词解释】</b><font color=red><b>rollout</b></font>（K8s 滚动升级）:Deployment 的镜像 / env / args 变更时,K8s 按 `maxSurge` / `maxUnavailable` 控制新旧 pod 比例平滑替换,默认行为是"先起新再杀旧",零停机。</font>

&emsp;&emsp;<b>触发滚动升级的命令</b>有两条:改镜像版本（生产发新版本）用 `kubectl set image deployment/backend backend=新镜像:新tag -n wensoutu`;只重启不换镜像（比如改了 ConfigMap 要让 pod 重新读）用 `kubectl rollout restart deployment/backend -n wensoutu`;升级进度用 `kubectl rollout status deployment/backend -n wensoutu` 看。

&emsp;&emsp;<b>滚动机制</b>:K8s 按 `maxSurge`（升级期间最多多出多少副本）和 `maxUnavailable`（最多少几个副本）两个旋钮平滑替换。默认都是 25%,3 副本算下来是"任意时刻最多多 1 个 pod、且旧 pod 必须等新 pod 真 Ready 才能杀"。过程就是"起 1 个新的 → 等它 Ready → 杀 1 个旧的",重复到全部换完——升级窗口内始终有 pod 接请求,这就是零停机的来源。

> <font size=2><b>【名词解释】</b><font color=red><b>maxSurge / maxUnavailable</b></font>（滚动升级两个旋钮）:前者控制升级期间最多多出多少副本,后者控制最多少几个副本,两者一起决定升级速度和可用性的权衡。</font>

&emsp;&emsp;<b>但 K8s 的"0 停机"有一个关键前提——探针守门</b>:必须给 pod 配 readinessProbe,让 K8s 知道"应用层啥时候真就绪"。本课 backend 模型 lazy load 要好几分钟,如果只看容器进程启动（`Ready=1/1`）就往里打流量,新 pod 模型还没 load 完,请求会撞 502。正确做法两步:① backend 实现一个 `/api/ready` 端点,模型没 load 完返回 503、load 完返回 200;② Deployment 配 `readinessProbe` 指向 `/api/ready`——探针不通过,Service 就不把这个 pod 收进流量后端。这样滚动期间,新 pod 在模型加载完之前不接流量,旧 pod 继续顶,才是真正的 0 停机。

> <font size=2><b>【名词解释】</b><font color=red><b>startupProbe / readinessProbe / livenessProbe</b></font>（K8s 三档探针）:startupProbe 等应用启动（过了才让 readiness/liveness 接管）;readinessProbe 决定 Service 是否分流量（应用就绪才接,这是 0 停机的关键）;livenessProbe 决定 pod 是否被杀重建（挂死自动恢复）。</font>

&emsp;&emsp;<b>探针没配齐是最容易踩的坑</b>:这是 K8s 课件最容易跳过、但生产发版第一次就会撞的细节——新 pod 进程一启动就被标 Ready,Service 立刻把流量打到模型还没 load 完的 pod,Traefik 报 502 Bad Gateway。配齐 `/api/ready` + `readinessProbe` 之后,新 pod 在模型就绪前不接流量,升级才真正不丢请求。

> <font size=2><b>【真实工程经验】</b>生产推理服务（vLLM / Triton / TGI）通常自带 `/health` 或 `/v1/models/<name>/ready` 健康端点,配套 readinessProbe 就能实现 0 停机发布。给镜像很大的老服务打这种 readiness 补丁,还可以用 K8s ConfigMap + subPath mount 把单个文件灌进容器、不用重 build 镜像——这是生产里给老服务打 K8s 补丁的常用技巧。</font>

&emsp;&emsp;<b>对照 docker-compose 升级</b>:不用 K8s 用 docker-compose 升级 backend——通常要先 `docker compose stop backend` 停旧容器,再 `docker compose up backend` 起新容器。停起之间<b>服务中断</b>,期间所有请求 ECONNREFUSED。如果想做"0 停机"必须自己写脚本起新容器、改 nginx upstream、热重载 nginx 配置。K8s 用 Deployment 可以直接使用滚动机制——<b>但 0 停机有前提:必须配齐探针 + 后端实现 /ready 端点</b>,这点在 docker-compose 时代不容易暴露,因为它没有"探针 + 自动接流量"这套机制。

&emsp;&emsp;<b>本章收尾</b>:你现在见过了 K8s 编排 GPU 推理服务的<b>6 项关键能力</b>——强约束保护节点资源、Pending 给出明确错误信号、自愈让进程崩溃不变成长期故障、共享方案提供多 pod 选择、多副本扩展 GPU 算力、滚动升级零停机替换版本。下一章我们回顾本课全部收获,收束核心框架。

---

## <center>第五章 课程回顾与进阶路径</center>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/k8s/2026-05-29/L5_第五章学习路径-157daf1b.png" width=80%></div>

&emsp;&emsp;前四章我们从 K8s 基本概念走到集群安装、项目迁移和 GPU 推理服务编排,这一章不再动手,而是把整门课收回来:先回顾前面四章走过的路,再整理一张关键节点速查卡,接着用 K8s 三件核心规律收束、聊聊怎么让 AI 帮写 yaml,最后做个总结。

### 5.1 课程回顾

&emsp;&emsp;回头看这门课走过的路:第一章建立 K8s 的基本模型——控制面怎么调度、工作节点怎么跑 pod、声明式和 reconcile loop 是怎么回事;第二章在一台 4× RTX 3090 的服务器上用 k3s 装了单节点集群,配好 mirror、装上 NVIDIA Device Plugin 把 GPU 变成可调度资源;第三章把文搜图的 5 个服务(etcd / minio / milvus / backend / frontend)从 docker-compose 翻译成 K8s 工作负载,逐个写 yaml、建 ConfigMap / Secret、配 Ingress,最后打包成一个 helm Chart;第四章专门盯 GPU 推理服务的关键能力——强约束、多副本、自愈、滚动升级。

&emsp;&emsp;走完这四章,我们手上有一套能跑 GPU 推理的单节点 K8s 集群、一整套可复刻的 yaml 和 helm Chart、一套"GPU 推理服务怎么上 K8s"的决策思路。下面几节把这些收束成可随时回看的速查卡和核心规律。

### 5.2 提示速查

&emsp;&emsp;前面几章里散布着六个最容易翻车的提示——下表把它们集中收纳成一张速查卡片,以后回看不用读完整章,按表 grep 章节定位、跑检查命令秒确认。

<p align="center"><font face="黑体" size=4>表 5-1 K8s 关键节点速查卡</font></p>
<div align=center>


<div align=center>

| 落点 | 提示 | 检查命令 |
|---|---|---|
| 第二章 2.6 节（GPU Device Plugin） | 业务 pod yaml 三件套:`runtimeClassName: nvidia` + `nodeSelector: kubernetes.io/hostname: k3s-server` + `resources.limits.nvidia.com/gpu: 1`（单节点 nodeSelector 是为生产多机扩展铺垫） | `kubectl get pod -A -o wide` 看所有 GPU pod 都落在 server 节点 |
| 第三章 3.3 节（etcd headless） | etcd `Service` 必须是 `clusterIP: None` 的 headless 形态,Service 名 = StatefulSet 名,pod DNS 才会展开成 `etcd-0.etcd` | `kubectl get svc etcd -n wensoutu -o jsonpath='{.spec.clusterIP}'` 返回 `None` |
| 第三章 3.5 节（milvus initContainer） | milvus pod 启动前必须等 etcd / minio 就绪,否则 milvus 报连不上、CrashLoopBackOff | `kubectl describe pod milvus-0 -n wensoutu \| grep -A 3 'Init Containers'` |
| 第三章 3.7 节（backend GPU） | backend Deployment 必须同时声明 `runtimeClassName: nvidia` + `nvidia.com/gpu: 1`,否则 pod 调度成功但容器里 `nvidia-smi` 找不到卡 | `kubectl exec -n wensoutu deploy/backend -- nvidia-smi \| head -5` |
| 第三章 3.8 节（Ingress className） | `ingressClassName` 在 k3s 上写 `traefik`,写成 `nginx` 会一直 Pending（k3s 默认装的是 traefik 不是 nginx-ingress） | `kubectl get ingress -n wensoutu -o jsonpath='{.items[*].spec.ingressClassName}'` |
| 第四章 4.2 节（GPU跨卡不可聚合） | `nvidia.com/gpu` 是 integer 整数申请,4 卡 = 4 个调度单位,理论 96 GiB / 12 GiB = 8 个 pod 装不下,K8s + CUDA + 硬件互连三层联合限制 | `kubectl describe pod <pending-pod> -n wensoutu \| grep -A 5 Events` 看到 `Insufficient nvidia.com/gpu` |

</div>

</div>

&emsp;&emsp;<b>卡片使用方式</b>:遇到 K8s 部署故障,先把这六行从上到下扫一遍——很多问题会落在前三行（node label / headless Service / initContainer）;GPU 相关的看后三行。哪一行的检查命令输出不对,回到对应章节再走一遍。

### 5.3 K8s 三件核心规律

&emsp;&emsp;K8s 资源对象会越出越多——本课只动过 Pod / Deployment / StatefulSet / Service / Ingress / ConfigMap / Secret / PVC / Init Container 这几个,只是冰山一角。但不管对象怎么变,底层规律永远是同三件事。学完任何新对象,用这三件事去套,大概率秒懂。

&emsp;&emsp;<b>第一件:声明式</b>。

&emsp;&emsp;我们写的所有 yaml 都不是"现在去把它跑起来"的命令,而是声明我们想要的样子。`replicas: 3` 不是"现在起 3 个 pod",是"我希望永远有 3 个 pod 在跑";`image: backend:0.2` 不是"现在把镜像换成 0.2",是"我希望所有 backend pod 跑 0.2 镜像"。K8s 自己想办法让实际态匹配你的声明。这跟 docker-compose 的命令式（"现在去 docker run 起 3 个容器"）是<b>认知层的根本差异</b>。

&emsp;&emsp;<b>第二件:reconcile loop</b>。

&emsp;&emsp;声明式依赖 K8s 控制器的 reconcile loop 实现——每个控制器（Deployment Controller / StatefulSet Controller / 等）不断对比 etcd 里的期望态和集群实际态,不一致就执行操作让实际态向期望态收敛。pod 被杀 → Deployment Controller 注意到副本数少 1 → 立刻新建一个;节点宕机 → scheduler 把那台节点上的 pod 重新调度到其它节点。这条循环永远在跑,自愈、滚动升级、扩缩容全是它的副作用。

&emsp;&emsp;<b>第三件:资源是节点的属性</b>。

&emsp;&emsp;CPU / 内存 / GPU 都是节点上的资源,pod 通过 `resources.requests/limits` 申请,scheduler 看节点上的可分配总量、扣掉已分配,够装下就调度过去,不够就 Pending。这条规律对 GPU 尤其重要——我们看到的"第 5 副本 Pending"就是这条规律的直接体现。理解了这条,K8s 的所有调度行为都是顺理成章的。

### 5.4 AI 辅助写 yaml

&emsp;&emsp;<b>本课写了 11 个 yaml 文件 + 1 个 helm Chart</b>（`runtime-class` / `nvidia-device-plugin` / `test-gpu-pod` / `etcd` / `minio` / `milvus` / `backend` / `frontend` / `ingress` / `configmap` / `secret`）。看到这么多 yaml 容易心想:以后部署是不是都要敲这些字段?<b>不用</b>。K8s yaml 是工程上最适合让 AI 帮写的东西,关键是<b>你给 AI 的"决策资料"</b>够不够精准。

&emsp;&emsp;<b>为什么 yaml 适合 AI 写</b>。① yaml 字段都是 K8s API 标准规范（api-server 严格 validate,拼错字段立刻被打回）;② 模式高度固定——StatefulSet / Deployment / Service / Ingress 都是套字段;③ 你的脑力应该花在"决策"上（选 StatefulSet 还是 Deployment、要不要探针、资源 limits 多少）,AI 把决策翻译成 yaml 字段的活儿做得比手敲快 10 倍且更不易出错。

&emsp;&emsp;<b>给 AI 提供什么资料</b>。资料越精准,AI 写得越对。分三层:

<p align="center"><font face="黑体" size=4>表 5-2 让 AI 写 K8s yaml 的资料清单</font></p>
<div align=center>


<div align=center>

| 层 | 资料 | 例 |
|---|---|---|
| <b>第一层 必给</b> | 架构 / 依赖关系 | "milvus 依赖 etcd + minio;backend 依赖 milvus;frontend 通过 Ingress 入口" |
| | 每个服务的状态属性 | "etcd/minio/milvus 有状态需 PVC → <b>StatefulSet</b>;backend/frontend 无状态 → <b>Deployment</b>" |
| | 镜像 + 端口 + 关键 env | `yanggggg/wensoutu-backend:0.2` / 监听 3001 / `MILVUS_URI=http://milvus:19530` 等 |
| | 基础设施约束 | k3s 单节点 / 4× RTX 3090 / Ubuntu / 国内走 DaoCloud mirror |
| <b>第二层 决策</b> | GPU 需求 | "backend 需要 1 块 GPU,RuntimeClass nvidia,resources.limits.nvidia.com/gpu: 1" |
| | 存储需求 | "etcd PVC 5Gi / minio 10Gi / milvus 20Gi;backend 模型权重 8GB 走 hostPath 不进 PVC" |
| | 网络入口 | "frontend 通过 Traefik Ingress 在 host=wensoutu.local 暴露,/api/* 转 backend,/ 转 frontend" |
| | 配置注入方式 | "非敏感配置走 ConfigMap envFrom;API key 走 Secret envFrom;模型路径走 hostPath" |
| | 健康检查 | "backend 模型加载 90s,需要 startupProbe（failureThreshold=30,periodSeconds=10）+ readinessProbe 指向 /api/ready" |
| <b>第三层 加分</b> | 现有 docker-compose.yml | AI 直接翻译——compose 的 `services / volumes / depends_on` 跟 K8s 的 `Deployment / PVC / initContainer` 一一对应 |
| | 项目 Dockerfile | AI 从中读 `WORKDIR / EXPOSE / CMD` 自动填进 yaml |
| | 类似项目的官方 K8s yaml | 给 AI 一份参考模板,它能仿写得更对 |
| | 生产约束 | 副本数 / CPU 内存 limits / 是否 HPA / 滚动升级 maxSurge 等 |

</div>

</div>

&emsp;&emsp;<b>实操避坑三条</b>:① <b>拆服务逐个写</b>——别让 AI 一次写 8 个 yaml,容易互相串字段。一次写一个,验证完再下一个。② <b>生成后先 `kubectl apply --dry-run=server -f xxx.yaml` 验证 schema</b>——这一步让 K8s api-server 检查字段名 / type / required 是否对,AI 拼错会立刻打回,比 apply 后 pod 起不来再排查省多了。③ <b>重复结构让 AI 出模板</b>——本课 3 个 StatefulSet 共享同一套模式（headless Service + StatefulSet + volumeClaimTemplates + nodeSelector）,让 AI 出一份 etcd 完整版做模板,minio / milvus 按字段填。

&emsp;&emsp;<b>一个可直接用的 prompt 模板</b>（给 Claude / GPT / Gemini 都行）:

```text
我要部署一个 K8s 服务,请帮我写完整 yaml manifest:

【服务】<服务名,如 milvus>
【作用】<一句话,如 向量数据库,存 embedding,本课用于文搜图检索>
【镜像】<full image:tag,如 milvusdb/milvus:v2.4.0>
【端口】<端口列表,如 19530 (gRPC) / 9091 (metrics)>
【存储】<PVC 多少,如 20Gi 持久化 milvus 数据>
【依赖】<其他服务,如 milvus 启动前必须等 etcd:2379 和 minio:9000 都通>
【环境变量】<key=value 列表,如 ETCD_ENDPOINTS=etcd:2379, MINIO_ADDRESS=minio:9000>
【K8s 约束】
  - 部署在 namespace=wensoutu
  - 节点 nodeSelector: kubernetes.io/hostname=k3s-server
  - 不需要 GPU
  - 单副本(本课单机)

请生成:
1. headless Service(同 service name=statefulset name,clusterIP: None)
2. StatefulSet(serviceName 绑定 headless service,volumeClaimTemplates 自动建 PVC)
3. initContainer 等 etcd + minio 就绪
4. 关键字段加 # 注释解释为什么这么写
```

&emsp;&emsp;<b>对应本课 milvus 的填法</b>——把上面模板的【】占位符全填上本课具体值,直接 copy 给 AI:

```text
我要部署一个 K8s 服务,请帮我写完整 yaml manifest:

【服务】milvus
【作用】向量数据库,存 Qwen3-VL-Embedding 输出的图片/文本向量,本课用于文搜图检索(召回阶段)
【镜像】milvusdb/milvus:v2.4.0
【端口】19530 (gRPC, backend 走这个端口连 milvus) / 9091 (metrics, 监控用)
【存储】20Gi 持久化 milvus 元数据 + 索引(本课 18 张图小数据集, 20Gi 足够; 生产按向量规模线性扩)
【依赖】启动前必须等 etcd:2379 + minio:9000 都通(用 initContainer 跑 `nc -z etcd 2379 && nc -z minio 9000` 循环探测)
【环境变量】
  ETCD_ENDPOINTS=etcd:2379                 # milvus 元数据存 etcd
  MINIO_ADDRESS=minio:9000                  # milvus 索引文件存 minio
  MINIO_ACCESS_KEY_ID=minioadmin
  MINIO_SECRET_ACCESS_KEY=minioadmin
  COMMON_STORAGETYPE=minio                  # 让 milvus 把 storage backend 切到 minio
【K8s 约束】
  - 部署在 namespace=wensoutu (本课业务 namespace)
  - 节点 nodeSelector: kubernetes.io/hostname=k3s-server (本课单节点拓扑)
  - 不需要 GPU (向量检索是 CPU 任务)
  - 单副本 (本课单机, 生产 milvus 集群模式走官方 helm Chart)
  - 镜像 mirror: docker.io 走 docker.m.daocloud.io (本课 2.3 节配过)

请生成:
1. headless Service (clusterIP: None, name=milvus, 给 Pod 提供稳定 DNS `milvus-0.milvus`)
2. StatefulSet (serviceName=milvus 绑定 headless service, replicas=1, volumeClaimTemplates 自动建 20Gi PVC)
3. initContainer (image=busybox:1.36, 循环 `nc -z etcd 2379 && nc -z minio 9000` 直到通才退出)
4. 关键字段加 # 注释:为什么用 StatefulSet 而不是 Deployment / 为什么 headless Service / volumeClaimTemplates 怎么工作
```

&emsp;&emsp;<b>对应本课 backend 的填法</b>（GPU 业务服务,比 milvus 复杂一点,多了 GPU 资源 + hostPath 模型挂载）:

```text
我要部署一个 K8s 服务,请帮我写完整 yaml manifest:

【服务】backend
【作用】文搜图项目核心,FastAPI 跑 Qwen3-VL-Embedding-2B 多模态模型,接收 /api/search HTTP 请求,调 milvus 检索
【镜像】yanggggg/wensoutu-backend:0.2
【端口】3001 (FastAPI HTTP)
【存储】无 PVC, 用 hostPath 挂宿主机模型权重 + 测试图片(模型 8GB 不进 PVC 太大, hostPath 单机够用)
  - /home/XiaoYangWorkSpace/wensoutu/volumes/models      挂到 /app/backend/data/models      (Qwen 模型权重)
  - /home/XiaoYangWorkSpace/wensoutu/volumes/images      挂到 /app/backend/data/images      (18 张测试 PNG)
  - /home/XiaoYangWorkSpace/wensoutu/volumes/uploads     挂到 /app/backend/data/uploads     (用户上传, DirectoryOrCreate)
  - /home/XiaoYangWorkSpace/wensoutu/volumes/caption_cache 挂到 /app/backend/data/caption_cache (DirectoryOrCreate)
【依赖】启动前等 milvus:19530 通(用 initContainer `nc -z milvus 19530`)
【环境变量】
  MILVUS_URI=http://milvus:19530
  MODELSCOPE_CACHE=/app/backend/data/models      # 让 modelscope SDK 读本地模型缓存
  HF_HOME=/app/backend/data/models/.hf
  HF_HUB_OFFLINE=1                                # 离线模式不联网
  TRANSFORMERS_OFFLINE=1
  OPENAI_API_KEY=占位                              # backend import 时必读(agent.py)
  OPENAI_BASE_URL=https://openrouter.ai/api/v1
  OPENROUTER_API_KEY=占位
  MINIO_ROOT_USER=minioadmin
  MINIO_ROOT_PASSWORD=minioadmin
【K8s 约束】
  - 部署在 namespace=wensoutu
  - 节点 nodeSelector: kubernetes.io/hostname=k3s-server
  - 需要 1 块 GPU (resources.limits.nvidia.com/gpu: 1)
  - runtimeClassName: nvidia (使用 nvidia-container-runtime, 第二章 2.6 节装好的 RuntimeClass)
  - 资源 limits: memory=16Gi, cpu=4 (本课实测够用)
  - 资源 requests: memory=12Gi, cpu=2
  - 副本数 1 (单副本压测看串行瓶颈, 第四章 4.5 节扩到 3 副本)

请生成:
1. Service (ClusterIP, name=backend, port=3001 → targetPort=3001)
2. Deployment (replicas=1, 关键字段如上)
3. initContainer 等 milvus 就绪
4. 关键字段加 # 注释:为什么用 Deployment 不是 StatefulSet(无状态)/ 为什么 hostPath 不用 PVC(模型 8GB 单机)/ runtimeClassName 和 nvidia.com/gpu 怎么配合
```

&emsp;&emsp;<b>把这两份 prompt 改改"【】"里的内容,出来的 yaml 通常改 1-2 处就能直接 `kubectl apply --dry-run=server` 通过</b>。本课的 11 个 yaml,大概率最初就是用类似 prompt 让 AI 生成,再人工调字段、加注释、加项目特定的 hostPath。<b>真正要修炼的是"知道决策资料该填什么"</b>,而不是手敲字段。

### 5.5 总结

&emsp;&emsp;这门课我们把一个真实的 GPU 推理项目(文搜图)完整搬上了 K8s——从理解声明式和 reconcile loop,到装集群、把 GPU 变成可调度资源,到逐个 yaml 部署 5 个服务、打包 helm Chart,再到看清 GPU 推理服务在多副本、自愈、滚动升级上的真实行为。两道最反直觉的坎——声明式编排、GPU 推理服务的特殊性——我们都迈过去了。

&emsp;&emsp;这只是 K8s 的入门编排。真实生产还有跨节点调度、多主节点高可用、GitOps、Operator 这些方向可以走深。